# 1. Configuration, Setup & Shared Utilities

## 1.1 Imports

In [ ]:
# ============================================
# 1.1 Imports
# ============================================

# Standard library
import os
import warnings
from pathlib import Path
from typing import Any, Optional, Sequence

# Reduce TensorFlow C++ backend logs (must be set before importing TensorFlow).
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"

# Suppress non-critical warnings (only known noisy categories, not all warnings).
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=DeprecationWarning)

# Numerical computing
import numpy as np

# Python Version
import sys

# Data visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Deep learning
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras import mixed_precision

tf.get_logger().setLevel("ERROR")

# Model evaluation
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    precision_recall_fscore_support,
)

# Experiment tracking
import wandb

## 1.2 Reproducibility & Environment

In [ ]:
# ============================================
# 1.2 Reproducibility & Environment
# ============================================

# --------------------------------------------
# Random Seeds
# --------------------------------------------

SEED = 42
keras.utils.set_random_seed(SEED)

# Optional: enforce fully deterministic ops (bit-exact reproducibility).
# Note: this can slow down training, especially on GPU. Disabled by default.
ENABLE_FULL_DETERMINISM = False

if ENABLE_FULL_DETERMINISM:
    tf.config.experimental.enable_op_determinism()


# --------------------------------------------
# GPU / Hardware Information
# --------------------------------------------

print("GPU Information:")

gpus = tf.config.list_physical_devices("GPU")

if gpus:
    print(f"Number of GPUs: {len(gpus)}")
    for gpu in gpus:
        print(f" - {gpu}")
else:
    print("No GPU detected. Training will use CPU.")


# --------------------------------------------
# Mixed Precision
# --------------------------------------------

# Enable mixed-precision training when a compatible GPU is available.
# On CPU, fall back to float32 because mixed_float16 is not beneficial
# for the current training environment.
ENABLE_MIXED_PRECISION = True
MIXED_PRECISION_POLICY = "mixed_float16"

if ENABLE_MIXED_PRECISION and gpus:
    mixed_precision.set_global_policy(MIXED_PRECISION_POLICY)
else:
    mixed_precision.set_global_policy("float32")

ACTIVE_MIXED_PRECISION_POLICY = mixed_precision.global_policy().name


# --------------------------------------------
# Environment Snapshot
# --------------------------------------------

print("\nEnvironment Snapshot:")
print(f"Python Version         : {sys.version.split()[0]}")
print(f"TensorFlow Version     : {tf.__version__}")
print(f"Keras Version          : {keras.__version__}")
print(f"NumPy Version          : {np.__version__}")
print(f"Random Seed            : {SEED}")
print(f"Full Determinism       : {ENABLE_FULL_DETERMINISM}")
print(f"GPU Available           : {bool(gpus)}")
print(f"GPU Count               : {len(gpus)}")
print(f"Mixed Precision        : {ENABLE_MIXED_PRECISION}")
print(f"Mixed Precision Policy : {ACTIVE_MIXED_PRECISION_POLICY}")

## 1.3 Project Configuration

In [ ]:
# ============================================
# 1.3 Project Configuration
# ============================================

# --------------------------------------------
# Project Information
# --------------------------------------------

PROJECT_NAME = "CNN_Image_Classification"
PROJECT_VERSION = "1.0"

# Identify the current model notebook.
# Change this value for each model-specific notebook.
MODEL_NAME = "high"  # "high" | "low" | "mobilenetv2"


# --------------------------------------------
# Dataset Configuration
# --------------------------------------------

IMAGE_SIZE = (64, 64)
IMAGE_CHANNELS = 3
NUM_CLASSES = 10
CLASS_NAMES = None  # Populated in Section 1.4


# --------------------------------------------
# Data Loading Configuration
# --------------------------------------------

BATCH_SIZE = 32


# --------------------------------------------
# Project Paths
# --------------------------------------------

# Assumes the notebook is run from the project root directory.
PROJECT_ROOT = Path.cwd()

# --------------------------------------------
# Project Paths
# --------------------------------------------

DATA_DIR = PROJECT_ROOT / "data"
OUTPUT_DIR = PROJECT_ROOT / "outputs"

# Model-specific subdirectories, keyed by MODEL_NAME,
# so that outputs from different notebooks
# (low / high / mobilenetv2) never overwrite each other.
MODEL_OUTPUT_DIR = OUTPUT_DIR / MODEL_NAME

CHECKPOINT_DIR = MODEL_OUTPUT_DIR / "checkpoints"
REPORTS_DIR = MODEL_OUTPUT_DIR / "reports"
MODEL_DIR = MODEL_OUTPUT_DIR / "models"
PLOTS_DIR = MODEL_OUTPUT_DIR / "plots"


# --------------------------------------------
# Create Required Output Directories
# --------------------------------------------

for directory in [
    OUTPUT_DIR,
    MODEL_OUTPUT_DIR,
    CHECKPOINT_DIR,
    REPORTS_DIR,
    MODEL_DIR,
    PLOTS_DIR,
]:
    directory.mkdir(parents=True, exist_ok=True)


# --------------------------------------------
# Configuration Summary
# --------------------------------------------

print("Project Configuration")
print("-" * 40)

print(f"Project Name       : {PROJECT_NAME}")
print(f"Project Version    : {PROJECT_VERSION}")
print(f"Model Name         : {MODEL_NAME}")
print(f"Image Size         : {IMAGE_SIZE}")
print(f"Image Channels     : {IMAGE_CHANNELS}")
print(f"Number of Classes  : {NUM_CLASSES}")
print(f"Batch Size         : {BATCH_SIZE}")

print(f"Project Root       : {PROJECT_ROOT}")
print(f"Data Directory     : {DATA_DIR}")
print(f"Output Directory   : {MODEL_OUTPUT_DIR}")
print(f"Checkpoint Dir     : {CHECKPOINT_DIR}")
print(f"Reports Dir        : {REPORTS_DIR}")
print(f"Model Dir          : {MODEL_DIR}")
print(f"Plots Dir          : {PLOTS_DIR}")

## 1.4 Class Names & Dataset Configuration

In [ ]:
# ============================================
# 1.4 Class Names & Dataset Configuration
# ============================================

# --------------------------------------------
# Dataset Paths
# --------------------------------------------

# Update DATASET_NAME if the dataset folder has a different name.
DATASET_NAME = "EuroSAT_RGB"

DATASET_DIR = DATA_DIR / DATASET_NAME

TRAIN_DIR = DATASET_DIR / "train"
VAL_DIR = DATASET_DIR / "val"
TEST_DIR = DATASET_DIR / "test"


# --------------------------------------------
# Validate Dataset Structure
# --------------------------------------------

if not DATASET_DIR.exists():
    raise FileNotFoundError(
        f"Dataset directory not found: {DATASET_DIR}"
    )

for split_name, split_dir in [
    ("train", TRAIN_DIR),
    ("validation", VAL_DIR),
    ("test", TEST_DIR),
]:
    if not split_dir.exists():
        raise FileNotFoundError(
            f"{split_name.capitalize()} directory not found: {split_dir}"
        )


# --------------------------------------------
# Class Names
# --------------------------------------------

# Extract class names from the training directory.
# Each subdirectory represents one class.
CLASS_NAMES = sorted(
    directory.name
    for directory in TRAIN_DIR.iterdir()
    if directory.is_dir()
)

if not CLASS_NAMES:
    raise ValueError(
        f"No class directories found in training directory: {TRAIN_DIR}"
    )


# --------------------------------------------
# Validate Class Consistency Across Splits
# --------------------------------------------

# Ensure val/test directories contain exactly the same classes as train.
# A mismatch here could silently corrupt training or evaluation later,
# so we fail fast and report the exact difference.
for split_name, split_dir in [
    ("validation", VAL_DIR),
    ("test", TEST_DIR),
]:
    split_classes = sorted(
        directory.name
        for directory in split_dir.iterdir()
        if directory.is_dir()
    )

    if split_classes != CLASS_NAMES:
        missing = set(CLASS_NAMES) - set(split_classes)
        extra = set(split_classes) - set(CLASS_NAMES)

        raise ValueError(
            f"Class mismatch in {split_name} directory ({split_dir}).\n"
            f"Missing: {sorted(missing) if missing else 'None'}\n"
            f"Extra   : {sorted(extra) if extra else 'None'}"
        )


# --------------------------------------------
# Validate Class Configuration
# --------------------------------------------

if len(CLASS_NAMES) != NUM_CLASSES:
    raise ValueError(
        f"Expected {NUM_CLASSES} classes, "
        f"but found {len(CLASS_NAMES)} classes in {TRAIN_DIR}"
    )


# --------------------------------------------
# Dataset Summary
# --------------------------------------------

print("Dataset Configuration")
print("-" * 40)

print(f"Dataset Name       : {DATASET_NAME}")
print(f"Dataset Directory  : {DATASET_DIR}")
print(f"Train Directory    : {TRAIN_DIR}")
print(f"Validation Dir     : {VAL_DIR}")
print(f"Test Directory     : {TEST_DIR}")
print(f"Number of Classes  : {len(CLASS_NAMES)}")

print("\nClass Names:")
for index, class_name in enumerate(CLASS_NAMES):
    print(f"{index}: {class_name}")

In [ ]:
print("CLASS_NAMES:", repr(CLASS_NAMES))
print("CLASS_NAMES type:", type(CLASS_NAMES))

## 1.5 Dataset Loading & Splitting

In [ ]:
# ============================================
# 1.5 Dataset Loading & Splitting
# ============================================
#
# Note: The dataset is already split into train/val/test on disk
# (see Section 1.4). No in-code splitting or stratification is
# performed here. This section only loads each split as a tf.data.Dataset.


# --------------------------------------------
# Dataset Loading
# --------------------------------------------

def load_dataset(
    directory: Path,
    image_size: tuple[int, int],
    batch_size: int,
    class_names: list[str],
    shuffle: bool,
    seed: int | None = None,
) -> tf.data.Dataset:
    """Load an image dataset from a directory structure.

    Args:
        directory: Path to the split directory (train/val/test),
            containing one subdirectory per class.
        image_size: Target (height, width) for image resizing.
        batch_size: Number of samples per batch.
        class_names: Explicit ordered list of class names. Ensures
            consistent label indices across train/val/test.
        shuffle: Whether to shuffle the dataset. Should be True
            only for the training split.
        seed: Random seed used when shuffling is enabled.

    Returns:
        A batched tf.data.Dataset containing images and one-hot labels.
    """

    dataset = tf.keras.utils.image_dataset_from_directory(
        directory,
        labels="inferred",
        label_mode="categorical",
        class_names=class_names,
        image_size=image_size,
        batch_size=batch_size,
        shuffle=shuffle,
        seed=seed,
    )

    return dataset


# --------------------------------------------
# Train / Validation / Test Pipeline
# --------------------------------------------

train_ds = load_dataset(
    TRAIN_DIR,
    IMAGE_SIZE,
    BATCH_SIZE,
    CLASS_NAMES,
    shuffle=True,
    seed=SEED,
)

val_ds = load_dataset(
    VAL_DIR,
    IMAGE_SIZE,
    BATCH_SIZE,
    CLASS_NAMES,
    shuffle=False,
)

# --------------------------------------------
# Test-Set Lock
# --------------------------------------------
# The test set is loaded here for pipeline consistency, but must NOT
# be evaluated until Section 4.2 (First and Only Official Test-Set
# Evaluation). Do not use test_ds for training, tuning, or model
# selection before that point.

test_ds = load_dataset(
    TEST_DIR,
    IMAGE_SIZE,
    BATCH_SIZE,
    CLASS_NAMES,
    shuffle=False,
)


# --------------------------------------------
# Performance Optimization
# --------------------------------------------

AUTOTUNE = tf.data.AUTOTUNE

train_ds = train_ds.prefetch(buffer_size=AUTOTUNE)
val_ds = val_ds.prefetch(buffer_size=AUTOTUNE)
test_ds = test_ds.prefetch(buffer_size=AUTOTUNE)


# --------------------------------------------
# Dataset Summary
# --------------------------------------------

print("Dataset Summary")
print("-" * 40)
print(f"Train batches      : {train_ds.cardinality().numpy()}")
print(f"Validation batches : {val_ds.cardinality().numpy()}")
print(f"Test batches       : {test_ds.cardinality().numpy()}")


# --------------------------------------------
# Inspect Tensor Shapes
# --------------------------------------------

for images, labels in train_ds.take(1):
    print("\nSample Batch")
    print(f"Images shape       : {images.shape}")
    print(f"Labels shape       : {labels.shape}")

## 1.6 Class Imbalance Handling

In [ ]:
# ============================================
# 1.6 Class Imbalance Handling
# ============================================

# Note:
# This section evaluates class imbalance and pre-computes class
# weights so they are ready for use in later phases, even if the
# baseline model does not enable them.


# --------------------------------------------
# Configuration
# --------------------------------------------

# Maximum acceptable ratio between the largest and smallest class
# before flagging imbalance as significant.
IMBALANCE_THRESHOLD = 1.50


# --------------------------------------------
# Count Training Samples per Class
# --------------------------------------------

def count_samples_per_class(
    directory: Path,
    class_names: list[str],
) -> dict[str, int]:
    """Count image files for each class in a dataset split.

    Args:
        directory: Path to the dataset split directory.
        class_names: Ordered list of class names.

    Returns:
        Dictionary mapping each class name to its sample count.
    """

    image_extensions = {
        ".jpg",
        ".jpeg",
        ".png",
        ".bmp",
        ".gif",
        ".webp",
    }

    class_counts = {}

    for class_name in class_names:
        class_dir = directory / class_name

        count = sum(
            1
            for file_path in class_dir.iterdir()
            if file_path.is_file()
            and file_path.suffix.lower() in image_extensions
        )

        class_counts[class_name] = count

    return class_counts


train_class_counts = count_samples_per_class(
    TRAIN_DIR,
    CLASS_NAMES,
)


# --------------------------------------------
# Calculate Imbalance Ratio
# --------------------------------------------

min_class_count = min(train_class_counts.values())
max_class_count = max(train_class_counts.values())

if min_class_count == 0:
    raise ValueError(
        "At least one class contains zero training samples."
    )

imbalance_ratio = max_class_count / min_class_count

IMBALANCE_STRATEGY = (
    "review_required"
    if imbalance_ratio > IMBALANCE_THRESHOLD
    else "none"
)


# --------------------------------------------
# Compute Class Weights
# --------------------------------------------
# Class weights are pre-computed for optional use in later phases.
# They are not applied to the baseline automatically.
#
# Formula:
#     total_samples / (num_classes * class_count)
#
# Keys are class indices, matching the format expected by
# `model.fit(..., class_weight=...)`.

total_samples = sum(train_class_counts.values())
num_classes = len(CLASS_NAMES)

CLASS_WEIGHT_DICT = {
    class_index: total_samples
    / (num_classes * train_class_counts[class_name])
    for class_index, class_name in enumerate(CLASS_NAMES)
}


# --------------------------------------------
# Imbalance Handling State
# --------------------------------------------

# These flags document whether each strategy is currently applied.
CLASS_WEIGHT_ENABLED = False
WEIGHTED_LOSS_ENABLED = False
OVERSAMPLING_ENABLED = False


# --------------------------------------------
# Class Distribution Summary
# --------------------------------------------

print("Class Imbalance Analysis")
print("-" * 40)

for class_index, class_name in enumerate(CLASS_NAMES):
    print(
        f"{class_index}: "
        f"{class_name:<20} "
        f"{train_class_counts[class_name]:>6} samples   "
        f"weight={CLASS_WEIGHT_DICT[class_index]:.3f}"
    )


# --------------------------------------------
# Imbalance Summary
# --------------------------------------------

print("\nImbalance Summary")
print("-" * 40)

print(f"Minimum class count : {min_class_count}")
print(f"Maximum class count : {max_class_count}")
print(f"Imbalance ratio     : {imbalance_ratio:.2f}")
print(f"Selected strategy   : {IMBALANCE_STRATEGY}")

print("\nHandling Methods")
print("-" * 40)

print(f"class_weight enabled : {CLASS_WEIGHT_ENABLED}")
print(f"weighted loss enabled: {WEIGHTED_LOSS_ENABLED}")
print(f"oversampling enabled : {OVERSAMPLING_ENABLED}")


# --------------------------------------------
# Decision Rationale
# --------------------------------------------

if IMBALANCE_STRATEGY == "none":
    print(
        "\nDecision: Class distribution is within the acceptable "
        f"threshold (ratio {imbalance_ratio:.2f} <= "
        f"{IMBALANCE_THRESHOLD}). Class weights are pre-computed "
        "and available in CLASS_WEIGHT_DICT, but no imbalance "
        "handling method is applied in the baseline."
    )
else:
    print(
        "\nDecision: Class imbalance exceeds the acceptable threshold "
        f"(ratio {imbalance_ratio:.2f} > {IMBALANCE_THRESHOLD}). "
        "Class weights are available in CLASS_WEIGHT_DICT, but the "
        "final handling strategy should be selected after baseline "
        "error analysis."
    )

## 1.7 Data Augmentation

In [ ]:
# ============================================
# 1.7 Data Augmentation
# ============================================
#
# Note: This only defines the augmentation pipeline. It is applied
# as layers at the start of the model architecture (Section 2.2),
# alongside Rescaling — Keras augmentation layers are automatically
# active only during training (training=True) and inactive during
# evaluation/inference, so no manual train-only filtering is needed.

def build_augmentation() -> tf.keras.Sequential:
    """Create the training-time data augmentation pipeline.

    Rotation and flip ranges are intentionally aggressive because the
    dataset consists of satellite/aerial imagery, which has no
    canonical orientation (unlike natural photos) — any rotation or
    flip direction produces a realistic sample.
    """
    return tf.keras.Sequential(
        [
            layers.RandomFlip("horizontal_and_vertical"),
            layers.RandomRotation(1.0),
            layers.RandomZoom(0.2),
        ],
        name="data_augmentation",
    )


augmentation = build_augmentation()

## 1.8 Dataset Sanity Checks

In [ ]:
# ============================================
# 1.8 Dataset Sanity Checks
# ============================================

# Note:
# This section performs lightweight validation checks on the
# train/validation/test datasets before model training.
# No preprocessing, augmentation, splitting, or model evaluation
# is performed here.


# --------------------------------------------
# Expected Tensor Configuration
# --------------------------------------------

EXPECTED_IMAGE_SHAPE = (*IMAGE_SIZE, IMAGE_CHANNELS)
EXPECTED_LABEL_SHAPE = (NUM_CLASSES,)


# --------------------------------------------
# Validate a Dataset Batch
# --------------------------------------------

def validate_dataset_batch(
    dataset: tf.data.Dataset,
    dataset_name: str,
) -> None:
    """Run sanity checks on the first batch of a dataset."""

    for images, labels in dataset.take(1):

        # Check image tensor shape.
        if images.shape.rank != 4:
            raise ValueError(
                f"{dataset_name}: Expected image tensor with rank 4, "
                f"got shape {images.shape}."
            )

        if tuple(images.shape[1:]) != EXPECTED_IMAGE_SHAPE:
            raise ValueError(
                f"{dataset_name}: Expected image shape "
                f"(batch, {EXPECTED_IMAGE_SHAPE[0]}, "
                f"{EXPECTED_IMAGE_SHAPE[1]}, {EXPECTED_IMAGE_SHAPE[2]}), "
                f"got {images.shape}."
            )

        # Check label tensor shape.
        if labels.shape.rank != 2:
            raise ValueError(
                f"{dataset_name}: Expected one-hot labels with rank 2, "
                f"got shape {labels.shape}."
            )

        if labels.shape[-1] != NUM_CLASSES:
            raise ValueError(
                f"{dataset_name}: Expected {NUM_CLASSES} label columns, "
                f"got {labels.shape[-1]}."
            )

        # Check image dtype.
        # image_dataset_from_directory returns float32 by default,
        # even though values are still in the raw [0, 255] range
        # (normalization is deferred to the model's Rescaling layer).
        if images.dtype != tf.float32:
            raise TypeError(
                f"{dataset_name}: Expected images with dtype float32 "
                f"(raw [0, 255] range), got {images.dtype}."
            )

        # Check image value range.
        image_min = tf.reduce_min(images).numpy()
        image_max = tf.reduce_max(images).numpy()

        if image_min < 0 or image_max > 255:
            raise ValueError(
                f"{dataset_name}: Image values must be in [0, 255], "
                f"got range [{image_min}, {image_max}]."
            )

        # Check for invalid numeric values.
        if not tf.reduce_all(tf.math.is_finite(images)):
            raise ValueError(
                f"{dataset_name}: Images contain NaN or Inf values."
            )

        if not tf.reduce_all(tf.math.is_finite(labels)):
            raise ValueError(
                f"{dataset_name}: Labels contain NaN or Inf values."
            )

        # Check one-hot encoding.
        label_sums = tf.reduce_sum(labels, axis=1)

        if not tf.reduce_all(tf.equal(label_sums, 1.0)):
            raise ValueError(
                f"{dataset_name}: Labels are not valid one-hot vectors."
            )

        if not tf.reduce_all(
            tf.logical_or(tf.equal(labels, 0.0), tf.equal(labels, 1.0))
        ):
            raise ValueError(
                f"{dataset_name}: One-hot labels must contain only 0 or 1."
            )

        print(
            f"{dataset_name:<12} "
            f"✓ images={images.shape} | "
            f"labels={labels.shape} | "
            f"dtype={images.dtype.name} | "
            f"range=[{image_min}, {image_max}]"
        )

        return

    raise ValueError(f"{dataset_name}: Dataset is empty.")


# --------------------------------------------
# Validate Train / Validation / Test Datasets
# --------------------------------------------

print("Dataset Sanity Checks")
print("-" * 60)

datasets_to_check = [
    ("Train", train_ds),
    ("Validation", val_ds),
    ("Test", test_ds),
]

for dataset_name, dataset in datasets_to_check:
    validate_dataset_batch(dataset, dataset_name)


# --------------------------------------------
# Dataset Cardinality Checks
# --------------------------------------------

for dataset_name, dataset in datasets_to_check:
    cardinality = dataset.cardinality().numpy()

    if cardinality <= 0:
        raise ValueError(f"{dataset_name}: Dataset contains no batches.")


# --------------------------------------------
# Final Sanity Check
# --------------------------------------------

print("\nAll dataset sanity checks passed successfully.")

## 1.9 Visualization Utilities

In [ ]:
# ============================================
# 1.9 Visualization Utilities
# ============================================
# PLOTS_DIR / REPORTS_DIR are already defined and created
# in Section 1.3 — nothing to redefine here.

import os
import numpy as np
import matplotlib.pyplot as plt

from sklearn.metrics import (
    confusion_matrix,
    ConfusionMatrixDisplay,
    classification_report
)


# --------------------------------------------
# Prediction Extraction
# --------------------------------------------

def extract_predictions(
    model: tf.keras.Model,
    dataset: tf.data.Dataset,
) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    """
    Extract true labels, predicted labels, and probabilities.

    Runs a single batched model.predict() call over the full
    dataset instead of looping batch-by-batch.

    Args:
        model: A trained Keras model.
        dataset: A batched tf.data.Dataset of (images, labels).

    Returns:
        Tuple of (y_true, y_pred, y_prob):
            y_true: Ground-truth class indices.
            y_pred: Predicted class indices.
            y_prob: Predicted class probabilities.
    """
    y_prob = model.predict(dataset, verbose=0)
    y_pred = np.argmax(y_prob, axis=1)

    y_true = []

    for _, labels in dataset:
        if labels.ndim > 1:
            labels = np.argmax(labels.numpy(), axis=1)
        else:
            labels = labels.numpy()

        y_true.extend(labels)

    y_true = np.asarray(y_true)

    if len(y_true) != len(y_pred):
        raise ValueError(
            "Mismatch between ground-truth labels and predictions."
        )

    return y_true, y_pred, y_prob


# --------------------------------------------
# Training History
# --------------------------------------------

def plot_training_history(
    history: tf.keras.callbacks.History,
    model_name: str | None = None,
    phase: str = "phase-1",
    experiment_type: str = "baseline",
    save_dir: str | None = None,
) -> str:
    """
    Plot and save training/validation accuracy and loss.

    Args:
        history: History object returned by model.fit().
        model_name: Identifier used in the plot title and filename;
            defaults to the MODEL_NAME config variable if not provided.
        phase: Pipeline phase this history corresponds to
            (e.g. "phase-1", "phase-2").
        experiment_type: Type of experiment this history corresponds
            to (e.g. "baseline", "retrain", "hpt"). Included in the
            filename so different experiment types under the same
            phase never overwrite each other.
        save_dir: Directory to save the plot into; defaults to the
            PLOTS_DIR config variable if not provided.

    Returns:
        Path to the saved PNG plot.
    """
    model_name = MODEL_NAME if model_name is None else model_name
    save_dir = PLOTS_DIR if save_dir is None else save_dir

    acc = history.history.get("accuracy")
    val_acc = history.history.get("val_accuracy")
    loss = history.history.get("loss")
    val_loss = history.history.get("val_loss")

    if acc is None or val_acc is None:
        raise ValueError("Accuracy history was not found.")

    if loss is None or val_loss is None:
        raise ValueError("Loss history was not found.")

    epochs = range(1, len(acc) + 1)

    fig, axes = plt.subplots(1, 2, figsize=(12, 5))

    # Accuracy
    axes[0].plot(epochs, acc, label="Training Accuracy")
    axes[0].plot(epochs, val_acc, label="Validation Accuracy")
    axes[0].set_title(f"{model_name} - {phase} - {experiment_type} Accuracy")
    axes[0].set_xlabel("Epochs")
    axes[0].set_ylabel("Accuracy")
    axes[0].legend()
    axes[0].grid(True)

    # Loss
    axes[1].plot(epochs, loss, label="Training Loss")
    axes[1].plot(epochs, val_loss, label="Validation Loss")
    axes[1].set_title(f"{model_name} - {phase} - {experiment_type} Loss")
    axes[1].set_xlabel("Epochs")
    axes[1].set_ylabel("Loss")
    axes[1].legend()
    axes[1].grid(True)

    fig.tight_layout()

    save_path = os.path.join(
        save_dir,
        f"{model_name}_{phase}_{experiment_type}_training_history.png"
    )

    fig.savefig(save_path, dpi=300, bbox_inches="tight")

    plt.show()
    plt.close(fig)

    return save_path


# --------------------------------------------
# Confusion Matrix
# --------------------------------------------

def plot_confusion_matrix(
    y_true: np.ndarray,
    y_pred: np.ndarray,
    class_names: list[str],
    model_name: str | None = None,
    phase: str = "phase-1",
    experiment_type: str = "baseline",
    dataset_name: str = "validation",
    save_dir: str | None = None,
) -> str:
    """
    Plot and save a confusion matrix.

    Args:
        y_true: Ground-truth class indices.
        y_pred: Predicted class indices.
        class_names: Display names for each class index.
        model_name: Identifier used in the plot title and filename;
            defaults to the MODEL_NAME config variable if not provided.
        phase: Pipeline phase this matrix corresponds to.
        experiment_type: Type of experiment this matrix corresponds
            to (e.g. "baseline", "retrain", "hpt"). Included in the
            filename so different experiment types under the same
            phase never overwrite each other.
        dataset_name: Name of the dataset split being evaluated
            (e.g. "validation", "test").
        save_dir: Directory to save the plot into; defaults to the
            PLOTS_DIR config variable if not provided.

    Returns:
        Path to the saved PNG plot.
    """
    model_name = MODEL_NAME if model_name is None else model_name
    save_dir = PLOTS_DIR if save_dir is None else save_dir

    cm = confusion_matrix(y_true, y_pred)

    disp = ConfusionMatrixDisplay(
        confusion_matrix=cm,
        display_labels=class_names
    )

    fig, ax = plt.subplots(figsize=(8, 8))

    disp.plot(cmap="Blues", values_format="d", ax=ax)

    ax.set_title(
        f"{model_name} - {phase} - {experiment_type} - "
        f"{dataset_name} Confusion Matrix"
    )

    fig.tight_layout()

    save_path = os.path.join(
        save_dir,
        f"{model_name}_{phase}_{experiment_type}_{dataset_name}_confusion_matrix.png"
    )

    fig.savefig(save_path, dpi=300, bbox_inches="tight")

    plt.show()
    plt.close(fig)

    return save_path


# --------------------------------------------
# Classification Report
# --------------------------------------------

def save_classification_report(
    y_true: np.ndarray,
    y_pred: np.ndarray,
    class_names: list[str],
    model_name: str | None = None,
    phase: str = "phase-1",
    experiment_type: str = "baseline",
    dataset_name: str = "validation",
    save_dir: str | None = None,
) -> tuple[str, dict]:
    """
    Generate, print, and save a classification report.

    Args:
        y_true: Ground-truth class indices.
        y_pred: Predicted class indices.
        class_names: Display names for each class index.
        model_name: Identifier used in the printed header and
            filename; defaults to the MODEL_NAME config variable
            if not provided.
        phase: Pipeline phase this report corresponds to.
        experiment_type: Type of experiment this report corresponds
            to (e.g. "baseline", "retrain", "hpt"). Included in the
            filename so different experiment types under the same
            phase never overwrite each other.
        dataset_name: Name of the dataset split being evaluated
            (e.g. "validation", "test").
        save_dir: Directory to save the report into; defaults to the
            REPORTS_DIR config variable if not provided.

    Returns:
        Tuple of (save_path, report_dict):
            save_path: Path to the saved text report.
            report_dict: The report as a dict (for cross-trial
                metric comparison), via output_dict=True.
    """
    model_name = MODEL_NAME if model_name is None else model_name
    save_dir = REPORTS_DIR if save_dir is None else save_dir

    report_str = classification_report(
        y_true,
        y_pred,
        target_names=class_names,
        digits=4
    )

    report_dict = classification_report(
        y_true,
        y_pred,
        target_names=class_names,
        digits=4,
        output_dict=True
    )

    print(
        f"{model_name} - {phase} - {experiment_type} - "
        f"{dataset_name} Classification Report:\n"
    )
    print(report_str)

    save_path = os.path.join(
        save_dir,
        f"{model_name}_{phase}_{experiment_type}_{dataset_name}_classification_report.txt"
    )

    with open(save_path, "w", encoding="utf-8") as file:
        file.write(report_str)

    return save_path, report_dict



# --------------------------------------------
# Plot Error Samples
# --------------------------------------------

def plot_error_samples(
    file_paths: np.ndarray,
    true_labels: np.ndarray,
    predicted_labels: np.ndarray,
    confidences: np.ndarray,
    class_names: list[str],
    model_name: str | None = None,
    phase: str = "phase-1",
    experiment_type: str = "baseline",
    max_samples: int = 16,
    save_dir: str | None = None,
) -> str:
    """
    Plot and save incorrectly classified samples.

    Samples are sorted by confidence in descending order so the
    most confident errors are displayed first.

    Args:
        file_paths: Path to each sample's source image file.
        true_labels: Ground-truth class index for each sample.
        predicted_labels: Predicted class index for each sample.
        confidences: Model confidence (max predicted probability)
            for each sample.
        class_names: Display names for each class index.
        model_name: Identifier used in the plot title and filename;
            defaults to the MODEL_NAME config variable if not provided.
        phase: Pipeline phase this plot corresponds to.
        experiment_type: Type of experiment this plot corresponds
            to (e.g. "baseline", "retrain", "hpt"). Included in the
            filename so different experiment types under the same
            phase never overwrite each other.
        max_samples: Maximum number of samples to plot.
        save_dir: Directory to save the plot into; defaults to the
            PLOTS_DIR config variable if not provided.

    Returns:
        Path to the saved PNG plot.
    """
    model_name = MODEL_NAME if model_name is None else model_name
    save_dir = PLOTS_DIR if save_dir is None else save_dir

    if not (
        len(file_paths)
        == len(true_labels)
        == len(predicted_labels)
        == len(confidences)
    ):
        raise ValueError("Error sample arrays have inconsistent lengths.")

    if len(file_paths) == 0:
        raise ValueError("No error samples were provided.")

    if max_samples <= 0:
        raise ValueError("max_samples must be greater than zero.")

    num_samples = min(max_samples, len(file_paths))

    sorted_indices = np.argsort(confidences)[::-1]
    selected_indices = sorted_indices[:num_samples]

    num_columns = min(4, num_samples)
    num_rows = int(np.ceil(num_samples / num_columns))

    fig, axes = plt.subplots(
        num_rows,
        num_columns,
        figsize=(4 * num_columns, 4 * num_rows),
        constrained_layout=True,
    )

    axes = np.atleast_1d(axes).ravel()

    for axis, index in zip(axes, selected_indices):
        image_path = file_paths[index]

        if not os.path.isfile(image_path):
            raise FileNotFoundError(f"Error image was not found: {image_path}")

        image = plt.imread(image_path)

        true_label = class_names[true_labels[index]]
        predicted_label = class_names[predicted_labels[index]]
        confidence = confidences[index]

        axis.imshow(image)
        axis.set_title(
            f"True: {true_label}\n"
            f"Pred: {predicted_label}\n"
            f"Confidence: {confidence:.4f}"
        )
        axis.axis("off")

    for axis in axes[num_samples:]:
        axis.axis("off")

    fig.suptitle(
        f"{model_name} - {phase} - {experiment_type} Error Analysis",
        fontsize=14,
    )

    save_path = os.path.join(
        save_dir,
        f"{model_name}_{phase}_{experiment_type}_error_analysis.png"
    )

    fig.savefig(save_path, dpi=300, bbox_inches="tight")

    plt.show()
    plt.close(fig)

    return save_path



# --------------------------------------------
# Plot Grad-CAM
# --------------------------------------------

def plot_grad_cam(
    image: np.ndarray,
    true_heatmap: np.ndarray,
    predicted_heatmap: np.ndarray,
    true_label: str,
    predicted_label: str,
    confidence: float,
    model_name: str | None = None,
    phase: str = "phase-1",
    experiment_type: str = "baseline",
    sample_index: int | None = None,
    save_dir: str | None = None,
) -> str:
    """
    Plot and save a Grad-CAM visualization.

    Shows the original image alongside the Grad-CAM overlay for
    the true class and for the predicted class, side by side —
    where the model should have looked for the correct answer
    versus where it actually looked when it made its prediction.
    Both heatmaps are resized to match the original image's
    dimensions.

    Args:
        image: Original image in raw pixel-value format with
            shape (H, W, C).
        true_heatmap: Grad-CAM heatmap for the true class, from
            generate_grad_cam.
        predicted_heatmap: Grad-CAM heatmap for the predicted
            class, from generate_grad_cam.
        true_label: Ground-truth class name for the sample.
        predicted_label: Predicted class name for the sample.
        confidence: Model confidence (max predicted probability)
            for the sample.
        model_name: Identifier used in the plot title and filename;
            defaults to the MODEL_NAME config variable if not provided.
        phase: Pipeline phase this plot corresponds to.
        experiment_type: Type of experiment this plot corresponds
            to (e.g. "baseline", "retrain", "hpt"). Included in the
            filename so different experiment types under the same
            phase never overwrite each other.
        sample_index: Optional sample identifier, appended to the
            title and filename to distinguish multiple Grad-CAM
            plots from the same run.
        save_dir: Directory to save the plot into; defaults to the
            PLOTS_DIR config variable if not provided.

    Returns:
        Path to the saved PNG plot.
    """
    model_name = MODEL_NAME if model_name is None else model_name
    save_dir = PLOTS_DIR if save_dir is None else save_dir

    if image.ndim != 3:
        raise ValueError("Image must have shape (H, W, C).")

    if true_heatmap.ndim != 2 or predicted_heatmap.ndim != 2:
        raise ValueError("Grad-CAM heatmaps must have shape (H, W).")

    if true_heatmap.shape != predicted_heatmap.shape:
        raise ValueError("Grad-CAM heatmaps must have the same shape.")

    if not 0.0 <= confidence <= 1.0:
        raise ValueError("Confidence must be within the [0, 1] range.")

    display_image = np.clip(image, 0, 255).astype(np.uint8)

    # Both heatmaps come from the same target layer for the same
    # image, so one resize target covers both.
    target_size = display_image.shape[:2]

    resized_true_heatmap = tf.image.resize(
        true_heatmap[..., np.newaxis],
        size=target_size,
        method="bilinear",
    ).numpy()[..., 0]

    resized_predicted_heatmap = tf.image.resize(
        predicted_heatmap[..., np.newaxis],
        size=target_size,
        method="bilinear",
    ).numpy()[..., 0]

    fig, axes = plt.subplots(
        1,
        3,
        figsize=(12, 4),
        constrained_layout=True,
    )

    axes[0].imshow(display_image)
    axes[0].set_title("Original Image")
    axes[0].axis("off")

    axes[1].imshow(display_image)
    axes[1].imshow(resized_true_heatmap, cmap="jet", alpha=0.45)
    axes[1].set_title(f"Grad-CAM: True\n({true_label})")
    axes[1].axis("off")

    axes[2].imshow(display_image)
    axes[2].imshow(resized_predicted_heatmap, cmap="jet", alpha=0.45)
    axes[2].set_title(
        f"Grad-CAM: Predicted\n({predicted_label})\n"
        f"Confidence: {confidence:.4f}"
    )
    axes[2].axis("off")

    title = f"{model_name} - {phase} - {experiment_type} - Grad-CAM"

    if sample_index is not None:
        title += f" - Sample {sample_index}"

    fig.suptitle(title, fontsize=14)

    sample_suffix = (
        f"_sample_{sample_index}" if sample_index is not None else ""
    )

    save_path = os.path.join(
        save_dir,
        f"{model_name}_{phase}_{experiment_type}"
        f"_grad_cam{sample_suffix}.png",
    )

    fig.savefig(save_path, dpi=300, bbox_inches="tight")

    plt.show()
    plt.close(fig)

    return save_path

## 1.10 Evaluation Utilities

In [ ]:
# ============================================
# 1.10 Evaluation Utilities
# ============================================

import os
import json

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)


# --------------------------------------------
# Calculate Classification Metrics
# --------------------------------------------

def calculate_classification_metrics(
    y_true: np.ndarray,
    y_pred: np.ndarray,
    average: str = "macro",
) -> dict:
    """
    Calculate precision, recall, and F1-score
    using an explicit averaging strategy.

    Args:
        y_true: Ground-truth class indices.
        y_pred: Predicted class indices.
        average: Averaging strategy. Supported values:
            "macro" or "weighted".

    Returns:
        Dict with precision, recall, and f1_score.
    """
    if average not in {"macro", "weighted"}:
        raise ValueError(
            "average must be either 'macro' or 'weighted'."
        )

    metrics = {
        "precision": float(precision_score(
            y_true, y_pred, average=average, zero_division=0
        )),
        "recall": float(recall_score(
            y_true, y_pred, average=average, zero_division=0
        )),
        "f1_score": float(f1_score(
            y_true, y_pred, average=average, zero_division=0
        )),
    }

    return metrics


# --------------------------------------------
# Calculate All Classification Metrics
# --------------------------------------------

def calculate_all_classification_metrics(
    y_true: np.ndarray,
    y_pred: np.ndarray,
) -> dict:
    """
    Calculate accuracy and classification metrics using both
    macro and weighted averaging.

    Reuses calculate_classification_metrics() as the single
    source of truth for precision/recall/f1, rather than
    recomputing them separately.

    Args:
        y_true: Ground-truth class indices.
        y_pred: Predicted class indices.

    Returns:
        Dict with "accuracy", "macro", and "weighted" keys.
    """
    return {
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "macro": calculate_classification_metrics(
            y_true, y_pred, average="macro"
        ),
        "weighted": calculate_classification_metrics(
            y_true, y_pred, average="weighted"
        ),
    }


# --------------------------------------------
# Save Evaluation Metrics
# --------------------------------------------

def save_evaluation_metrics(
    metrics: dict,
    model_name: str | None = None,
    phase: str = "phase-1",
    experiment_type: str = "baseline",
    dataset_name: str = "validation",
    save_dir: str | None = None,
) -> str:
    """
    Save evaluation metrics as a JSON file.

    Args:
        metrics: Dict of evaluation metrics to save.
        model_name: Identifier used in the filename; defaults to
            the MODEL_NAME config variable if not provided.
        phase: Pipeline phase these metrics correspond to
            (e.g. "phase-1", "phase-2").
        experiment_type: Type of experiment these metrics correspond
            to (e.g. "baseline", "retrain", "hpt"). Included in the
            filename so different experiment types under the same
            phase never overwrite each other.
        dataset_name: Name of the dataset split evaluated
            (e.g. "validation", "test").
        save_dir: Directory to save the file into; defaults to the
            REPORTS_DIR config variable if not provided.

    Returns:
        Path to the saved JSON file.
    """
    model_name = MODEL_NAME if model_name is None else model_name
    save_dir = REPORTS_DIR if save_dir is None else save_dir

    save_path = os.path.join(
        save_dir,
        f"{model_name}_{phase}_{experiment_type}_{dataset_name}_metrics.json"
    )

    with open(save_path, "w", encoding="utf-8") as file:
        json.dump(metrics, file, indent=4)

    return save_path

## 1.11 Error Analysis Utilities

In [ ]:
# ============================================
# 1.11 Error Analysis Utilities
# ============================================

import numpy as np
import pandas as pd

# --------------------------------------------
# Calculate Prediction Confidence
# --------------------------------------------

def calculate_prediction_confidence(
    y_prob
):
    """
    Calculate the confidence of each prediction.

    Confidence is defined as the maximum
    predicted class probability.

    Returns
    -------
    confidences : np.ndarray
        Maximum class probability for each sample.
    """

    return np.max(
        y_prob,
        axis=1
    )


# --------------------------------------------
# Extract Error Samples
# --------------------------------------------

def extract_error_samples(
    y_true,
    y_pred,
    y_prob
):
    """
    Extract incorrect predictions and their
    corresponding confidence scores.

    Returns
    -------
    error_indices : np.ndarray
        Indices of incorrectly classified samples.

    true_labels : np.ndarray
        Ground-truth labels of error samples.

    predicted_labels : np.ndarray
        Predicted labels of error samples.

    confidences : np.ndarray
        Prediction confidence for each error sample.
    """

    error_mask = y_true != y_pred

    error_indices = np.where(
        error_mask
    )[0]

    true_labels = y_true[
        error_indices
    ]

    predicted_labels = y_pred[
        error_indices
    ]

    confidences = calculate_prediction_confidence(
        y_prob[error_indices]
    )

    return (
        error_indices,
        true_labels,
        predicted_labels,
        confidences
    )


# --------------------------------------------
# Extract High-Confidence Errors
# --------------------------------------------

def extract_high_confidence_errors(
    y_true,
    y_pred,
    y_prob,
    threshold=0.80
):
    """
    Extract incorrect predictions made with
    confidence greater than or equal to threshold.

    Builds on extract_error_samples() and applies
    a confidence-based filter.

    Returns
    -------
    high_conf_indices : np.ndarray
        Indices of high-confidence error samples.

    true_labels : np.ndarray
        Ground-truth labels.

    predicted_labels : np.ndarray
        Predicted labels.

    confidences : np.ndarray
        Prediction confidence values.
    """

    (
        error_indices,
        true_labels,
        predicted_labels,
        confidences
    ) = extract_error_samples(
        y_true,
        y_pred,
        y_prob
    )

    high_conf_mask = (
        confidences >= threshold
    )

    return (
        error_indices[high_conf_mask],
        true_labels[high_conf_mask],
        predicted_labels[high_conf_mask],
        confidences[high_conf_mask]
    )


# --------------------------------------------
# Calculate Error Statistics
# --------------------------------------------

def calculate_error_statistics(
    y_true,
    y_pred,
    y_prob,
    class_names
):
    """
    Calculate overall and per-class error statistics.

    Returns
    -------
    statistics : dict
        Overall and per-class error statistics.
    """

    confidences = calculate_prediction_confidence(
        y_prob
    )

    total_samples = len(y_true)

    error_mask = y_true != y_pred

    total_errors = int(
        np.sum(error_mask)
    )

    overall_error_rate = (
        total_errors / total_samples
        if total_samples > 0
        else 0.0
    )

    mean_confidence = (
        float(np.mean(confidences))
        if total_samples > 0
        else 0.0
    )

    mean_error_confidence = (
        float(np.mean(confidences[error_mask]))
        if total_errors > 0
        else 0.0
    )

    statistics = {
        "total_samples": total_samples,
        "total_errors": total_errors,
        "overall_error_rate": float(
            overall_error_rate
        ),
        "mean_confidence": mean_confidence,
        "mean_error_confidence": mean_error_confidence,
        "per_class": {}
    }

    for class_id, class_name in enumerate(
        class_names
    ):

        class_mask = y_true == class_id

        class_total = int(
            np.sum(class_mask)
        )

        class_errors = int(
            np.sum(
                class_mask & error_mask
            )
        )

        class_error_rate = (
            class_errors / class_total
            if class_total > 0
            else 0.0
        )

        statistics["per_class"][
            class_name
        ] = {
            "total_samples": class_total,
            "errors": class_errors,
            "error_rate": float(
                class_error_rate
            )
        }

    return statistics


# --------------------------------------------
# Calculate High-Confidence Error Distribution
# --------------------------------------------

def calculate_high_confidence_error_distribution(
    true_labels,
    predicted_labels,
    confidences,
    class_names,
):
    """
    Calculate the distribution of high-confidence errors
    grouped by True Class -> Predicted Class.

    This utility assumes that the input arrays already contain
    only high-confidence errors. No confidence thresholding or
    error filtering is performed here.

    Parameters
    ----------
    true_labels : np.ndarray
        Ground-truth class indices for high-confidence errors.

    predicted_labels : np.ndarray
        Predicted class indices for high-confidence errors.

    confidences : np.ndarray
        Prediction confidence values for high-confidence errors.

    class_names : list or tuple
        Class names indexed by class ID.

    Returns
    -------
    distribution : pd.DataFrame
        Grouped error statistics for each
        True Class -> Predicted Class pair.

        Columns:
        - true_class
        - predicted_class
        - error_count
        - mean_confidence
        - max_confidence
        - min_confidence
    """

    true_labels = np.asarray(
        true_labels
    )

    predicted_labels = np.asarray(
        predicted_labels
    )

    confidences = np.asarray(
        confidences
    )

    # ----------------------------------------
    # Input Validation
    # ----------------------------------------

    if not (
        len(true_labels)
        == len(predicted_labels)
        == len(confidences)
    ):
        raise ValueError(
            "true_labels, predicted_labels, and "
            "confidences must have the same length."
        )

    if len(class_names) == 0:
        raise ValueError(
            "class_names must contain at least one class."
        )

    if len(true_labels) == 0:
        return pd.DataFrame(
            columns=[
                "true_class",
                "predicted_class",
                "error_count",
                "mean_confidence",
                "max_confidence",
                "min_confidence",
            ]
        )

    # ----------------------------------------
    # Convert Class IDs to Class Names
    # ----------------------------------------

    true_class_names = np.asarray(
        class_names,
        dtype=object,
    )[true_labels]

    predicted_class_names = np.asarray(
        class_names,
        dtype=object,
    )[predicted_labels]

    # ----------------------------------------
    # Build Error-Level DataFrame
    # ----------------------------------------

    error_data = pd.DataFrame(
        {
            "true_class": true_class_names,
            "predicted_class": predicted_class_names,
            "confidence": confidences,
        }
    )

    # ----------------------------------------
    # Group True -> Predicted Errors
    # ----------------------------------------

    distribution = (
        error_data
        .groupby(
            [
                "true_class",
                "predicted_class",
            ],
            as_index=False,
        )
        .agg(
            error_count=(
                "confidence",
                "size",
            ),
            mean_confidence=(
                "confidence",
                "mean",
            ),
            max_confidence=(
                "confidence",
                "max",
            ),
            min_confidence=(
                "confidence",
                "min",
            ),
        )
    )

    # ----------------------------------------
    # Sort by Most Frequent Error Pairs
    # ----------------------------------------

    distribution = distribution.sort_values(
        by=[
            "error_count",
            "mean_confidence",
        ],
        ascending=[
            False,
            False,
        ],
    ).reset_index(
        drop=True
    )

    return distribution

# --------------------------------------------
# Grad-CAM
# --------------------------------------------

def generate_grad_cam(
    model: tf.keras.Model,
    image: np.ndarray,
    target_class: int,
    layer_name: str,
) -> np.ndarray:
    """
    Generate a Grad-CAM heatmap for a target class.

    Gradients are computed with respect to the pre-softmax
    logit for the target class. This avoids the class coupling
    introduced by softmax normalization.

    The implementation is compatible with mixed-precision models.
    In particular, the classifier input is explicitly cast to the
    dtype of the final Dense layer kernel before the manual logits
    computation.

    Parameters
    ----------
    model : tf.keras.Model
        Trained classification model.

    image : np.ndarray
        Input image in raw pixel-value format with
        shape (H, W, C).

    target_class : int
        Class index used as the Grad-CAM target.

    layer_name : str
        Name of the convolutional layer used for Grad-CAM.

    Returns
    -------
    heatmap : np.ndarray
        Normalized Grad-CAM heatmap with values in [0, 1].
    """

    target_layer = model.get_layer(
        layer_name
    )

    if len(target_layer.output.shape) != 4:
        raise ValueError(
            f"Target layer '{layer_name}' must produce "
            "a 4D feature map."
        )

    output_layer = model.get_layer(
        "predictions"
    )

    if not isinstance(
        output_layer,
        tf.keras.layers.Dense,
    ):
        raise ValueError(
            "The final 'predictions' layer must be a Dense layer."
        )

    if not (
        0 <= target_class < output_layer.units
    ):
        raise ValueError(
            f"target_class must be in the range "
            f"[0, {output_layer.units - 1}]."
        )

    grad_model = tf.keras.models.Model(
        inputs=model.inputs,
        outputs=[
            target_layer.output,
            output_layer.input,
        ],
    )

    image_tensor = tf.convert_to_tensor(
        image,
        dtype=tf.float32,
    )

    if len(image_tensor.shape) == 3:
        image_tensor = tf.expand_dims(
            image_tensor,
            axis=0,
        )

    if len(image_tensor.shape) != 4:
        raise ValueError(
            "Image must have shape (H, W, C) or (B, H, W, C)."
        )

    with tf.GradientTape() as tape:

        conv_outputs, classifier_input = grad_model(
            image_tensor,
            training=False,
        )

        # ----------------------------------------
        # Mixed-Precision Compatibility
        # ----------------------------------------
        #
        # Under mixed_float16:
        #   classifier_input -> float16
        #   output_layer.kernel -> float32
        #
        # MatMul requires compatible dtypes, so the classifier
        # input is explicitly cast to the kernel dtype.
        #
        # This cast does NOT modify the model weights or the
        # model's mixed-precision configuration.

        classifier_input = tf.cast(
            classifier_input,
            output_layer.kernel.dtype,
        )

        logits = tf.matmul(
            classifier_input,
            output_layer.kernel,
        )

        if output_layer.bias is not None:
            logits = logits + tf.cast(
                output_layer.bias,
                logits.dtype,
            )

        class_score = logits[
            :,
            target_class,
        ]

    gradients = tape.gradient(
        class_score,
        conv_outputs,
    )

    if gradients is None:
        raise ValueError(
            "Gradients could not be computed for "
            f"target layer '{layer_name}'."
        )

    pooled_gradients = tf.reduce_mean(
        gradients,
        axis=(1, 2),
    )

    conv_outputs = conv_outputs[0]
    pooled_gradients = pooled_gradients[0]

    weighted_features = (
        conv_outputs
        * pooled_gradients
    )

    heatmap = tf.reduce_sum(
        weighted_features,
        axis=-1,
    )

    heatmap = tf.maximum(
        heatmap,
        0,
    )

    max_value = tf.reduce_max(
        heatmap
    )

    heatmap = tf.where(
        max_value > 0,
        heatmap / max_value,
        tf.zeros_like(heatmap),
    )

    return heatmap.numpy()




## 1.12 Training Utilities

In [ ]:
# ============================================
# 1.12 Training Utilities
# ============================================

import os
import tensorflow as tf


# --------------------------------------------
# Early Stopping
# --------------------------------------------

def build_early_stopping(
    monitor: str = "val_loss",
    patience: int = 5,
    mode: str = "min",
    restore_best_weights: bool = True,
) -> tf.keras.callbacks.EarlyStopping:
    """
    Create an EarlyStopping callback.

    Args:
        monitor: Metric to monitor.
        patience: Number of epochs with no improvement before stopping.
        mode: "min" or "max", direction of improvement for `monitor`.
        restore_best_weights: Whether to restore model weights from
            the epoch with the best value of `monitor`.

    Returns:
        A configured EarlyStopping callback.
    """
    return tf.keras.callbacks.EarlyStopping(
        monitor=monitor,
        patience=patience,
        mode=mode,
        restore_best_weights=restore_best_weights,
        verbose=1,
    )


# --------------------------------------------
# Model Checkpoint
# --------------------------------------------

def build_model_checkpoint(
    filepath: str,
    monitor: str = "val_loss",
    mode: str = "min",
    save_best_only: bool = True,
    save_weights_only: bool = False,
) -> tf.keras.callbacks.ModelCheckpoint:
    """
    Create a ModelCheckpoint callback.

    Args:
        filepath: Path to save the checkpoint file to; the parent
            directory is created automatically if it doesn't exist.
        monitor: Metric to monitor.
        mode: "min" or "max", direction of improvement for `monitor`.
        save_best_only: Whether to only keep the best checkpoint
            according to `monitor`.
        save_weights_only: Whether to save only the model weights,
            instead of the full model.

    Returns:
        A configured ModelCheckpoint callback.
    """
    checkpoint_dir = os.path.dirname(filepath)

    if checkpoint_dir:
        os.makedirs(checkpoint_dir, exist_ok=True)

    return tf.keras.callbacks.ModelCheckpoint(
        filepath=filepath,
        monitor=monitor,
        mode=mode,
        save_best_only=save_best_only,
        save_weights_only=save_weights_only,
        verbose=1,
    )


# --------------------------------------------
# Reduce Learning Rate on Plateau
# --------------------------------------------
#
# NOTE: This callback and build_cosine_lr_schedule() below are
# ALTERNATIVE learning-rate strategies, not complementary ones.
# Do not use both in the same training run.
#
# ReduceLROnPlateau mutates the optimizer's learning rate in place,
# which only works when learning_rate is a plain scalar/variable.
# If the optimizer's learning_rate is instead a LearningRateSchedule
# (e.g. CosineDecay), this callback has no effect (or errors,
# depending on the TF version), since the schedule computes the
# rate internally from the step count instead of an external value.
#
# Pick one strategy per training run:
#   - ReduceLROnPlateau  -> pass a plain float as learning_rate
#   - CosineDecay        -> pass the schedule as learning_rate,
#                            and do NOT add ReduceLROnPlateau
#                            to the callbacks list
# --------------------------------------------

def build_reduce_lr_on_plateau(
    monitor: str = "val_loss",
    factor: float = 0.5,
    patience: int = 2,
    min_lr: float = 1e-6,
    mode: str = "min",
) -> tf.keras.callbacks.ReduceLROnPlateau:
    """
    Create a ReduceLROnPlateau callback.

    Requires the optimizer's learning_rate to be a plain scalar,
    not a LearningRateSchedule (see module-level note above).

    Args:
        monitor: Metric to monitor.
        factor: Factor by which the learning rate is reduced
            (new_lr = lr * factor).
        patience: Number of epochs with no improvement before
            reducing the learning rate.
        min_lr: Lower bound on the learning rate.
        mode: "min" or "max", direction of improvement for `monitor`.

    Returns:
        A configured ReduceLROnPlateau callback.
    """
    return tf.keras.callbacks.ReduceLROnPlateau(
        monitor=monitor,
        factor=factor,
        patience=patience,
        min_lr=min_lr,
        mode=mode,
        verbose=1,
    )


# --------------------------------------------
# Cosine Learning Rate Schedule
# --------------------------------------------

def build_cosine_lr_schedule(
    initial_learning_rate: float,
    decay_steps: int,
    alpha: float = 0.0,
) -> tf.keras.optimizers.schedules.CosineDecay:
    """
    Create a CosineDecay learning-rate schedule.

    Pass the returned schedule directly as the optimizer's
    learning_rate. Do not combine with ReduceLROnPlateau
    (see module-level note above).

    Args:
        initial_learning_rate: Initial learning rate.
        decay_steps: Number of optimizer steps over which the
            learning rate decays.
        alpha: Minimum learning-rate ratio relative to the initial
            learning rate.

    Returns:
        A CosineDecay learning-rate schedule.
    """
    if decay_steps <= 0:
        raise ValueError("decay_steps must be greater than zero.")

    return tf.keras.optimizers.schedules.CosineDecay(
        initial_learning_rate=initial_learning_rate,
        decay_steps=decay_steps,
        alpha=alpha,
    )

## 1.13 Model Reporting Utilities

In [ ]:
# ============================================
# 1.13 Model Reporting Utilities
# ============================================

import os
import json
import numpy as np


# --------------------------------------------
# Model Parameter Statistics
# --------------------------------------------

def get_model_parameter_stats(model: tf.keras.Model) -> dict:
    """
    Calculate model parameter statistics.

    Args:
        model: A built (or compiled) Keras model.

    Returns:
        Dict with total, trainable, and non-trainable parameter counts.
    """
    trainable_params = int(
        sum(
            np.prod(w.shape)
            for w in model.trainable_weights
        )
    )

    non_trainable_params = int(
        sum(
            np.prod(w.shape)
            for w in model.non_trainable_weights
        )
    )

    total_params = trainable_params + non_trainable_params

    return {
        "total_params": total_params,
        "trainable_params": trainable_params,
        "non_trainable_params": non_trainable_params,
    }


# --------------------------------------------
# Model Configuration
# --------------------------------------------

def get_model_config(model: tf.keras.Model) -> dict:
    """
    Extract basic model configuration information.

    Args:
        model: A built (or compiled) Keras model.

    Returns:
        Dict with model name, input/output shape, and layer count.
    """
    return {
        "name": model.name,
        "input_shape": str(model.input_shape),
        "output_shape": str(model.output_shape),
        "num_layers": len(model.layers),
    }


# --------------------------------------------
# Build Model Report
# --------------------------------------------

def build_model_report(
    model: tf.keras.Model,
    model_name: str | None = None,
    phase: str = "phase1",
) -> dict:
    """
    Build a structured model report.

    Args:
        model: A built (or compiled) Keras model.
        model_name: Identifier used in the saved report; defaults to
            the MODEL_NAME config variable if not provided.
        phase: Pipeline phase this report corresponds to
            (e.g. "phase1", "phase2").

    Returns:
        Dict combining model configuration and parameter statistics.
    """
    model_name = MODEL_NAME if model_name is None else model_name

    parameter_stats = get_model_parameter_stats(model)
    model_config = get_model_config(model)

    return {
        "model_name": model_name,
        "phase": phase,
        "model": model_config,
        "parameters": parameter_stats,
    }


# --------------------------------------------
# Save Model Report
# --------------------------------------------

def save_model_report(
    report: dict,
    save_dir: str | None = None,
) -> str:
    """
    Save a structured model report as JSON.

    model_name and phase are read from the report itself to keep the
    saved filename consistent with the report.

    Args:
        report: Dict produced by `build_model_report`.
        save_dir: Directory to save the report into; defaults to the
            REPORTS_DIR config variable if not provided.

    Returns:
        Path to the saved JSON report.
    """
    save_dir = REPORTS_DIR if save_dir is None else save_dir

    os.makedirs(save_dir, exist_ok=True)

    save_path = os.path.join(
        save_dir,
        f"{report['model_name']}_{report['phase']}_model_report.json",
    )

    with open(save_path, "w", encoding="utf-8") as file:
        json.dump(report, file, indent=4)

    return save_path

## 1.14 Experiment Tracking Utilities (W&B)

### 1.14.1 W&B Project Configuration

In [ ]:
# --------------------------------------------
# 1.14.1 W&B Project Configuration
# --------------------------------------------

# Main W&B project for the CNN experiments.
# High-FLOPs CNN, Low-FLOPs CNN, and MobileNetV2
# experiments will be organized inside this project.
WANDB_PROJECT = "cnn-classification"

# W&B entity / username.
# Keep this configurable rather than hard-coding
# authentication credentials in the notebook.
WANDB_ENTITY = os.getenv("WANDB_ENTITY", "dparsiai-independent")

# Default mode for normal experiment tracking.
# "online"  -> logs experiments to W&B
# "offline" -> stores logs locally for later sync
WANDB_MODE = os.getenv("WANDB_MODE", "online")


# --------------------------------------------
# Shared Internal Helpers
# --------------------------------------------
# Used throughout Section 1.14. Defined here (before 1.14.2)
# since every subsequent W&B utility depends on them.

def _require_active_run(run: "wandb.Run", action: str) -> None:
    """
    Validate that an active W&B Run was provided.

    Parameters
    ----------
    run : wandb.Run
        Run object to validate.

    action : str
        Short description of the action being attempted,
        used in the error message (e.g. "log metrics").

    Raises
    ------
    ValueError
        If run is None.
    """

    if run is None:
        raise ValueError(
            f"A valid, active W&B Run is required to {action}."
        )


def _filter_numeric_metrics(
    metrics: dict[str, Any],
    *,
    namespace: Optional[str] = None,
) -> dict[str, float]:
    """
    Filter a metrics dict down to scalar numeric values.

    Parameters
    ----------
    metrics : dict
        Dictionary of metric names and values. Non-numeric
        values are silently skipped.

    namespace : str, optional
        Optional namespace to prepend to each key.
        Example: namespace="test" turns "accuracy" into
        "test/accuracy".

    Returns
    -------
    dict
        Dictionary containing only numeric values, cast to
        float, with keys optionally namespaced.
    """

    filtered: dict[str, float] = {}

    for key, value in metrics.items():
        if isinstance(value, (int, float, np.integer, np.floating)):
            metric_key = f"{namespace}/{key}" if namespace else str(key)
            filtered[metric_key] = float(value)

    return filtered

### 1.14.2 Run Management

In [ ]:
# --------------------------------------------
# 1.14.2 Run Management
# --------------------------------------------

def build_wandb_run_name(
    model_type: str,
    experiment_type: str,
    run_index: Optional[int] = None,
) -> str:
    """
    Build a consistent W&B run name.

    Parameters
    ----------
    model_type : str
        Short model identifier.
        Examples: "high", "low", "mobilenetv2"

    experiment_type : str
        Type of experiment.
        Examples: "baseline", "retrain", "hpt", "final"

    run_index : int, optional
        Optional numeric index, mainly used for repeated
        HPT / sweep experiments.

    Returns
    -------
    str
        Standardized W&B run name.
    """

    model_type = model_type.strip().lower()
    experiment_type = experiment_type.strip().lower()

    if not model_type or not experiment_type:
        raise ValueError(
            "model_type and experiment_type must be non-empty."
        )

    if run_index is not None:
        return f"{model_type}-{experiment_type}-{run_index:03d}"

    return f"{model_type}-{experiment_type}"


def initialize_wandb_run(
    *,
    model_type: str,
    phase: str,
    experiment_type: str,
    config: Optional[dict[str, Any]] = None,
    run_name: Optional[str] = None,
    run_index: Optional[int] = None,
    extra_tags: Optional[Sequence[str]] = None,
    notes: Optional[str] = None,
    run_id: Optional[str] = None,
) -> "wandb.Run":
    """
    Initialize a W&B experiment run, or resume an existing one.

    This function centralizes W&B run creation so that all
    experiments in the project follow the same structure.

    Parameters
    ----------
    model_type : str
        Short model identifier (see build_wandb_run_name).

    phase : str
        Current project phase.
        Examples: "phase-1", "phase-2", "final"

    experiment_type : str
        Type of experiment.

    config : dict, optional
        Experiment configuration / hyperparameters, typically
        produced by build_wandb_config(). Ignored when run_id
        is provided (the resumed run already has its config).

    run_name : str, optional
        Custom W&B run name. If omitted, a standardized name
        is generated. Ignored when run_id is provided.

    run_index : int, optional
        Numeric run index, mainly useful for HPT runs.

    extra_tags : sequence of str, optional
        Additional tags. Ignored when run_id is provided.

    notes : str, optional
        Human-readable description of the experiment.
        Ignored when run_id is provided.

    run_id : str, optional
        Existing W&B Run ID. When provided, that run is resumed
        instead of creating a new one — used to reconnect to the
        same run after an interruption (e.g. a Colab runtime
        reset), so later logging (evaluation, error analysis)
        attaches to the original run rather than a fresh, empty
        one.

    Returns
    -------
    wandb.Run
        Active W&B Run object.

    Notes
    -----
    reinit=True allows re-running the initialization cell
    multiple times in a notebook session without manually
    finishing the previous run first. This does not apply to
    the resume path, which always reconnects to run_id.
    """

    if run_id is not None:
        run = wandb.init(
            project=WANDB_PROJECT,
            entity=WANDB_ENTITY,
            id=run_id,
            resume="must",
            mode=WANDB_MODE,
        )
        return run

    if run_name is None:
        run_name = build_wandb_run_name(
            model_type=model_type,
            experiment_type=experiment_type,
            run_index=run_index,
        )

    tags = build_wandb_tags(
        model_type=model_type,
        phase=phase,
        experiment_type=experiment_type,
        extra_tags=extra_tags,
    )

    run = wandb.init(
        project=WANDB_PROJECT,
        entity=WANDB_ENTITY,
        name=run_name,
        tags=tags,
        notes=notes,
        config=config,
        mode=WANDB_MODE,
        reinit=True,
    )

    return run


def finish_wandb_run(run: Optional["wandb.Run"] = None) -> None:
    """
    Finish the active W&B run cleanly.

    Parameters
    ----------
    run : wandb.Run, optional
        Active W&B Run object.

    Notes
    -----
    Passing the run explicitly keeps the function usable
    in both notebook and script-based workflows.
    """

    if run is not None:
        run.finish()
    else:
        wandb.finish()


def save_wandb_run_state(
    run: "wandb.Run",
    path: str,
) -> None:
    """
    Save essential W&B Run identity to a local JSON file.

    Parameters
    ----------
    run : wandb.Run
        Active W&B Run.

    path : str
        Destination JSON file path.

    Notes
    -----
    Only the run's identity is saved (id/name/project/entity),
    not its config or metrics — those already live on W&B
    itself and are reloaded automatically when the run is
    resumed via its id.
    """

    _require_active_run(
        run,
        "save W&B run state",
    )

    state = {
        "run_id": run.id,
        "run_name": run.name,
        "project": run.project,
        "entity": run.entity,
    }

    with open(path, "w", encoding="utf-8") as file:
        json.dump(state, file, indent=4)


def load_wandb_run_state(path: str) -> dict[str, str]:
    """
    Load a previously saved W&B Run identity from disk.

    Parameters
    ----------
    path : str
        Path to the saved run-state JSON file.

    Returns
    -------
    dict
        Stored run identity, containing at least "run_id".

    Raises
    ------
    FileNotFoundError
        If the state file does not exist.

    ValueError
        If the state file does not contain a valid run_id.
    """

    if not os.path.isfile(path):
        raise FileNotFoundError(
            f"W&B run state file was not found: {path}"
        )

    with open(path, "r", encoding="utf-8") as file:
        state = json.load(file)

    if not state.get("run_id"):
        raise ValueError(
            "W&B run state does not contain a valid run_id."
        )

    return state

### 1.14.3 Experiment Configuration

In [ ]:
# --------------------------------------------
# 1.14.3 Experiment Configuration
# --------------------------------------------

def build_wandb_config(
    *,
    model_name: str,
    model_type: str,
    image_size: tuple[int, int],
    image_channels: int,
    num_classes: int,
    batch_size: int,
    epochs: int,
    optimizer_name: str,
    learning_rate: float,
    loss_name: str,
    augmentation_config: Optional[dict[str, Any]] = None,
    class_imbalance_config: Optional[dict[str, Any]] = None,
    extra_config: Optional[dict[str, Any]] = None,
) -> dict[str, Any]:
    """
    Build a standardized configuration dictionary for W&B.

    Parameters
    ----------
    model_name : str
        Human-readable model name.
        Example: "High-FLOPs Custom CNN"

    model_type : str
        Short model identifier used elsewhere in 1.14.
        Examples: "high", "low", "mobilenetv2"

    image_size : tuple of int
        Input image spatial dimensions.
        Example: (64, 64)

    image_channels : int
        Number of input image channels.

    num_classes : int
        Number of target classes.

    batch_size : int
        Training batch size.

    epochs : int
        Number of training epochs.

    optimizer_name : str
        Optimizer name.
        Example: "Adam"

    learning_rate : float
        Initial learning rate.

    loss_name : str
        Loss function name.

    augmentation_config : dict, optional
        Data augmentation configuration.

    class_imbalance_config : dict, optional
        Class imbalance handling configuration.

    extra_config : dict, optional
        Additional experiment-specific configuration.

    Returns
    -------
    dict
        Standardized W&B configuration dictionary.
    """

    config: dict[str, Any] = {
        # ------------------------------------
        # Model
        # ------------------------------------
        "model_name": model_name,
        "model_type": model_type,

        # ------------------------------------
        # Input / Dataset
        # ------------------------------------
        "image_size": list(image_size),
        "image_channels": image_channels,
        "num_classes": num_classes,

        # ------------------------------------
        # Training
        # ------------------------------------
        "batch_size": batch_size,
        "epochs": epochs,

        # ------------------------------------
        # Optimization
        # ------------------------------------
        "optimizer_name": optimizer_name,
        "learning_rate": learning_rate,

        # ------------------------------------
        # Loss
        # ------------------------------------
        "loss_name": loss_name,
    }

    # ----------------------------------------
    # Data Augmentation
    # ----------------------------------------

    if augmentation_config is not None:
        config["augmentation"] = augmentation_config

    # ----------------------------------------
    # Class Imbalance
    # ----------------------------------------

    if class_imbalance_config is not None:
        config["class_imbalance"] = class_imbalance_config

    # ----------------------------------------
    # Additional Configuration
    # ----------------------------------------

    if extra_config is not None:
        config.update(extra_config)

    return config


def update_wandb_config(
    config: dict[str, Any],
    updates: Optional[dict[str, Any]] = None,
) -> dict[str, Any]:
    """
    Update an existing W&B configuration dictionary.

    Parameters
    ----------
    config : dict
        Existing configuration dictionary.

    updates : dict, optional
        New or updated configuration values.

    Returns
    -------
    dict
        Updated configuration dictionary.

    Notes
    -----
    The original configuration dictionary is not modified.

    This is a shallow update. Nested values are replaced as a whole
    rather than recursively merged.
    """

    updated_config = config.copy()

    if updates is not None:
        updated_config.update(updates)

    return updated_config

### 1.14.4 Tags & Run Metadata

In [ ]:
# --------------------------------------------
# 1.14.4 Tags & Run Metadata
# --------------------------------------------

def _normalize_experiment_identity(
    model_type: str,
    phase: str,
    experiment_type: str,
) -> tuple[str, str, str]:
    """
    Normalize and validate the core experiment identity fields.

    Parameters
    ----------
    model_type : str
        Short model identifier.

    phase : str
        Project phase.

    experiment_type : str
        Experiment type.

    Returns
    -------
    tuple of str
        Normalized (model_type, phase, experiment_type).

    Raises
    ------
    ValueError
        If model_type, phase, or experiment_type is empty after stripping.
    """

    model_type = model_type.strip().lower()
    phase = phase.strip().lower()
    experiment_type = experiment_type.strip().lower()

    if not model_type or not phase or not experiment_type:
        raise ValueError(
            "model_type, phase and experiment_type must be non-empty."
        )

    return model_type, phase, experiment_type


def build_wandb_tags(
    model_type: str,
    phase: str,
    experiment_type: str,
    extra_tags: Optional[Sequence[str]] = None,
) -> list[str]:
    """
    Build a standardized set of W&B tags.

    Parameters
    ----------
    model_type : str
        Short model identifier.
        Examples:
            "high"
            "low"
            "mobilenetv2"

    phase : str
        Project phase.
        Examples:
            "phase-1"
            "phase-2"
            "final"

    experiment_type : str
        Experiment type.
        Examples:
            "baseline"
            "retraining"
            "hpt"
            "final"

    extra_tags : sequence of str, optional
        Additional custom tags.

    Returns
    -------
    list of str
        Standardized and deduplicated tag list.

    Raises
    ------
    ValueError
        If model_type, phase, or experiment_type is empty after stripping.
    """

    # Normalize and validate required values.
    model_type, phase, experiment_type = _normalize_experiment_identity(
        model_type, phase, experiment_type
    )

    # Core tags are always included.
    tags = [
        model_type,
        phase,
        experiment_type,
    ]

    # Add optional custom tags.
    if extra_tags is not None:
        tags.extend(
            tag.strip().lower()
            for tag in extra_tags
            if isinstance(tag, str) and tag.strip()
        )

    # Preserve insertion order while removing duplicates.
    return list(dict.fromkeys(tags))


def build_wandb_run_metadata(
    *,
    model_type: str,
    phase: str,
    experiment_type: str,
    run_index: Optional[int] = None,
) -> dict[str, Any]:
    """
    Build standardized metadata that identifies a W&B Run.

    Parameters
    ----------
    model_type : str
        Short model identifier.

    phase : str
        Project phase.

    experiment_type : str
        Type of experiment.

    run_index : int, optional
        Numeric run index, mainly useful for HPT experiments.

    Returns
    -------
    dict
        Standardized Run metadata.

    Raises
    ------
    ValueError
        If model_type, phase, or experiment_type is empty after stripping.

    Notes
    -----
    These values describe the identity and context of the
    experiment. They are kept separate from performance metrics.
    """

    # Normalize and validate required metadata.
    model_type, phase, experiment_type = _normalize_experiment_identity(
        model_type, phase, experiment_type
    )

    metadata: dict[str, Any] = {
        "model_type": model_type,
        "experiment_phase": phase,
        "experiment_type": experiment_type,
    }

    # Add run index only when explicitly provided.
    if run_index is not None:
        metadata["run_index"] = int(run_index)

    return metadata

### 1.14.5 Training Metric Logging

In [ ]:
# --------------------------------------------
# 1.14.5 Training Metric Logging
# --------------------------------------------


def _flatten_nested_metrics(
    metrics: dict[str, Any],
    *,
    namespace: Optional[str] = None,
) -> dict[str, float]:
    """
    Flatten nested metric dictionaries into scalar W&B metrics.

    Parameters
    ----------
    metrics : dict
        Metric dictionary that may contain nested dictionaries.

    namespace : str, optional
        Optional top-level namespace to prepend to metric keys.

    Returns
    -------
    dict
        Flattened dictionary containing only numeric leaf values.

    Examples
    --------
    {
        "accuracy": 0.95,
        "macro": {
            "precision": 0.96,
            "recall": 0.95,
            "f1_score": 0.95,
        }
    }

    becomes:

    {
        "accuracy": 0.95,
        "macro/precision": 0.96,
        "macro/recall": 0.95,
        "macro/f1_score": 0.95,
    }
    """

    flat: dict[str, Any] = {}

    def _flatten(
        current: dict[str, Any],
        prefix: str = "",
    ) -> None:

        for key, value in current.items():

            current_key = (
                f"{prefix}/{key}"
                if prefix
                else str(key)
            )

            if isinstance(value, dict):
                _flatten(
                    value,
                    prefix=current_key,
                )
            else:
                flat[current_key] = value

    _flatten(metrics)

    return _filter_numeric_metrics(
        flat,
        namespace=namespace,
    )


def _split_train_val_metrics(
    logs: dict[str, Any],
) -> tuple[dict[str, Any], dict[str, Any]]:
    """
    Split a Keras logs dict into train/validation metric dicts.

    Parameters
    ----------
    logs : dict
        Metrics dict as produced by Keras (e.g. the `logs` argument
        in `on_epoch_end`, or one epoch's slice of `history.history`).
        Validation metrics are prefixed with "val_".

    Returns
    -------
    tuple of dict
        (train_metrics, val_metrics), with the "val_" prefix
        stripped from validation metric keys.
    """

    train_metrics = {
        key: value for key, value in logs.items()
        if not key.startswith("val_")
    }

    val_metrics = {
        key[4:]: value for key, value in logs.items()
        if key.startswith("val_")
    }

    return train_metrics, val_metrics


class WandBEpochLogger(tf.keras.callbacks.Callback):
    """
    Keras callback that logs training/validation metrics to
    W&B in real time, at the end of every epoch.

    This is the primary metric-logging path for the project
    (as opposed to logging only after model.fit() completes),
    so that:
      - a crash / disconnect mid-training does not lose all
        metrics logged up to that point, and
      - W&B Sweep early-termination (Hyperband, in Phase 2)
        can observe metrics epoch-by-epoch.

    Parameters
    ----------
    run : wandb.Run
        Active W&B Run to log into.
    """

    def __init__(self, run: "wandb.Run"):
        super().__init__()
        _require_active_run(run, "attach WandBEpochLogger")
        self.run = run

    def on_epoch_end(
        self,
        epoch: int,
        logs: Optional[dict[str, Any]] = None,
    ) -> None:

        if not logs:
            return

        train_logs, val_logs = _split_train_val_metrics(logs)

        step = epoch + 1

        train_logged = _filter_numeric_metrics(
            train_logs,
            namespace="train",
        )

        val_logged = _filter_numeric_metrics(
            val_logs,
            namespace="val",
        )

        self.run.log(
            {
                **train_logged,
                **val_logged,
                "epoch": step,
            },
            step=step,
        )


def log_wandb_metrics(
    run: "wandb.Run",
    metrics: dict[str, Any],
    *,
    step: Optional[int] = None,
    namespace: Optional[str] = None,
) -> None:
    """
    Log arbitrary scalar metrics to W&B.

    Parameters
    ----------
    run : wandb.Run
        Active W&B Run.

    metrics : dict
        Dictionary containing metric names and values.
        Non-numeric values are silently skipped.

    step : int, optional
        Explicit W&B step.

    namespace : str, optional
        Optional metric namespace.
        Example: namespace="test" turns "accuracy" into
        "test/accuracy".
    """

    _require_active_run(run, "log metrics")

    logged_metrics = _filter_numeric_metrics(
        metrics,
        namespace=namespace,
    )

    if not logged_metrics:
        return

    run.log(
        logged_metrics,
        step=step,
    )


def get_training_history_dict(
    history,
) -> dict[str, list]:
    """
    Convert a Keras History object into a standard dictionary.

    Parameters
    ----------
    history : tf.keras.callbacks.History
        Training history returned by model.fit().

    Returns
    -------
    dict
        Dictionary containing training metrics for each epoch.
    """

    if history is None:
        raise ValueError(
            "A valid Keras History object is required."
        )

    return {
        key: list(values)
        for key, values in history.history.items()
    }


def log_wandb_training_history(
    run: "wandb.Run",
    history,
) -> None:
    """
    Log a complete Keras training history to W&B, epoch by epoch.

    Parameters
    ----------
    run : wandb.Run
        Active W&B Run.

    history : tf.keras.callbacks.History
        Training history returned by model.fit().

    Notes
    -----
    FALLBACK PATH ONLY. The primary logging mechanism is
    WandBEpochLogger, attached as a callback during model.fit(),
    which logs in real time and survives mid-training crashes.
    Use this function only when training was run without that
    callback and only a completed `history` object is available
    (e.g. an offline script, or a debugging session).
    """

    _require_active_run(
        run,
        "log training history",
    )

    history_dict = get_training_history_dict(history)

    if not history_dict:
        return

    num_epochs = len(
        next(iter(history_dict.values()))
    )

    for epoch in range(num_epochs):

        epoch_values = {
            key: values[epoch]
            for key, values in history_dict.items()
            if epoch < len(values)
        }

        train_values, val_values = _split_train_val_metrics(
            epoch_values
        )

        step = epoch + 1

        train_logged = _filter_numeric_metrics(
            train_values,
            namespace="train",
        )

        val_logged = _filter_numeric_metrics(
            val_values,
            namespace="val",
        )

        run.log(
            {
                **train_logged,
                **val_logged,
                "epoch": step,
            },
            step=step,
        )


def log_wandb_training_summary(
    run: "wandb.Run",
    history,
) -> None:
    """
    Log final and best training metrics to the W&B Run Summary.

    Parameters
    ----------
    run : wandb.Run
        Active W&B Run.

    history : tf.keras.callbacks.History
        Training history returned by model.fit().

    Notes
    -----
    For each tracked metric, records final_<metric>. For
    performance metrics (everything except learning_rate/lr),
    also records best_<metric>: for metrics containing "loss",
    the minimum value is treated as best; otherwise the maximum
    is used.

    This heuristic assumes higher-is-better for all non-loss
    performance metrics (accuracy, precision, recall, F1).
    learning_rate / lr are excluded from "best_" entirely —
    they are a changing hyperparameter, not a performance
    metric, so "best" has no meaningful interpretation for them
    (e.g. with ReduceLROnPlateau, the learning rate only ever
    stays flat or decreases, so "max" would trivially always be
    the initial value).
    """

    _require_active_run(
        run,
        "log training summary",
    )

    history_dict = get_training_history_dict(history)

    if not history_dict:
        return

    non_performance_metrics = {"learning_rate", "lr"}

    for metric_name, values in history_dict.items():

        numeric_values = [
            float(v)
            for v in values
            if isinstance(
                v,
                (
                    int,
                    float,
                    np.integer,
                    np.floating,
                ),
            )
        ]

        if not numeric_values:
            continue

        if metric_name.startswith("val_"):
            metric_key = metric_name[4:]
            namespace = "val"
        else:
            metric_key = metric_name
            namespace = "train"

        run.summary[
            f"{namespace}/final_{metric_key}"
        ] = numeric_values[-1]

        if metric_key.lower() in non_performance_metrics:
            continue

        best_value = (
            min(numeric_values)
            if "loss" in metric_key.lower()
            else max(numeric_values)
        )

        run.summary[
            f"{namespace}/best_{metric_key}"
        ] = best_value

### 1.14.6 Run Summary Management

In [ ]:
# --------------------------------------------
# 1.14.6 Run Summary Management
# --------------------------------------------

def log_wandb_parameter_summary(
    run: "wandb.Run",
    parameter_stats: dict[str, Any],
) -> None:
    """
    Log parameter statistics to the W&B Run Summary.

    Parameters
    ----------
    run : wandb.Run
        Active W&B Run.

    parameter_stats : dict
        Parameter statistics generated by
        Model Reporting Utilities (1.13).
    """

    _require_active_run(run, "log parameter statistics")

    summary_mapping = {
        "model/total_params": "total_params",
        "model/trainable_params": "trainable_params",
        "model/non_trainable_params": "non_trainable_params",
    }

    for wandb_key, stats_key in summary_mapping.items():
        if stats_key in parameter_stats:
            run.summary[wandb_key] = int(parameter_stats[stats_key])


def log_wandb_complexity_summary(
    run: "wandb.Run",
    complexity_stats: dict[str, Any],
) -> None:
    """
    Log model complexity statistics to the W&B Run Summary.

    Parameters
    ----------
    run : wandb.Run
        Active W&B Run.

    complexity_stats : dict
        Complexity statistics (e.g. FLOPs) generated by the
        existing Model Reporting Utilities.

    Notes
    -----
    Only scalar numeric values are added to the Run Summary.
    """

    _require_active_run(run, "log complexity statistics")

    numeric_stats = _filter_numeric_metrics(complexity_stats)

    for key, value in numeric_stats.items():
        run.summary[f"model/{key}"] = value





### 1.14.7 Model Metadata Logging

In [ ]:
# --------------------------------------------
# 1.14.7 Model Metadata Logging
# --------------------------------------------

def build_wandb_model_metadata(
    model,
    *,
    model_name: str,
    model_type: str,
    input_shape: Optional[tuple] = None,
    num_classes: Optional[int] = None,
    parameter_stats: Optional[dict[str, Any]] = None,
    complexity_stats: Optional[dict[str, Any]] = None,
    extra_metadata: Optional[dict[str, Any]] = None,
) -> dict[str, Any]:
    """
    Build standardized model metadata for W&B.

    Parameters
    ----------
    model : tf.keras.Model
        Built Keras model.

    model_name : str
        Human-readable model name.

    model_type : str
        Short model identifier.
        Examples: "high", "low", "mobilenetv2"

    input_shape : tuple, optional
        Model input shape. Example: (64, 64, 3)

    num_classes : int, optional
        Number of output classes.

    parameter_stats : dict, optional
        Parameter statistics from Model Reporting Utilities (1.13).

    complexity_stats : dict, optional
        Model complexity information (e.g. FLOPs).

    extra_metadata : dict, optional
        Additional model metadata.

    Returns
    -------
    dict
        Standardized model metadata dictionary.
    """

    metadata: dict[str, Any] = {
        "model/name": model_name,
        "model/type": model_type,
        "model/architecture": model.__class__.__name__,
        "model/layers": len(model.layers),
    }

    if input_shape is not None:
        metadata["model/input_shape"] = list(input_shape)

    if num_classes is not None:
        metadata["model/num_classes"] = int(num_classes)

    if parameter_stats is not None:
        for key in ("total_params", "trainable_params", "non_trainable_params"):
            if key in parameter_stats:
                metadata[f"model/{key}"] = int(parameter_stats[key])

    if complexity_stats is not None:
        for key, value in complexity_stats.items():
            metadata_key = f"model/{key}"
            if isinstance(value, (int, float, np.integer, np.floating)):
                metadata[metadata_key] = float(value)
            else:
                metadata[metadata_key] = value

    if extra_metadata is not None:
        metadata.update(extra_metadata)

    return metadata


def log_wandb_model_metadata(run: "wandb.Run", metadata: dict[str, Any]) -> None:
    """
    Log model metadata to the W&B Run config.

    Parameters
    ----------
    run : wandb.Run
        Active W&B Run.

    metadata : dict
        Model metadata produced by build_wandb_model_metadata().
    """

    _require_active_run(run, "log model metadata")
    run.config.update(metadata)

### 1.14.8 Evaluation Logging

In [ ]:
# --------------------------------------------
# 1.14.8 Evaluation Logging
# --------------------------------------------

def _normalize_split(split: str) -> str:
    """
    Normalize and validate a dataset split identifier.

    Parameters
    ----------
    split : str
        Dataset split associated with the evaluation.
        Examples: "train", "validation", "test".

    Returns
    -------
    str
        Normalized split string.

    Raises
    ------
    ValueError
        If split is empty after stripping.
    """

    split = split.strip().lower()

    if not split:
        raise ValueError(
            "split must be a non-empty string."
        )

    return split


def log_wandb_evaluation_metrics(
    run: "wandb.Run",
    metrics: dict[str, Any],
    *,
    split: str = "test",
    step: Optional[int] = None,
) -> None:
    """
    Log evaluation metrics to W&B.

    Parameters
    ----------
    run : wandb.Run
        Active W&B Run.

    metrics : dict
        Evaluation metrics produced by the existing
        Evaluation Utilities. Nested metric dictionaries
        are supported and flattened automatically.

    split : str, optional
        Dataset split associated with the evaluation.
        Examples: "train", "validation", "test".

    step : int, optional
        Optional explicit W&B step.

    Raises
    ------
    ValueError
        If split is empty after stripping.

    Notes
    -----
    Numeric leaf values from nested dictionaries are logged.

    Metrics are namespaced by the evaluation split.

    Example:
        {
            "accuracy": 0.95,
            "macro": {
                "precision": 0.96,
                "f1_score": 0.95,
            }
        }

    with split="validation" becomes:
        "validation/accuracy": 0.95
        "validation/macro/precision": 0.96
        "validation/macro/f1_score": 0.95
    """

    _require_active_run(
        run,
        "log evaluation metrics",
    )

    split = _normalize_split(split)

    logged_metrics = _flatten_nested_metrics(
        metrics,
        namespace=split,
    )

    if not logged_metrics:
        return

    run.log(
        logged_metrics,
        step=step,
    )


def log_wandb_evaluation_summary(
    run: "wandb.Run",
    metrics: dict[str, Any],
    *,
    split: str = "test",
) -> None:
    """
    Log final evaluation metrics to the W&B Run Summary.

    Parameters
    ----------
    run : wandb.Run
        Active W&B Run.

    metrics : dict
        Evaluation metrics produced by the existing
        Evaluation Utilities. Nested metric dictionaries
        are supported and flattened automatically.

    split : str, optional
        Dataset split associated with the evaluation.
        Examples: "train", "validation", "test".

    Raises
    ------
    ValueError
        If split is empty after stripping.

    Notes
    -----
    Numeric leaf values from nested dictionaries are added
    to the Run Summary.

    Metrics are namespaced by the evaluation split.

    Example:
        {
            "accuracy": 0.95,
            "macro": {
                "precision": 0.96,
                "f1_score": 0.95,
            }
        }

    with split="test" becomes:
        "test/accuracy": 0.95
        "test/macro/precision": 0.96
        "test/macro/f1_score": 0.95
    """

    _require_active_run(
        run,
        "log evaluation summary",
    )

    split = _normalize_split(split)

    numeric_metrics = _flatten_nested_metrics(
        metrics,
        namespace=split,
    )

    for key, value in numeric_metrics.items():
        run.summary[key] = value

### 1.14.9 Error Analysis Logging

In [ ]:
# --------------------------------------------
# 1.14.9 Error Analysis Logging
# --------------------------------------------

def log_wandb_error_analysis_metrics(
    run: "wandb.Run",
    metrics: dict[str, Any],
) -> None:
    """
    Log error analysis metrics to W&B.

    Parameters
    ----------
    run : wandb.Run
        Active W&B Run.

    metrics : dict
        Error analysis metrics produced by the existing
        Error Analysis Utilities (1.11). Nested metric
        dictionaries (e.g. per-class breakdowns) are
        supported and flattened automatically.

        Example:
            {
                "total_samples": 10000,
                "total_errors": 350,
                "overall_error_rate": 0.035,
                "mean_confidence": 0.91,
                "mean_error_confidence": 0.72,
                "per_class": {
                    "Forest": {
                        "total_samples": 500,
                        "errors": 12,
                        "error_rate": 0.024,
                    },
                    ...
                },
            }

        becomes:
            "error_analysis/total_samples": 10000
            "error_analysis/total_errors": 350
            "error_analysis/overall_error_rate": 0.035
            "error_analysis/mean_confidence": 0.91
            "error_analysis/mean_error_confidence": 0.72
            "error_analysis/per_class/Forest/total_samples": 500
            "error_analysis/per_class/Forest/errors": 12
            "error_analysis/per_class/Forest/error_rate": 0.024

    Notes
    -----
    Numeric leaf values from nested dictionaries are logged.

    Metrics are stored under the "error_analysis/" namespace.
    """

    _require_active_run(
        run,
        "log error analysis metrics",
    )

    logged_metrics = _flatten_nested_metrics(
        metrics,
        namespace="error_analysis",
    )

    if not logged_metrics:
        return

    run.log(logged_metrics)


def log_wandb_error_analysis_summary(
    run: "wandb.Run",
    metrics: dict[str, Any],
) -> None:
    """
    Log error analysis metrics to the W&B Run Summary.

    Parameters
    ----------
    run : wandb.Run
        Active W&B Run.

    metrics : dict
        Error analysis metrics produced by the existing
        Error Analysis Utilities (1.11). Nested metric
        dictionaries (e.g. per-class breakdowns) are
        supported and flattened automatically.

    Notes
    -----
    Numeric leaf values from nested dictionaries are added
    to the Run Summary.

    Metrics are stored under the "error_analysis/" namespace.
    """

    _require_active_run(
        run,
        "log error analysis summary",
    )

    numeric_metrics = _flatten_nested_metrics(
        metrics,
        namespace="error_analysis",
    )

    for key, value in numeric_metrics.items():
        run.summary[key] = value

### 1.14.10 Tables & Media Logging

In [ ]:
# --------------------------------------------
# 1.14.10 Tables & Media Logging
# --------------------------------------------

def _require_nonempty_name(name: str, param_name: str) -> str:
    """
    Strip and validate a non-empty identifier name.

    Parameters
    ----------
    name : str
        Name to validate.

    param_name : str
        Parameter name used in the error message.

    Returns
    -------
    str
        Stripped name.

    Raises
    ------
    ValueError
        If name is empty after stripping.
    """

    name = name.strip()

    if not name:
        raise ValueError(f"{param_name} must be a non-empty string.")

    return name


def log_wandb_table(
    run: "wandb.Run",
    table_name: str,
    data: Optional[Any] = None,
    *,
    columns: Optional[Sequence[str]] = None,
    rows: Optional[Sequence[Sequence[Any]]] = None,
) -> None:
    """
    Log a W&B Table to the active run.

    Parameters
    ----------
    run : wandb.Run
        Active W&B Run.

    table_name : str
        Name used to identify the table in W&B.

    data : optional
        Existing wandb.Table instance or table-compatible data.

    columns : sequence of str, optional
        Column names used when creating a new table.

    rows : sequence of sequences, optional
        Table rows used when creating a new table.

    Raises
    ------
    ValueError
        If table_name is empty or if insufficient table
        construction arguments are provided.

    Notes
    -----
    This utility supports both an existing wandb.Table and
    creation of a new table from columns and rows.
    """

    _require_active_run(
        run,
        "log W&B table",
    )

    table_name = _require_nonempty_name(table_name, "table_name")

    if isinstance(data, wandb.Table):
        table = data

    elif columns is not None:
        table = wandb.Table(
            columns=list(columns),
        )

        if rows is not None:
            for row in rows:
                table.add_data(*row)

    else:
        raise ValueError(
            "Provide an existing wandb.Table or "
            "columns to create a new table."
        )

    run.log({
        table_name: table,
    })


def log_wandb_media(
    run: "wandb.Run",
    media_name: str,
    media: Any,
    *,
    step: Optional[int] = None,
) -> None:
    """
    Log a media object to the active W&B Run.

    Parameters
    ----------
    run : wandb.Run
        Active W&B Run.

    media_name : str
        Name used to identify the media in W&B.

    media : Any
        W&B-compatible media object.
        Examples include:
            - wandb.Image
            - wandb.Video
            - wandb.Audio
            - wandb.Html
            - wandb.Object3D

    step : int, optional
        Optional explicit W&B step.

    Raises
    ------
    ValueError
        If media_name is empty.
    """

    _require_active_run(
        run,
        "log W&B media",
    )

    media_name = _require_nonempty_name(media_name, "media_name")

    payload = {
        media_name: media,
    }

    run.log(payload, step=step)


def log_wandb_media_collection(
    run: "wandb.Run",
    media_name: str,
    media_items: Sequence[Any],
    *,
    step: Optional[int] = None,
) -> None:
    """
    Log a collection of W&B-compatible media objects.

    Parameters
    ----------
    run : wandb.Run
        Active W&B Run.

    media_name : str
        Name used to identify the media collection in W&B.

    media_items : sequence
        Sequence of W&B-compatible media objects.

    step : int, optional
        Optional explicit W&B step.

    Raises
    ------
    ValueError
        If media_name is empty or media_items is empty.
    """

    _require_active_run(
        run,
        "log W&B media collection",
    )

    media_name = _require_nonempty_name(media_name, "media_name")

    if not media_items:
        raise ValueError(
            "media_items must contain at least one item."
        )

    payload = {
        media_name: list(media_items),
    }

    run.log(payload, step=step)

### 1.14.11 Reports

In [ ]:
# --------------------------------------------
# 1.14.11 Reports
# --------------------------------------------

def log_wandb_report_link(
    run: "wandb.Run",
    report_name: str,
    report_url: str,
) -> None:
    """
    Log a W&B report link to the active run.

    Parameters
    ----------
    run : wandb.Run
        Active W&B Run.

    report_name : str
        Name used to identify the report.

    report_url : str
        URL of the W&B report.

    Raises
    ------
    ValueError
        If report_name or report_url is empty.

    Notes
    -----
    The report link is stored in the Run Summary so that it
    remains available as persistent experiment metadata.
    """

    _require_active_run(
        run,
        "log W&B report link",
    )

    report_name = _require_nonempty_name(
        report_name,
        "report_name",
    )

    report_url = _require_nonempty_name(
        report_url,
        "report_url",
    )

    run.summary[f"reports/{report_name}"] = report_url

### 1.14.12 Artifact Management

In [ ]:
# --------------------------------------------
# 1.14.12 Artifact Management
# --------------------------------------------

def _require_artifact(artifact: "wandb.Artifact") -> None:
    """
    Validate that a W&B Artifact object was provided.

    Parameters
    ----------
    artifact : wandb.Artifact
        Artifact object to validate.

    Raises
    ------
    ValueError
        If artifact is None.
    """

    if artifact is None:
        raise ValueError(
            "A valid W&B artifact is required."
        )


def build_wandb_artifact(
    artifact_name: str,
    *,
    artifact_type: str,
    description: Optional[str] = None,
    metadata: Optional[dict[str, Any]] = None,
) -> "wandb.Artifact":
    """
    Build a W&B Artifact object.

    Parameters
    ----------
    artifact_name : str
        Name used to identify the artifact.

    artifact_type : str
        Artifact type.
        Examples: "model", "dataset", "checkpoint", "config".

    description : str, optional
        Human-readable artifact description.

    metadata : dict, optional
        Additional artifact metadata.

    Returns
    -------
    wandb.Artifact
        Configured W&B Artifact object.
    """

    artifact_name = _require_nonempty_name(
        artifact_name,
        "artifact_name",
    )

    artifact_type = _require_nonempty_name(
        artifact_type,
        "artifact_type",
    )

    artifact = wandb.Artifact(
        name=artifact_name,
        type=artifact_type,
        description=description,
        metadata=metadata,
    )

    return artifact


def add_wandb_artifact_file(
    artifact: "wandb.Artifact",
    file_path: str,
    *,
    name: Optional[str] = None,
) -> None:
    """
    Add a file to a W&B Artifact.

    Parameters
    ----------
    artifact : wandb.Artifact
        Artifact object created by build_wandb_artifact().

    file_path : str
        Local path to the file.

    name : str, optional
        Optional artifact-internal filename.

    Raises
    ------
    ValueError
        If the artifact is invalid or file_path is empty.
    """

    _require_artifact(artifact)

    file_path = _require_nonempty_name(
        file_path,
        "file_path",
    )

    artifact.add_file(
        local_path=file_path,
        name=name,
    )


def add_wandb_artifact_directory(
    artifact: "wandb.Artifact",
    directory_path: str,
    *,
    name: Optional[str] = None,
) -> None:
    """
    Add a directory to a W&B Artifact.

    Parameters
    ----------
    artifact : wandb.Artifact
        Artifact object created by build_wandb_artifact().

    directory_path : str
        Local directory path.

    name : str, optional
        Optional artifact-internal directory name.

    Raises
    ------
    ValueError
        If the artifact is invalid or directory_path is empty.
    """

    _require_artifact(artifact)

    directory_path = _require_nonempty_name(
        directory_path,
        "directory_path",
    )

    artifact.add_dir(
        local_path=directory_path,
        name=name,
    )


def log_wandb_artifact(
    run: "wandb.Run",
    artifact: "wandb.Artifact",
    *,
    aliases: Optional[Sequence[str]] = None,
) -> None:
    """
    Log a W&B Artifact to the active run.

    Parameters
    ----------
    run : wandb.Run
        Active W&B Run.

    artifact : wandb.Artifact
        Artifact to log.

    aliases : sequence of str, optional
        Optional aliases for the artifact version.
        Examples: ["latest"], ["baseline", "best"].

    Raises
    ------
    ValueError
        If the run or artifact is invalid.
    """

    _require_active_run(
        run,
        "log W&B artifact",
    )

    _require_artifact(artifact)

    run.log_artifact(
        artifact,
        aliases=list(aliases) if aliases is not None else None,
    )

### 1.14.13 Model Handoff

In [ ]:
# --------------------------------------------
# 1.14.13 Model Handoff
# --------------------------------------------

def log_wandb_model_artifact(
    run: "wandb.Run",
    model_path: str,
    *,
    artifact_name: str,
    artifact_type: str = "model",
    aliases: Optional[Sequence[str]] = None,
    metadata: Optional[dict[str, Any]] = None,
) -> None:
    """
    Package a saved model as a W&B Artifact and log it
    to the active run.

    Parameters
    ----------
    run : wandb.Run
        Active W&B Run.

    model_path : str
        Local path to the saved model file or directory.

    artifact_name : str
        Name of the model artifact.

    artifact_type : str, optional
        W&B artifact type. Defaults to "model".

    aliases : sequence of str, optional
        Optional aliases for the artifact version.
        Examples: ["latest"], ["baseline"], ["best"].

    metadata : dict, optional
        Additional model artifact metadata.

    Raises
    ------
    ValueError
        If the run, model path, artifact name, or artifact
        type is invalid.

    Notes
    -----
    This utility provides the model handoff path from a local
    saved model to a versioned W&B Model Artifact.
    """

    _require_active_run(
        run,
        "log model artifact",
    )

    model_path = _require_nonempty_name(
        model_path,
        "model_path",
    )

    is_dir = os.path.isdir(model_path)
    is_file = os.path.isfile(model_path)

    if not is_dir and not is_file:
        raise ValueError(
            f"model_path does not exist: {model_path}"
        )

    artifact = build_wandb_artifact(
        artifact_name=artifact_name,
        artifact_type=artifact_type,
        metadata=metadata,
    )

    if is_dir:
        add_wandb_artifact_directory(
            artifact,
            model_path,
        )
    else:
        add_wandb_artifact_file(
            artifact,
            model_path,
        )

    log_wandb_artifact(
        run,
        artifact,
        aliases=aliases,
    )

### 1.14.14 Sweep / HPT Support

In [ ]:
# --------------------------------------------
# 1.14.14 Sweep / HPT Support
# --------------------------------------------

def build_wandb_sweep_config(
    *,
    method: str,
    metric_name: str,
    metric_goal: str,
    parameters: dict[str, Any],
    project: Optional[str] = None,
) -> dict[str, Any]:
    """
    Build a standardized W&B Sweep configuration.

    Parameters
    ----------
    method : str
        Sweep search method.
        Examples: "grid", "random", "bayes".

    metric_name : str
        Metric monitored by the sweep.
        Example: "val/loss".

    metric_goal : str
        Optimization direction.
        Must be either "minimize" or "maximize".

    parameters : dict
        W&B Sweep parameter definitions.

    project : str, optional
        W&B project name. Defaults to WANDB_PROJECT.

    Returns
    -------
    dict
        W&B Sweep configuration.

    Raises
    ------
    ValueError
        If required values are empty or invalid.
    """

    method = _require_nonempty_name(
        method,
        "method",
    ).lower()

    metric_name = _require_nonempty_name(
        metric_name,
        "metric_name",
    )

    metric_goal = _require_nonempty_name(
        metric_goal,
        "metric_goal",
    ).lower()

    if method not in {"grid", "random", "bayes"}:
        raise ValueError(
            "method must be one of: grid, random, bayes."
        )

    if metric_goal not in {"minimize", "maximize"}:
        raise ValueError(
            "metric_goal must be either 'minimize' or 'maximize'."
        )

    if not parameters:
        raise ValueError(
            "parameters must contain at least one sweep parameter."
        )

    sweep_project = (
        WANDB_PROJECT
        if project is None
        else _require_nonempty_name(project, "project")
    )

    return {
        "method": method,
        "metric": {
            "name": metric_name,
            "goal": metric_goal,
        },
        "parameters": parameters,
        "project": sweep_project,
    }

def create_wandb_sweep(
    sweep_config: dict[str, Any],
    *,
    project: Optional[str] = None,
    entity: Optional[str] = None,
) -> str:
    """
    Create a W&B Sweep and return its Sweep ID.

    Parameters
    ----------
    sweep_config : dict
        Sweep configuration produced by
        build_wandb_sweep_config().

    project : str, optional
        W&B project name. Defaults to the project defined
        in the sweep configuration or WANDB_PROJECT.

    entity : str, optional
        W&B entity name (username or team name) under which
        the sweep is created. Defaults to the entity
        configured in the W&B environment or settings.

    Returns
    -------
    str
        W&B Sweep ID.

    Raises
    ------
    ValueError
        If sweep_config is empty.
    """
    
    if not sweep_config:
        raise ValueError(
            "sweep_config must not be empty."
        )

    sweep_project = (
        project
        if project is not None
        else sweep_config.get("project", WANDB_PROJECT)
    )

    sweep_project = _require_nonempty_name(
        sweep_project,
        "project",
    )

    sweep_entity = (
        WANDB_ENTITY
        if entity is None
        else _require_nonempty_name(entity, "entity")
    )

    sweep_config = sweep_config.copy()
    sweep_config.pop("project", None)

    return wandb.sweep(
        sweep=sweep_config,
        entity=sweep_entity,
        project=sweep_project,
    )

## 1.15 Interpretability Utilities

# 2. Phase 1 — Baseline Training & Error Analysis

## 2.1 Residual Block

In [ ]:
# ============================================
# 2.1 Residual Block
# ============================================

def residual_block(
    x: tf.Tensor,
    filters: int,
    stride: int = 1,
    name: str | None = None,
) -> tf.Tensor:
    """
    Apply a standard two-layer residual block (ResNet v1 style).

    Main path: Conv -> BN -> ReLU -> Conv -> BN
    Shortcut path: identity, or a 1x1 projection Conv -> BN when the
    spatial resolution changes (stride != 1) or the channel count
    changes (shortcut channels != filters).
    The two paths are summed and passed through a final ReLU.

    Args:
        x: Input feature map.
        filters: Number of output filters for both conv layers.
        stride: Stride applied to the first conv (and the projection
            shortcut, if used). Use stride=2 to downsample.
        name: Optional prefix for naming sub-layers; if None, layers
            get auto-generated Keras names.

    Returns:
        Output feature map after the residual addition and final ReLU.
    """
    shortcut = x

    # Main path
    x = layers.Conv2D(
        filters=filters,
        kernel_size=3,
        strides=stride,
        padding="same",
        use_bias=False,
        name=None if name is None else f"{name}_conv1",
    )(x)

    x = layers.BatchNormalization(
        name=None if name is None else f"{name}_bn1",
    )(x)

    x = layers.ReLU(
        name=None if name is None else f"{name}_relu1",
    )(x)

    x = layers.Conv2D(
        filters=filters,
        kernel_size=3,
        strides=1,
        padding="same",
        use_bias=False,
        name=None if name is None else f"{name}_conv2",
    )(x)

    x = layers.BatchNormalization(
        name=None if name is None else f"{name}_bn2",
    )(x)

    # Projection shortcut for shape matching.
    if stride != 1 or shortcut.shape[-1] != filters:
        shortcut = layers.Conv2D(
            filters=filters,
            kernel_size=1,
            strides=stride,
            padding="same",
            use_bias=False,
            name=None if name is None else f"{name}_projection_conv",
        )(shortcut)

        shortcut = layers.BatchNormalization(
            name=None if name is None else f"{name}_projection_bn",
        )(shortcut)

    # Residual connection
    x = layers.Add(
        name=None if name is None else f"{name}_add",
    )([x, shortcut])

    x = layers.ReLU(
        name=None if name is None else f"{name}_output_relu",
    )(x)

    return x

## 2.2 Main Model Architecture

In [ ]:
# ==============================
# 2.2 Main Model Architecture
# ==============================

def build_custom_cnn_high_flops(
    input_shape: tuple[int, int, int],
    num_classes: int,
    augmentation: tf.keras.layers.Layer,
    dropout_rate: float = 0.4,
) -> tf.keras.Model:
    """
    Build the high-FLOPs custom residual CNN.

    Architecture: stem conv -> 3 residual stages (64/128/256 filters,
    2 blocks each) with strided-conv downsampling between stages ->
    global average pooling -> dense classifier head.

    Args:
        input_shape: Shape of a single input image, e.g. (64, 64, 3).
        num_classes: Number of output classes.
        augmentation: A Keras layer (or Sequential of layers) applying
            training-time data augmentation; should be a no-op at
            inference time (e.g. built from `layers.RandomFlip`, etc.).

    Returns:
        An uncompiled `tf.keras.Model` mapping raw uint8/float images
        in [0, 255] to class probabilities.
    """
    inputs = layers.Input(shape=input_shape, name="input")

    # Apply training-time data augmentation.
    x = augmentation(inputs)
    # Normalize input pixel values to the [0, 1] range.
    x = layers.Rescaling(
        scale=1.0 / 255,
        name="rescaling",
    )(x)

    # Initial feature extraction
    x = layers.Conv2D(
        filters=64,
        kernel_size=3,
        strides=1,
        padding="same",
        use_bias=False,
        name="stem_conv",
    )(x)
    x = layers.BatchNormalization(name="stem_bn")(x)
    x = layers.ReLU(name="stem_relu")(x)

    # Residual Stage 1
    x = residual_block(x, filters=64, name="stage1_block1")
    x = residual_block(x, filters=64, name="stage1_block2")

    # Downsample to 32x32 and increase channel capacity.
    x = layers.Conv2D(
        filters=128,
        kernel_size=3,
        strides=2,
        padding="same",
        use_bias=False,
        name="downsample1_conv",
    )(x)
    x = layers.BatchNormalization(name="downsample1_bn")(x)
    x = layers.ReLU(name="downsample1_relu")(x)

    # Residual Stage 2
    x = residual_block(x, filters=128, name="stage2_block1")
    x = residual_block(x, filters=128, name="stage2_block2")

    # Downsample to 16x16 and increase channel capacity.
    x = layers.Conv2D(
        filters=256,
        kernel_size=3,
        strides=2,
        padding="same",
        use_bias=False,
        name="downsample2_conv",
    )(x)
    x = layers.BatchNormalization(name="downsample2_bn")(x)
    x = layers.ReLU(name="downsample2_relu")(x)

    # Residual Stage 3
    x = residual_block(x, filters=256, name="stage3_block1")
    x = residual_block(x, filters=256, name="stage3_block2")

    # Global feature aggregation
    x = layers.GlobalAveragePooling2D(name="global_average_pooling")(x)

    # Classification head
    x = layers.Dense(
        256,
        activation="relu",
        name="classifier_dense",
    )(x)
    x = layers.Dropout(dropout_rate, name="classifier_dropout")(x)

    # Keep classifier outputs in float32 under mixed precision.
    outputs = layers.Dense(
        num_classes,
        activation="softmax",
        dtype="float32",
        name="predictions",
    )(x)

    return tf.keras.Model(
        inputs=inputs,
        outputs=outputs,
        name="custom_cnn_high_flops",
    )
 

## 2.3 Model Summary & Architecture Sanity Check

### 2.3.1 Build Model

In [ ]:
# ==============================
# 2.3.1 Build Model
# ==============================

input_shape = (*IMAGE_SIZE, IMAGE_CHANNELS)

model = build_custom_cnn_high_flops(
    input_shape=input_shape,
    num_classes=NUM_CLASSES,
    augmentation=augmentation,
)


### 2.3.2 Model Summary

In [ ]:
# ==============================
# 2.3.2 Model Summary
# ==============================

model.summary()

### 2.3.3 Parameter Sanity Check

In [ ]:
# ==============================
# 2.3.3 Parameter Sanity Check
# ==============================

model_stats = get_model_parameter_stats(model)

assert model_stats["total_params"] == model.count_params(), (
    "Manual parameter count does not match model.count_params()"
)

print("Model Parameter Statistics")
print("-" * 40)
print(f"Total parameters     : {model_stats['total_params']:,}")
print(f"Trainable parameters : {model_stats['trainable_params']:,}")
print(f"Non-trainable params : {model_stats['non_trainable_params']:,}")

### 2.3.4 Input / Output Shape Check

In [ ]:
# ==============================
# 2.3.4 Input / Output Shape Check
# ==============================

print("Model input shape :", model.input_shape)
print("Model output shape:", model.output_shape)

assert model.input_shape[1:] == EXPECTED_IMAGE_SHAPE, (
    f"Unexpected model input shape: {model.input_shape[1:]}"
)

assert model.output_shape[-1] == NUM_CLASSES, (
    f"Unexpected number of output classes: {model.output_shape[-1]}"
)

print("\nInput / Output shape check: PASSED")

### 2.3.5 Forward Pass Sanity Check

In [ ]:
# ==============================
# 2.3.5 Forward Pass Sanity Check
# ==============================

# Get one batch from the training dataset.
images, labels = next(iter(train_ds))

# Run a forward pass.
predictions = model(images, training=False)

print("Input batch shape :", images.shape)
print("Label batch shape :", labels.shape)
print("Prediction shape  :", predictions.shape)

In [ ]:
# ==============================
# Forward Pass Assertions
# ==============================

assert images.shape[1:] == EXPECTED_IMAGE_SHAPE, (
    f"Unexpected image shape: {images.shape[1:]}"
)

assert predictions.shape[0] == images.shape[0], (
    "Input and prediction batch sizes do not match."
)

assert predictions.shape[-1] == NUM_CLASSES, (
    f"Expected {NUM_CLASSES} output classes, "
    f"got {predictions.shape[-1]}"
)

print("Forward pass sanity check: PASSED")

### 2.3.6 Prediction / Softmax Check

In [ ]:
# ==============================
# 2.3.6 Prediction / Softmax Check
# ==============================

pred_min = tf.reduce_min(predictions).numpy()
pred_max = tf.reduce_max(predictions).numpy()

print(f"Prediction minimum: {pred_min:.6f}")
print(f"Prediction maximum: {pred_max:.6f}")

In [ ]:
# Sum of class probabilities for each sample.
probability_sums = tf.reduce_sum(predictions, axis=1)

print(
    "Probability sum range:",
    f"{tf.reduce_min(probability_sums).numpy():.6f}",
    "to",
    f"{tf.reduce_max(probability_sums).numpy():.6f}",
)

In [ ]:
# ==============================
# Softmax Assertions
# ==============================

assert pred_min >= 0.0, (
    "Predictions contain values below 0."
)

assert pred_max <= 1.0, (
    "Predictions contain values above 1."
)

assert tf.reduce_all(
    tf.abs(probability_sums - 1.0) < 1e-5
), (
    "Softmax probabilities do not sum to 1."
)

print("Prediction / Softmax check: PASSED")

### 2.3.7 Overfit-on-One-Batch Test

#### 2.3.7.1 Get a Fixed Batch

In [ ]:
# ==============================
# 2.3.7.1 Get a Fixed Batch
# ==============================

x_batch, y_batch = next(iter(train_ds))

print("Images shape:", x_batch.shape)
print("Labels shape:", y_batch.shape)

#### 2.3.7.2 Compile for Overfit Test

In [ ]:
# ==============================
# 2.3.7.2 Compile for Overfit Test
# ==============================

model.compile(
    optimizer=tf.keras.optimizers.Adam(),
    loss=tf.keras.losses.CategoricalCrossentropy(),
    metrics=["accuracy"],
)

print("Model compiled successfully.")

#### 2.3.7.3 One-Batch Overfit Test

In [ ]:
# # ==============================
# # 2.3.7.3 One-Batch Overfit Test
# # ==============================

# overfit_history = model.fit(
#     x_batch,
#     y_batch,
#     epochs=50,
#     verbose=1,
# )

In [ ]:
# # ==============================
# # One-Batch Overfit Test Result
# # ==============================

# final_loss = overfit_history.history["loss"][-1]
# final_accuracy = overfit_history.history["accuracy"][-1]
# best_accuracy = max(overfit_history.history["accuracy"])

# print(f"Final loss     : {final_loss:.6f}")
# print(f"Final accuracy : {final_accuracy:.4f}")
# print(f"Best accuracy  : {best_accuracy:.4f}")

## 2.4 Model Compilation

In [ ]:
# ============================================
# 2.4 Model Compilation
# ============================================

import tensorflow as tf


# --------------------------------------------
# Optimizer & Loss Configuration
# --------------------------------------------

LEARNING_RATE = 1e-4
WEIGHT_DECAY = 1e-4

OPTIMIZER_NAME = "AdamW"
LOSS_NAME = "CategoricalCrossentropy"


# --------------------------------------------
# Optimizer
# --------------------------------------------

optimizer = tf.keras.optimizers.AdamW(
    learning_rate=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
)


# --------------------------------------------
# Model Compilation
# --------------------------------------------

model.compile(
    optimizer=optimizer,
    loss=tf.keras.losses.CategoricalCrossentropy(),
    metrics=[
        tf.keras.metrics.CategoricalAccuracy(
            name="accuracy"
        )
    ],
)


print("Model compilation completed successfully.")
print(f"Optimizer     : {OPTIMIZER_NAME}")
print(f"Learning Rate : {LEARNING_RATE}")
print(f"Weight Decay  : {WEIGHT_DECAY}")
print(f"Loss Function : {LOSS_NAME}")

## 2.5 Initialize Baseline Run

In [ ]:
# ============================================
# 2.5 Initialize Baseline Run
# ============================================

# --------------------------------------------
# Run Identity
# --------------------------------------------

PHASE = "phase-1"
EXPERIMENT_TYPE = "baseline"
MODEL_DISPLAY_NAME = "High-FLOPs Custom CNN"
BASELINE_EPOCHS = 50

WANDB_RUN_STATE_PATH = os.path.join(
    REPORTS_DIR,
    f"{MODEL_NAME}_{PHASE}_{EXPERIMENT_TYPE}_wandb_run_state.json",
)


# --------------------------------------------
# Baseline Run Configuration
# --------------------------------------------

WANDB_CONFIG = build_wandb_config(
    model_name=MODEL_DISPLAY_NAME,
    model_type=MODEL_NAME,
    image_size=IMAGE_SIZE,
    image_channels=IMAGE_CHANNELS,
    num_classes=NUM_CLASSES,
    batch_size=BATCH_SIZE,
    epochs=BASELINE_EPOCHS,
    optimizer_name=OPTIMIZER_NAME,
    learning_rate=LEARNING_RATE,
    loss_name=LOSS_NAME,
    extra_config={
        "project_name": PROJECT_NAME,
        "project_version": PROJECT_VERSION,
        "optimizer_weight_decay": WEIGHT_DECAY,
    },
)


# --------------------------------------------
# Initialize or Resume W&B Run
# --------------------------------------------
# If a run-state file already exists (e.g. from before a Colab
# runtime reset), resume that exact run instead of creating a
# new one, so training/evaluation logs stay on one timeline.

if os.path.isfile(WANDB_RUN_STATE_PATH):
    previous_state = load_wandb_run_state(WANDB_RUN_STATE_PATH)

    wandb_run = initialize_wandb_run(
        model_type=MODEL_NAME,
        phase=PHASE,
        experiment_type=EXPERIMENT_TYPE,
        run_id=previous_state["run_id"],
    )

    print("Resumed existing baseline W&B run.")

else:
    wandb_run = initialize_wandb_run(
        model_type=MODEL_NAME,
        phase=PHASE,
        experiment_type=EXPERIMENT_TYPE,
        config=WANDB_CONFIG,
    )

    save_wandb_run_state(
        wandb_run,
        WANDB_RUN_STATE_PATH,
    )

    print("Baseline W&B run initialized successfully.")


print(f"Project         : {WANDB_PROJECT}")
print(f"Run Name        : {wandb_run.name}")
print(f"Run ID          : {wandb_run.id}")
print(f"Model Name      : {MODEL_DISPLAY_NAME}")
print(f"Phase           : {PHASE}")
print(f"Experiment Type : {EXPERIMENT_TYPE}")
print(f"Run State       : {WANDB_RUN_STATE_PATH}")

## 2.6 Log Training

In [ ]:
# ============================================
# 2.6 Log Training
# ============================================

# --------------------------------------------
# Training Callbacks
# --------------------------------------------

CHECKPOINT_PATH = os.path.join(
    CHECKPOINT_DIR,
    f"{MODEL_NAME}_{EXPERIMENT_TYPE}.keras",
)

early_stopping = build_early_stopping(
    monitor="val_loss",
    patience=5,
    mode="min",
    restore_best_weights=True,
)

model_checkpoint = build_model_checkpoint(
    filepath=CHECKPOINT_PATH,
    monitor="val_loss",
    mode="min",
    save_best_only=True,
    save_weights_only=False,
)

reduce_lr = build_reduce_lr_on_plateau(
    monitor="val_loss",
    factor=0.5,
    patience=2,
    min_lr=1e-6,
    mode="min",
)

wandb_epoch_logger = WandBEpochLogger(wandb_run)


# --------------------------------------------
# Callback List
# --------------------------------------------

TRAINING_CALLBACKS = [
    early_stopping,
    model_checkpoint,
    reduce_lr,
    wandb_epoch_logger,
]


# --------------------------------------------
# Baseline Training
# --------------------------------------------

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=BASELINE_EPOCHS,
    callbacks=TRAINING_CALLBACKS,
    verbose=1,
)


# --------------------------------------------
# Actual Epoch Count
# --------------------------------------------

COMPLETED_EPOCHS = len(history.history["loss"])


# --------------------------------------------
# Persist Training History
# --------------------------------------------
# Saved to disk so later sections (e.g. Error Analysis) can
# reload it even in a fresh Colab runtime where `history` no
# longer exists in memory.

TRAINING_HISTORY_PATH = os.path.join(
    REPORTS_DIR,
    f"{MODEL_NAME}_{PHASE}_{EXPERIMENT_TYPE}_training_history.json",
)

training_history_dict = get_training_history_dict(history)

with open(TRAINING_HISTORY_PATH, "w", encoding="utf-8") as file:
    json.dump(training_history_dict, file, indent=4)


# --------------------------------------------
# Sync Actual Epoch Count to W&B
# --------------------------------------------

wandb_run.config.update(
    {
        "training_completed_epochs": COMPLETED_EPOCHS,
    },
    allow_val_change=True,
)


# --------------------------------------------
# Log Training Summary
# --------------------------------------------
# Note: log_wandb_training_history() is NOT called here.
# WandBEpochLogger already logged every epoch in real time
# during model.fit(). Calling it here would duplicate those logs.

log_wandb_training_summary(
    wandb_run,
    history,
)


# --------------------------------------------
# Training Completion Summary
# --------------------------------------------

print("Baseline training completed successfully.")
print(f"Planned Epochs    : {BASELINE_EPOCHS}")
print(f"Completed Epochs  : {COMPLETED_EPOCHS}")
print(f"Checkpoint        : {CHECKPOINT_PATH}")
print(f"Training History  : {TRAINING_HISTORY_PATH}")

##  2.7 Log Training Analysis

In [ ]:
# ============================================
# 2.7 Log Training Analysis
# ============================================

# --------------------------------------------
# Load Training History
# --------------------------------------------
# Always reloaded from disk (rather than reusing the in-memory
# `history` object) so this section behaves identically whether
# run right after training or in a fresh runtime after a
# Colab reset.

TRAINING_HISTORY_PATH = os.path.join(
    REPORTS_DIR,
    f"{MODEL_NAME}_{PHASE}_{EXPERIMENT_TYPE}_training_history.json",
)

if not os.path.isfile(TRAINING_HISTORY_PATH):
    raise FileNotFoundError(
        f"Training history file was not found: {TRAINING_HISTORY_PATH}"
    )

with open(TRAINING_HISTORY_PATH, "r", encoding="utf-8") as file:
    training_history_dict = json.load(file)

if not training_history_dict:
    raise ValueError("Training history is empty.")


# --------------------------------------------
# Reconstruct Keras History Object
# --------------------------------------------

training_history = tf.keras.callbacks.History()
training_history.history = training_history_dict


# --------------------------------------------
# Plot Training History
# --------------------------------------------

TRAINING_HISTORY_PLOT_PATH = plot_training_history(
    history=training_history,
    model_name=MODEL_NAME,
    phase=PHASE,
    experiment_type=EXPERIMENT_TYPE,
    save_dir=PLOTS_DIR,
)


# --------------------------------------------
# Log Training Analysis Metadata
# --------------------------------------------

wandb_run.summary["training_history_plot_path"] = TRAINING_HISTORY_PLOT_PATH

log_wandb_media(
    wandb_run,
    "training_history_plot",
    wandb.Image(TRAINING_HISTORY_PLOT_PATH),
)


# --------------------------------------------
# Training Analysis Summary
# --------------------------------------------

COMPLETED_EPOCHS = len(training_history_dict["loss"])

print("Training analysis completed successfully.")
print(f"Training History Plot : {TRAINING_HISTORY_PLOT_PATH}")
print(f"Tracked Metrics       : {list(training_history_dict.keys())}")
print(f"Completed Epochs      : {COMPLETED_EPOCHS}")

## 2.8 Log Validation Evaluation

In [ ]:
# ============================================
# 2.8 Log Validation Evaluation
# ============================================

# --------------------------------------------
# Validation Evaluation
# --------------------------------------------
# Note: this evaluates the restored best-weights model
# (EarlyStopping.restore_best_weights=True), not the final
# training epoch. Logged under "validation_keras/" (not
# "validation/") to avoid colliding with the richer sklearn-based
# metrics logged in Section 2.9 under "validation/" — both
# currently agree numerically since they evaluate the same
# model on the same data, but keeping them namespaced separately
# avoids one silently overwriting the other if that ever changes.

val_metrics = model.evaluate(
    val_ds,
    verbose=1,
    return_dict=True,
)


# --------------------------------------------
# Log Validation Metrics
# --------------------------------------------

log_wandb_evaluation_metrics(
    wandb_run,
    val_metrics,
    split="validation_keras",
)


# --------------------------------------------
# Log Validation Summary
# --------------------------------------------

log_wandb_evaluation_summary(
    wandb_run,
    val_metrics,
    split="validation_keras",
)


# --------------------------------------------
# Validation Evaluation Summary
# --------------------------------------------

print("Validation evaluation completed successfully.")

for metric_name, metric_value in val_metrics.items():
    print(f"{metric_name:<20}: {metric_value:.4f}")

## 2.9 Prediction Collection

In [ ]:
# ============================================
# 2.9 Prediction Collection
# ============================================

# --------------------------------------------
# Collect Validation Predictions
# --------------------------------------------

y_true, y_pred, y_prob = extract_predictions(
    model=model,
    dataset=val_ds
)


# --------------------------------------------
# Prediction Sanity Check
# --------------------------------------------

if (
    len(y_true) != len(y_pred)
    or len(y_true) != len(y_prob)
):
    raise ValueError(
        "Mismatch between validation labels, "
        "predictions, and probabilities."
    )

if not np.allclose(
    y_prob.sum(axis=1),
    1.0,
    atol=1e-5,
):
    raise ValueError(
        "Validation prediction probabilities do not sum to 1."
    )


# --------------------------------------------
# Calculate Prediction Confidence
# --------------------------------------------

confidence = np.max(
    y_prob,
    axis=1,
)


# --------------------------------------------
# Persist Predictions for Error Analysis
# --------------------------------------------
# Saved to disk so Section 2.10 (Error Analysis) — including
# its Interpretable ML subsection (Grad-CAM / Saliency Maps) —
# can reload the exact same predictions even in a fresh Colab
# runtime, where these arrays no longer exist in memory.
#
# The validation file paths are obtained from the LOCAL validation
# directory created in Section 1.5. A fresh unbatched dataset is
# created only to access its file_paths attribute; no validation
# training or evaluation is performed here.
#
# The dataset is loaded with shuffle=False, so the file ordering
# matches the ordering used to generate y_true/y_pred/y_prob.
#
# Predictions are saved before the classification-metrics/W&B calls
# below so the raw predictions are preserved even if one of those
# later steps fails (e.g. a W&B network error).


val_file_paths = load_local_dataset(
    LOCAL_VAL_DIR,
    IMAGE_SIZE,
    CLASS_NAMES,
).file_paths


if len(val_file_paths) != len(y_true):
    raise ValueError(
        "Mismatch between validation file paths "
        "and predictions."
    )


PREDICTIONS_PATH = os.path.join(
    REPORTS_DIR,
    f"{MODEL_NAME}_{PHASE}_{EXPERIMENT_TYPE}_predictions.npz",
)


np.savez(
    PREDICTIONS_PATH,
    y_true=y_true,
    y_pred=y_pred,
    y_prob=y_prob,
    confidence=confidence,
    file_paths=np.array(val_file_paths),
)


# --------------------------------------------
# Calculate Classification Metrics
# --------------------------------------------

baseline_metrics = calculate_all_classification_metrics(
    y_true=y_true,
    y_pred=y_pred,
)


# --------------------------------------------
# Save Evaluation Metrics
# --------------------------------------------

metrics_path = save_evaluation_metrics(
    metrics=baseline_metrics,
    model_name=MODEL_NAME,
    phase=PHASE,
    experiment_type=EXPERIMENT_TYPE,
    dataset_name="validation",
)


# --------------------------------------------
# Log Classification Metrics to W&B
# --------------------------------------------

log_wandb_evaluation_metrics(
    wandb_run,
    baseline_metrics,
    split="validation",
)


# --------------------------------------------
# Log Classification Summary to W&B
# --------------------------------------------

log_wandb_evaluation_summary(
    wandb_run,
    baseline_metrics,
    split="validation",
)


# --------------------------------------------
# Summary
# --------------------------------------------

print(
    "Validation prediction collection "
    "completed successfully."
)

print("\nValidation Metrics:")

print(
    f"Accuracy    : "
    f"{baseline_metrics['accuracy']:.4f}"
)

print(
    f"Macro F1    : "
    f"{baseline_metrics['macro']['f1_score']:.4f}"
)

print(
    f"Weighted F1 : "
    f"{baseline_metrics['weighted']['f1_score']:.4f}"
)

print(
    f"\nMetrics saved to: "
    f"{metrics_path}"
)

print(
    f"Predictions saved to: "
    f"{PREDICTIONS_PATH}"
)

print("\nPrediction Shapes:")

print(
    f"y_true      : "
    f"{y_true.shape}"
)

print(
    f"y_pred      : "
    f"{y_pred.shape}"
)

print(
    f"y_prob      : "
    f"{y_prob.shape}"
)

print(
    f"confidence  : "
    f"{confidence.shape}"
)

## 2.10 Log Error Analysis

### 2.10.1 Prediction Analysis

In [ ]:
# ============================================
# 2.10.1 Prediction Analysis
# ============================================

# --------------------------------------------
# Load Predictions
# --------------------------------------------
# Always reload predictions from disk instead of reusing
# the in-memory arrays from Section 2.9.
#
# This keeps Error Analysis reproducible and allows this
# section to work correctly after a fresh Colab runtime.

PREDICTIONS_PATH = os.path.join(
    REPORTS_DIR,
    f"{MODEL_NAME}_{PHASE}_{EXPERIMENT_TYPE}_predictions.npz",
)

if not os.path.isfile(PREDICTIONS_PATH):
    raise FileNotFoundError(
        f"Predictions file was not found: {PREDICTIONS_PATH}"
    )


predictions_data = np.load(
    PREDICTIONS_PATH,
    allow_pickle=False,
)


y_true = predictions_data["y_true"]
y_pred = predictions_data["y_pred"]
y_prob = predictions_data["y_prob"]
confidence = predictions_data["confidence"]
file_paths = predictions_data["file_paths"]


# --------------------------------------------
# Validate Loaded Predictions
# --------------------------------------------

if len(y_true) == 0:
    raise ValueError(
        "Loaded predictions are empty."
    )


if not (
    len(y_true)
    == len(y_pred)
    == len(y_prob)
    == len(confidence)
    == len(file_paths)
):
    raise ValueError(
        "Loaded prediction arrays have inconsistent lengths."
    )


if y_prob.ndim != 2:
    raise ValueError(
        "Loaded prediction probabilities must be a 2D array."
    )


if y_prob.shape[1] != len(CLASS_NAMES):
    raise ValueError(
        "Prediction probability shape does not match CLASS_NAMES."
    )


if not np.allclose(
    y_prob.sum(axis=1),
    1.0,
    atol=1e-5,
):
    raise ValueError(
        "Loaded prediction probabilities do not sum to 1."
    )


# --------------------------------------------
# Calculate Error Statistics
# --------------------------------------------

error_statistics = calculate_error_statistics(
    y_true=y_true,
    y_pred=y_pred,
    y_prob=y_prob,
    class_names=CLASS_NAMES,
)


# --------------------------------------------
# Log Error Analysis Metrics to W&B
# --------------------------------------------

log_wandb_error_analysis_metrics(
    wandb_run,
    error_statistics,
)


# --------------------------------------------
# Log Error Analysis Summary to W&B
# --------------------------------------------

log_wandb_error_analysis_summary(
    wandb_run,
    error_statistics,
)


# --------------------------------------------
# Prediction Analysis Summary
# --------------------------------------------

print(
    "Prediction analysis completed successfully."
)

print("\nOverall Error Statistics:")
print(
    f"Total samples          : "
    f"{error_statistics['total_samples']}"
)
print(
    f"Total errors           : "
    f"{error_statistics['total_errors']}"
)
print(
    f"Overall error rate     : "
    f"{error_statistics['overall_error_rate']:.4f}"
)
print(
    f"Mean confidence        : "
    f"{error_statistics['mean_confidence']:.4f}"
)
print(
    f"Mean error confidence  : "
    f"{error_statistics['mean_error_confidence']:.4f}"
)

### 2.10.2 Confusion Matrix

In [ ]:
# ============================================
# 2.10.2 Confusion Matrix
# ============================================

# --------------------------------------------
# Generate and Save Confusion Matrix
# --------------------------------------------

CONFUSION_MATRIX_PATH = plot_confusion_matrix(
    y_true=y_true,
    y_pred=y_pred,
    class_names=CLASS_NAMES,
    model_name=MODEL_NAME,
    phase=PHASE,
    experiment_type=EXPERIMENT_TYPE,
    dataset_name="validation",
    save_dir=PLOTS_DIR,
)


# --------------------------------------------
# Validate Saved Confusion Matrix
# --------------------------------------------

if not os.path.isfile(CONFUSION_MATRIX_PATH):
    raise FileNotFoundError(
        f"Confusion matrix plot was not found: "
        f"{CONFUSION_MATRIX_PATH}"
    )


# --------------------------------------------
# Log Confusion Matrix to W&B
# --------------------------------------------

wandb_run.summary[
    "validation_confusion_matrix_plot"
] = CONFUSION_MATRIX_PATH

log_wandb_media(
    wandb_run,
    "validation_confusion_matrix",
    wandb.Image(CONFUSION_MATRIX_PATH),
)


# --------------------------------------------
# Confusion Matrix Summary
# --------------------------------------------

print("Confusion matrix analysis completed successfully.")

print(
    f"Confusion matrix saved to: "
    f"{CONFUSION_MATRIX_PATH}"
)

### 2.10.3 Per-Class Metrics

In [ ]:
# ============================================
# 2.10.3 Per-Class Metrics
# ============================================

# --------------------------------------------
# Generate and Save Classification Report
# --------------------------------------------

CLASSIFICATION_REPORT_PATH, classification_report_dict = (
    save_classification_report(
        y_true=y_true,
        y_pred=y_pred,
        class_names=CLASS_NAMES,
        model_name=MODEL_NAME,
        phase=PHASE,
        experiment_type=EXPERIMENT_TYPE,
        dataset_name="validation",
        save_dir=REPORTS_DIR,
    )
)


# --------------------------------------------
# Validate Saved Classification Report
# --------------------------------------------

if not os.path.isfile(CLASSIFICATION_REPORT_PATH):
    raise FileNotFoundError(
        f"Classification report was not found: "
        f"{CLASSIFICATION_REPORT_PATH}"
    )


# --------------------------------------------
# Extract Per-Class Metrics
# --------------------------------------------

per_class_metrics = {
    class_name: {
        "precision": float(
            classification_report_dict[class_name]["precision"]
        ),
        "recall": float(
            classification_report_dict[class_name]["recall"]
        ),
        "f1_score": float(
            classification_report_dict[class_name]["f1-score"]
        ),
        "support": int(
            classification_report_dict[class_name]["support"]
        ),
    }
    for class_name in CLASS_NAMES
}


# --------------------------------------------
# Log Per-Class Metrics to W&B
# --------------------------------------------

log_wandb_evaluation_metrics(
    wandb_run,
    per_class_metrics,
    split="validation_per_class",
)


# --------------------------------------------
# Log Per-Class Metrics Summary to W&B
# --------------------------------------------

log_wandb_evaluation_summary(
    wandb_run,
    per_class_metrics,
    split="validation_per_class",
)


# --------------------------------------------
# Per-Class Metrics Summary
# --------------------------------------------

print("Per-class metrics analysis completed successfully.")
print(f"\nClassification report saved to: {CLASSIFICATION_REPORT_PATH}")

### 2.10.4 Confidence Analysis

In [ ]:
# ============================================
# 2.10.4 Confidence Analysis
# ============================================

# --------------------------------------------
# Validate Confidence Data
# --------------------------------------------

if len(confidence) != len(y_true):
    raise ValueError(
        "Confidence array length does not match validation predictions."
    )


if np.any(confidence < 0.0) or np.any(confidence > 1.0):
    raise ValueError(
        "Confidence values must be within the [0, 1] range."
    )


# --------------------------------------------
# Calculate Confidence Statistics
# --------------------------------------------

error_mask = y_true != y_pred
correct_mask = ~error_mask

confidence_statistics = {
    "median_confidence": float(
        np.median(confidence)
    ),
    "min_confidence": float(
        np.min(confidence)
    ),
    "max_confidence": float(
        np.max(confidence)
    ),
    "mean_correct_confidence": float(
        np.mean(confidence[correct_mask])
    ) if np.any(correct_mask) else 0.0,
}


# --------------------------------------------
# Log Confidence Metrics to W&B
# --------------------------------------------

log_wandb_error_analysis_metrics(
    wandb_run,
    confidence_statistics,
)


# --------------------------------------------
# Log Confidence Summary to W&B
# --------------------------------------------

log_wandb_error_analysis_summary(
    wandb_run,
    confidence_statistics,
)


# --------------------------------------------
# Confidence Analysis Summary
# --------------------------------------------

print("Confidence analysis completed successfully.")

print("\nConfidence Statistics:")
print(
    f"Median confidence       : "
    f"{confidence_statistics['median_confidence']:.4f}"
)
print(
    f"Minimum confidence      : "
    f"{confidence_statistics['min_confidence']:.4f}"
)
print(
    f"Maximum confidence      : "
    f"{confidence_statistics['max_confidence']:.4f}"
)
print(
    f"Mean correct confidence : "
    f"{confidence_statistics['mean_correct_confidence']:.4f}"
)

### 2.10.5 High-Confidence Errors

In [ ]:
# ============================================
# 2.10.5 High-Confidence Errors
# ============================================

# --------------------------------------------
# High-Confidence Error Configuration
# --------------------------------------------

HIGH_CONFIDENCE_THRESHOLD = 0.80


# --------------------------------------------
# Extract High-Confidence Errors
# --------------------------------------------

(
    high_conf_indices,
    high_conf_true_labels,
    high_conf_predicted_labels,
    high_conf_confidences,
) = extract_high_confidence_errors(
    y_true=y_true,
    y_pred=y_pred,
    y_prob=y_prob,
    threshold=HIGH_CONFIDENCE_THRESHOLD,
)


# --------------------------------------------
# Calculate High-Confidence Error Statistics
# --------------------------------------------

total_high_confidence_errors = int(
    len(high_conf_indices)
)

high_confidence_error_rate = (
    total_high_confidence_errors / len(y_true)
    if len(y_true) > 0
    else 0.0
)

high_confidence_error_fraction = (
    total_high_confidence_errors / error_statistics["total_errors"]
    if error_statistics["total_errors"] > 0
    else 0.0
)

mean_high_confidence_error_confidence = (
    float(np.mean(high_conf_confidences))
    if total_high_confidence_errors > 0
    else 0.0
)


high_confidence_error_statistics = {
    "threshold": HIGH_CONFIDENCE_THRESHOLD,
    "total_high_confidence_errors": total_high_confidence_errors,
    "high_confidence_error_rate": high_confidence_error_rate,
    "high_confidence_error_fraction": high_confidence_error_fraction,
    "mean_high_confidence_error_confidence": mean_high_confidence_error_confidence,
}


# --------------------------------------------
# Log High-Confidence Error Metrics to W&B
# --------------------------------------------

log_wandb_error_analysis_metrics(
    wandb_run,
    high_confidence_error_statistics,
)


# --------------------------------------------
# Log High-Confidence Error Summary to W&B
# --------------------------------------------

log_wandb_error_analysis_summary(
    wandb_run,
    high_confidence_error_statistics,
)


# --------------------------------------------
# High-Confidence Error Summary
# --------------------------------------------

print("High-confidence error analysis completed successfully.")

print("\nHigh-Confidence Error Statistics:")
print(
    f"Confidence threshold             : "
    f"{HIGH_CONFIDENCE_THRESHOLD:.2f}"
)
print(
    f"Total high-confidence errors     : "
    f"{total_high_confidence_errors}"
)
print(
    f"High-confidence error rate       : "
    f"{high_confidence_error_rate:.4f}"
)
print(
    f"Fraction of all errors           : "
    f"{high_confidence_error_fraction:.4f}"
)
print(
    f"Mean high-confidence error score : "
    f"{mean_high_confidence_error_confidence:.4f}"
)

### 2.10.6 Visual Error Analysis

In [ ]:
# ============================================
# 2.10.6 Visual Error Analysis
# ============================================

# --------------------------------------------
# Generate Visual Error Analysis
# --------------------------------------------

MAX_ERROR_SAMPLES_TO_PLOT = 16

VISUAL_ERROR_ANALYSIS_PATH = plot_error_samples(
    file_paths=file_paths[high_conf_indices],
    true_labels=high_conf_true_labels,
    predicted_labels=high_conf_predicted_labels,
    confidences=high_conf_confidences,
    class_names=CLASS_NAMES,
    model_name=MODEL_NAME,
    phase=PHASE,
    experiment_type=EXPERIMENT_TYPE,
    max_samples=MAX_ERROR_SAMPLES_TO_PLOT,
    save_dir=PLOTS_DIR,
)


# --------------------------------------------
# Validate Saved Error Analysis Plot
# --------------------------------------------

if not os.path.isfile(VISUAL_ERROR_ANALYSIS_PATH):
    raise FileNotFoundError(
        f"Visual error analysis plot was not found: "
        f"{VISUAL_ERROR_ANALYSIS_PATH}"
    )


# --------------------------------------------
# Log Visual Error Analysis to W&B
# --------------------------------------------

wandb_run.summary["high_confidence_errors_plot"] = VISUAL_ERROR_ANALYSIS_PATH

log_wandb_media(
    wandb_run,
    "high_confidence_errors",
    wandb.Image(VISUAL_ERROR_ANALYSIS_PATH),
)


# --------------------------------------------
# Visual Error Analysis Summary
# --------------------------------------------

print("Visual error analysis completed successfully.")

print(
    f"\nHigh-confidence errors available : "
    f"{len(high_conf_indices)}"
)
print(
    f"Samples plotted                  : "
    f"{min(MAX_ERROR_SAMPLES_TO_PLOT, len(high_conf_indices))}"
)
print(
    f"Visual analysis saved to         : "
    f"{VISUAL_ERROR_ANALYSIS_PATH}"
)

### 2.10.7 High-Confidence Error Distribution

In [ ]:
# ============================================
# 2.10.7 High-Confidence Error Distribution
# ============================================

# --------------------------------------------
# Calculate High-Confidence Error Distribution
# --------------------------------------------

high_confidence_error_distribution = (
    calculate_high_confidence_error_distribution(
        true_labels=high_conf_true_labels,
        predicted_labels=high_conf_predicted_labels,
        confidences=high_conf_confidences,
        class_names=CLASS_NAMES,
    )
)


# --------------------------------------------
# Display High-Confidence Error Distribution
# --------------------------------------------

display(
    high_confidence_error_distribution
)


# --------------------------------------------
# Log High-Confidence Error Distribution to W&B
# --------------------------------------------

high_confidence_error_distribution_table = (
    wandb.Table(
        dataframe=high_confidence_error_distribution
    )
)

wandb_run.log(
    {
        "error_analysis/high_confidence_error_distribution":
            high_confidence_error_distribution_table
    }
)


# --------------------------------------------
# High-Confidence Error Distribution Summary
# --------------------------------------------

print(
    "High-confidence error distribution "
    "analysis completed successfully."
)

### 2.10.8 Interpretable ML (IML)

In [ ]:
# ============================================
# 2.10.8 Model Interpretability
# ============================================

GRAD_CAM_LAYER_NAME = "stage3_block2_conv2"
GRAD_CAM_NUM_SAMPLES = 14
GRAD_CAM_MAX_ERROR_PAIRS = 7


if len(high_conf_indices) == 0:

    print(
        "No high-confidence errors were found. "
        "Grad-CAM analysis was skipped."
    )

else:

    # ----------------------------------------
    # Identify Error Pairs
    # ----------------------------------------
    # High-confidence errors are grouped by their
    # True -> Predicted class pair so that Grad-CAM
    # analysis covers multiple error patterns instead
    # of selecting only the globally most confident errors.

    error_pairs = np.column_stack(
        (
            high_conf_true_labels,
            high_conf_predicted_labels,
        )
    )

    unique_error_pairs, pair_counts = np.unique(
        error_pairs,
        axis=0,
        return_counts=True,
    )

    pair_order = np.argsort(
        pair_counts
    )[
        ::-1
    ]

    pair_order = pair_order[
        :GRAD_CAM_MAX_ERROR_PAIRS
    ]


    # ----------------------------------------
    # Select Representative Samples
    # ----------------------------------------
    # For each selected True -> Predicted pair:
    #   1. Select the highest-confidence error.
    #   2. Select a representative error whose confidence
    #      is closest to the pair's median confidence.
    #
    # This provides both an extreme high-confidence error
    # and a more typical error for each major error pattern.

    selected_positions = []

    for pair_index in pair_order:

        true_label_index = int(
            unique_error_pairs[pair_index][0]
        )

        predicted_label_index = int(
            unique_error_pairs[pair_index][1]
        )

        pair_mask = (
            (high_conf_true_labels == true_label_index)
            & (
                high_conf_predicted_labels
                == predicted_label_index
            )
        )

        pair_positions = np.where(
            pair_mask
        )[0]

        if len(pair_positions) == 0:
            continue


        # ------------------------------------
        # Highest-Confidence Sample
        # ------------------------------------

        highest_confidence_position = int(
            pair_positions[
                np.argmax(
                    high_conf_confidences[
                        pair_positions
                    ]
                )
            ]
        )

        selected_positions.append(
            highest_confidence_position
        )


        # ------------------------------------
        # Median-Confidence Sample
        # ------------------------------------

        if len(pair_positions) > 1:

            pair_confidences = (
                high_conf_confidences[
                    pair_positions
                ]
            )

            median_confidence = float(
                np.median(
                    pair_confidences
                )
            )

            remaining_positions = (
                pair_positions[
                    pair_positions
                    != highest_confidence_position
                ]
            )

            median_position = int(
                remaining_positions[
                    np.argmin(
                        np.abs(
                            high_conf_confidences[
                                remaining_positions
                            ]
                            - median_confidence
                        )
                    )
                ]
            )

            selected_positions.append(
                median_position
            )


    # ----------------------------------------
    # Complete Sample Selection
    # ----------------------------------------
    # If the selected error pairs contain fewer
    # than the requested number of samples, fill
    # the remaining slots with the highest-confidence
    # errors that have not already been selected.

    selected_positions = list(
        dict.fromkeys(
            selected_positions
        )
    )

    remaining_positions = [
        position
        for position in range(
            len(high_conf_indices)
        )
        if position not in selected_positions
    ]

    remaining_positions = sorted(
        remaining_positions,
        key=lambda position:
            high_conf_confidences[position],
        reverse=True,
    )

    remaining_slots = (
        GRAD_CAM_NUM_SAMPLES
        - len(selected_positions)
    )

    if remaining_slots > 0:

        selected_positions.extend(
            remaining_positions[
                :remaining_slots
            ]
        )


    # ----------------------------------------
    # Final Sample Selection
    # ----------------------------------------

    selected_positions = selected_positions[
        :GRAD_CAM_NUM_SAMPLES
    ]

    num_grad_cam_samples = len(
        selected_positions
    )

    grad_cam_paths = []


    # ----------------------------------------
    # Generate Grad-CAM Analysis
    # ----------------------------------------

    for rank, selected_position in enumerate(
        selected_positions,
        start=1,
    ):

        sample_index = int(
            high_conf_indices[
                selected_position
            ]
        )

        true_label_index = int(
            high_conf_true_labels[
                selected_position
            ]
        )

        predicted_label_index = int(
            high_conf_predicted_labels[
                selected_position
            ]
        )

        confidence_value = float(
            high_conf_confidences[
                selected_position
            ]
        )

        image_path = file_paths[
            sample_index
        ]


        if not os.path.isfile(
            image_path
        ):
            raise FileNotFoundError(
                f"Grad-CAM image was not found: "
                f"{image_path}"
            )


        # Reload the original image at IMAGE_SIZE because
        # this array is passed through the model itself.

        image = tf.keras.utils.load_img(
            image_path,
            target_size=IMAGE_SIZE,
        )

        image = tf.keras.utils.img_to_array(
            image
        )


        # Generate two heatmaps per sample: one for the
        # true class score and one for the predicted class
        # score, allowing direct comparison of the spatial
        # evidence associated with both classes.

        true_heatmap = generate_grad_cam(
            model=model,
            image=image,
            target_class=true_label_index,
            layer_name=GRAD_CAM_LAYER_NAME,
        )

        predicted_heatmap = generate_grad_cam(
            model=model,
            image=image,
            target_class=predicted_label_index,
            layer_name=GRAD_CAM_LAYER_NAME,
        )


        grad_cam_path = plot_grad_cam(
            image=image,
            true_heatmap=true_heatmap,
            predicted_heatmap=predicted_heatmap,
            true_label=CLASS_NAMES[
                true_label_index
            ],
            predicted_label=CLASS_NAMES[
                predicted_label_index
            ],
            confidence=confidence_value,
            model_name=MODEL_NAME,
            phase=PHASE,
            experiment_type=MODEL_OUTPUT_DIR,
            sample_index=sample_index,
            save_dir=PLOTS_DIR,
        )


        if not os.path.isfile(
            grad_cam_path
        ):
            raise FileNotFoundError(
                f"Grad-CAM plot was not found: "
                f"{grad_cam_path}"
            )


        grad_cam_paths.append(
            grad_cam_path
        )


        print(
            f"Grad-CAM {rank}/{num_grad_cam_samples} | "
            f"Sample index: {sample_index} | "
            f"True: "
            f"{CLASS_NAMES[true_label_index]} | "
            f"Pred: "
            f"{CLASS_NAMES[predicted_label_index]} | "
            f"Confidence: "
            f"{confidence_value:.4f}"
        )


    # ----------------------------------------
    # Log Final Grad-CAM Analysis to W&B
    # ----------------------------------------
    # The original 4-sample Grad-CAM analysis is preserved
    # under its previous W&B key. The expanded analysis is
    # logged separately as the final 14-sample analysis.

    wandb_grad_cam_media = [
        wandb.Image(path)
        for path in grad_cam_paths
    ]

    log_wandb_media_collection(
        wandb_run,
        "grad_cam_high_confidence_errors_final",
        wandb_grad_cam_media,
    )


    # ----------------------------------------
    # Log Final Grad-CAM Metadata
    # ----------------------------------------

    wandb_run.summary[
        "grad_cam_layer"
    ] = GRAD_CAM_LAYER_NAME

    wandb_run.summary[
        "grad_cam_samples"
    ] = int(
        num_grad_cam_samples
    )

    wandb_run.summary[
        "grad_cam_selection"
    ] = (
        "Top error pairs with highest-confidence "
        "and representative median-confidence samples"
    )


    # ----------------------------------------
    # Grad-CAM Analysis Summary
    # ----------------------------------------

    print(
        "\nGrad-CAM interpretability analysis "
        "completed successfully."
    )

    print(
        f"Samples analyzed     : "
        f"{num_grad_cam_samples}"
    )

    print(
        f"Error pairs covered  : "
        f"{min(GRAD_CAM_MAX_ERROR_PAIRS, len(unique_error_pairs))}"
    )

    print(
        f"Target layer         : "
        f"{GRAD_CAM_LAYER_NAME}"
    )

    print(
        f"Grad-CAM plots saved : "
        f"{len(grad_cam_paths)}"
    )

### 2.10.9 Error Analysis Summary / Findings

#### Error Analysis Summary / Findings

##### 1. Baseline Overview

The baseline CNN achieved strong overall classification performance with an accuracy of approximately **96.23%** on the evaluation set.

However, the remaining errors are not randomly distributed. Error analysis revealed several recurring confusion patterns, particularly among classes with similar visual characteristics, vegetation structure, texture, or spatial layout.

---

##### 2. Main Error Findings

##### 2.1 SeaLake → Forest

- Total errors: **21**
- High-confidence errors: **11**
- High-confidence fraction of this error pair: **52.4%**
- Mean confidence of high-confidence errors: **0.9182**
- Maximum confidence: **0.9952**

This is the **largest confusion pair** in the baseline.

Visual inspection showed that some SeaLake samples contain limited class-specific visual cues and are dominated by dark blue/green regions. In the analyzed Grad-CAM examples, the predicted Forest activations were relatively broad and did not consistently correspond to a clearly identifiable forest-specific structure.

**Finding:**  
The model may be relying on broad color/texture cues in ambiguous scenes, leading some SeaLake samples to be represented as Forest-like.

**Confidence level:** Strong evidence for a visual-overlap / ambiguity issue, but not sufficient to claim that color alone causes the error.

---

##### 2.2 Pasture → Forest

- Total errors: **10**
- High-confidence errors: **7**
- High-confidence fraction: **70.0%**
- Mean high-confidence score: **0.9387**
- Maximum confidence: **0.9923**

This is one of the most concentrated high-confidence error patterns.

Visual analysis and Grad-CAM suggest that the model frequently responds to vegetation-related regions when predicting Forest. The initial hypothesis that trees inside pasture areas may cause Forest predictions is plausible, but the evidence is more generally consistent with **shared vegetation and texture features** rather than trees alone.

**Finding:**  
Pasture and Forest share visual representations that can cause systematic confusion, often with high model confidence.

---

##### 2.3 PermanentCrop → HerbaceousVegetation

- Total errors: **14**
- High-confidence errors: **6**
- High-confidence fraction: **42.9%**
- Mean high-confidence score: **0.9138**
- Maximum confidence: **0.9983**

This is the **largest confusion pair for PermanentCrop**.

Multiple visual examples showed similar vegetation/texture patterns between the two classes. Grad-CAM activations were relatively broad and frequently concentrated on vegetation-related regions.

**Finding:**  
The model appears to have difficulty separating PermanentCrop from HerbaceousVegetation when their visual texture and vegetation structure overlap.

This may represent either a model representation limitation, an intrinsic dataset ambiguity, or a combination of both.

---

##### 2.4 AnnualCrop → PermanentCrop

- Total errors: **9**
- High-confidence errors: **5**
- High-confidence fraction: **55.6%**
- Mean high-confidence score: **0.8924**
- Maximum confidence: **0.9715**

The model generally recognizes these samples as agricultural environments but sometimes fails to distinguish the finer semantic difference between AnnualCrop and PermanentCrop.

**Finding:**  
This represents a relatively fine-grained classification problem in which both classes share similar agricultural structures, colors, textures, and spatial patterns.

---

##### 2.5 AnnualCrop → River

- Total errors: **10**
- High-confidence errors: **3**
- High-confidence fraction: **30.0%**
- Maximum confidence: **0.9959**

Visual and Grad-CAM inspection suggests that elongated or linear spatial structures may contribute to some River predictions.

However, the relatively low high-confidence fraction indicates that many AnnualCrop → River errors are associated with greater uncertainty than the Pasture → Forest or SeaLake → Forest errors.

**Finding:**  
The model may sometimes associate shared spatial structures with River, but the evidence suggests this is a less consistently confident failure mode.

---

##### 3. Other Observed Error Patterns

##### Urban / Infrastructure Classes

Confusion was also observed between:

- Industrial → Residential
- Residential → Industrial
- Highway → River
- River → Highway

These errors are consistent with visual overlap between built-up environments and elongated spatial structures.

The current evidence does not indicate an obvious model failure unique to these classes. Instead, some of these errors may reflect intrinsic visual similarity between the class definitions.

---

##### 4. Confidence Analysis

The baseline confidence distribution was strongly concentrated toward high values:

- Median confidence: **0.9988**
- Minimum confidence: **0.2849**
- Maximum confidence: **1.0000**
- Mean confidence on correct predictions: **0.9727**

A total of **55 high-confidence errors** were identified using a threshold of **0.80**.

- High-confidence error rate over all evaluation samples: **1.36%**
- Fraction of all errors represented by high-confidence errors: **35.95%**
- Mean confidence of high-confidence errors: **0.9187**

This indicates that a substantial portion of the model's errors are not simply low-confidence or ambiguous predictions. The model is highly confident in a significant subset of incorrect decisions.

> Note: High confidence should not automatically be interpreted as poor calibration. Calibration was not directly evaluated in this baseline analysis.

---

##### 5. Visual Error Analysis

Visual inspection of high-confidence errors showed recurring patterns involving:

- Shared vegetation appearance
- Similar texture patterns
- Ambiguous blue/green regions
- Similar agricultural structures
- Shared elongated or linear spatial patterns
- Overlap between built-up environments

These observations support the hypothesis that many remaining errors are concentrated around **class boundaries with strong visual overlap**.

---

##### 6. Grad-CAM Findings

Grad-CAM analysis was performed on representative high-confidence errors.

The main observations were:

1. In several vegetation-related confusions, the model's predicted-class activation was concentrated on broad vegetation/texture regions rather than a clearly class-specific structure.

2. In SeaLake → Forest examples, predicted Forest activations were relatively broad and did not consistently correspond to an obvious forest-specific object or structure.

3. In AnnualCrop/River and River/Highway examples, activations frequently aligned with elongated spatial structures, suggesting that shared geometric patterns may contribute to these confusions.

4. The Grad-CAM results provide supporting evidence about the model's spatial focus, but they do not establish causal explanations for the errors.

---

##### 7. Overall Error Patterns

The baseline errors can be broadly grouped into three categories:

##### A. Shared Visual Features

Examples:

- Pasture ↔ Forest
- PermanentCrop ↔ HerbaceousVegetation
- AnnualCrop ↔ PermanentCrop

These classes share vegetation, texture, or agricultural characteristics.

##### B. Ambiguous / Weak Visual Evidence

Example:

- SeaLake → Forest

Some samples contain limited class-specific visual cues, making the distinction difficult even during visual inspection.

##### C. Shared Spatial Structures

Examples:

- AnnualCrop → River
- River → Highway
- Highway → River

The model may detect meaningful spatial structures but associate them with the wrong semantic class.

---

##### 8. Key Baseline Conclusions

The baseline model demonstrates strong overall performance, but its remaining errors are concentrated around a limited number of recurring class boundaries.

The most important failure modes identified are:

1. **SeaLake → Forest**
2. **Pasture → Forest**
3. **PermanentCrop → HerbaceousVegetation**
4. **AnnualCrop → PermanentCrop**
5. **AnnualCrop → River**

Several of these errors occur with very high confidence, indicating that the model has learned strong but sometimes non-discriminative visual representations for the affected class boundaries.

The current evidence suggests that the main challenge is not general classification failure, but **fine-grained discrimination between visually overlapping classes**.

---

##### 9. Implications for the Next Phase

The Error Analysis does not yet justify an arbitrary architecture change.

Instead, the findings should guide the next experimentation phase.

The next stage should investigate whether targeted retraining and hyperparameter optimization can improve the identified failure modes while preserving the strong baseline performance.

Potential interventions should be selected based on the observed failure modes rather than applied blindly.

**Baseline Error Analysis Status: Completed**

### 2.10.10 Findings → Model / Dataset Improvement

#### 1. Purpose of This Section

The Error Analysis shows that the baseline model is already performing well overall, but the remaining errors are concentrated around a small number of difficult class boundaries.

The purpose of this section is to turn these findings into practical improvement hypotheses for the next phase.

At this stage, these observations are treated as **hypotheses, not confirmed causes**. The next phase will use controlled experiments to determine whether any of these factors have a meaningful effect on the identified failure patterns.

The goal is not simply to increase overall accuracy, but to understand which changes, if any, can improve the specific weaknesses identified during Error Analysis.

---

#### 2. Main Improvement Hypotheses

Based on the Error Analysis, three main explanations should be considered for the remaining errors:

* **Limited spatial information in the current input size**
* **Visual ambiguity between some classes**
* **Shared visual representations between visually similar classes**

These explanations are not mutually exclusive, and the current evidence is not sufficient to identify one of them as the single cause of the observed errors.

The baseline already provides strong overall performance, so a major architecture change is not justified at this stage.

The next phase should therefore focus on targeted and controlled experiments before increasing model complexity.

---

#### 3. Limited Spatial Information

The current model uses `64×64×3` RGB inputs.

Some of the remaining errors involve relatively fine-grained class distinctions or spatial patterns, particularly:

* `AnnualCrop → PermanentCrop`
* `AnnualCrop → River`
* `Pasture → Forest`

This raises the possibility that the current input size may limit the spatial representation available to the CNN for some difficult samples.

This does not necessarily mean that the source images are of poor quality.

A more precise hypothesis is:

> The current `64×64` input size may not provide the most effective spatial representation for learning some of the difficult class boundaries.

This hypothesis will be evaluated through a controlled input-size experiment in the next phase.

**Priority: High**

---

#### 4. Visual Ambiguity

Several of the major confusion pairs contain samples that are difficult to distinguish even during visual inspection.

This is particularly relevant to:

* `SeaLake → Forest`
* `Pasture → Forest`
* `PermanentCrop → HerbaceousVegetation`

Some `SeaLake` samples contain limited class-specific visual information and are dominated by broad blue/green regions.

Similarly, `Pasture`, `Forest`, `PermanentCrop`, and `HerbaceousVegetation` can share vegetation-related appearance and texture.

This suggests that some remaining errors may be related to the inherent visual ambiguity of the dataset rather than being caused entirely by model limitations.

Representative samples should therefore continue to be considered when interpreting the results of future experiments.

**Priority: High**

---

#### 5. Shared Visual Representation

The Error Analysis and Grad-CAM results suggest that the model sometimes responds to visual patterns that are shared by multiple classes.

Examples include:

* shared vegetation regions in `Pasture → Forest`
* similar vegetation and texture patterns in `PermanentCrop → HerbaceousVegetation`
* similar agricultural structures in `AnnualCrop → PermanentCrop`
* elongated spatial structures in `AnnualCrop → River` and other infrastructure-related confusions

These observations do not prove that a specific visual feature is the direct cause of a prediction.

For example, the `Pasture → Forest` errors do not provide enough evidence to conclude that the model is simply detecting trees.

A more cautious interpretation is that shared vegetation, texture, or spatial features may contribute to similar internal representations for different classes.

If earlier experiments do not sufficiently reduce these errors, model representation will become a stronger candidate for further investigation.

**Priority: Medium / Later Stage**

---

#### 6. Data Augmentation

Data augmentation has already been explored during the development of the baseline model using different augmentation settings.

Some configurations improved performance, while others reduced it.

For this reason, additional augmentation is not treated as the first new intervention after Error Analysis.

The previous augmentation experiments will still be considered when interpreting future results, but further augmentation will only be introduced if later evidence provides a specific reason to revisit it.

This avoids changing the augmentation pipeline simply because errors remain.

**Current priority: Not the first intervention**

---

#### 7. High-Confidence Errors as an Improvement Signal

The baseline contains `55` high-confidence errors, representing approximately `35.95%` of all errors.

This indicates that a meaningful portion of the remaining errors are not simply low-confidence predictions.

However, this result is not by itself evidence of a calibration problem.

For the next phase, high-confidence errors will instead be used as an additional signal for evaluating whether an intervention improves the difficult class boundaries.

A useful improvement should ideally reduce important high-confidence failure patterns while preserving the strong overall performance of the model.

---

#### 8. Priority of the Main Error Patterns

Based on error frequency, confidence, visual inspection, and Grad-CAM evidence, the initial focus should remain on the following error patterns.

**Highest priority:**

1. `SeaLake → Forest`
2. `Pasture → Forest`
3. `PermanentCrop → HerbaceousVegetation`

These errors combine relatively large confusion counts with strong evidence of visual overlap or high-confidence misclassification.

**Medium priority:**

4. `AnnualCrop → PermanentCrop`

This appears to be a more fine-grained distinction between visually similar agricultural classes.

**Lower priority for the initial experiments:**

5. `AnnualCrop → River`

This remains relevant, but its lower high-confidence fraction suggests a less consistently confident failure pattern compared with some of the other major confusions.

Other urban and infrastructure-related confusions will continue to be monitored, but they will not be the primary target of the first intervention.

---

#### 9. Improvement Strategy

The next phase should follow a controlled experimental process:

**Error Pattern → Hypothesis → One Main Change → Retraining → Evaluation → Decision**

The purpose of this approach is to make the results interpretable.

When practical, only one major factor should be changed at a time. The baseline configuration should otherwise remain unchanged so that any observed improvement or regression can be more confidently linked to the tested intervention.

The test set must remain locked during this phase.

Validation results and error patterns will be used to compare experiments against the baseline, while the final test evaluation will be reserved for the final selected model.

The detailed experiment definitions and execution steps will be documented in the following **Phase 1 Findings & Action Plan** section.

---

#### 10. Expected Outcome

The purpose of the next phase is to determine whether the identified hypotheses can explain or reduce the main failure patterns of the baseline model.

A successful improvement should ideally:

* reduce the main recurring confusion pairs,
* reduce important high-confidence errors,
* improve difficult classes without significantly hurting other classes,
* and maintain or improve the strong overall baseline performance.

If an intervention produces little or no meaningful improvement, that result will also be useful because it provides evidence against that particular hypothesis and helps guide the next experiment.

The objective is not to keep changing the model until a higher accuracy is obtained, but to use evidence from the Error Analysis to make informed engineering decisions.


## 2.11 Phase 1 Findings & Action Plan

### 2.11.1 Findings Summary

The baseline model achieves strong overall performance (~96.23% validation accuracy), but Error Analysis identified several persistent failure patterns.

The most important confusion pairs are:

* `SeaLake → Forest`
* `PermanentCrop → HerbaceousVegetation`
* `Pasture → Forest`
* `AnnualCrop → River`
* `AnnualCrop → PermanentCrop`

High-confidence errors also indicate that a subset of incorrect predictions are made with relatively high model confidence, suggesting that some failures are not simply caused by low-confidence uncertainty.

Grad-CAM and visual inspection provide supporting evidence that several difficult cases involve:

* shared vegetation and texture patterns,
* visual ambiguity between classes,
* ambiguous blue/green regions,
* similar agricultural structures,
* elongated or linear spatial patterns.

These observations lead to three working hypotheses:

1. **Limited Spatial Information / Input Size**

   * The `64×64` input may limit the spatial representation available for some difficult class boundaries.

2. **Visual Ambiguity**

   * Some classes may remain visually ambiguous even when sufficient spatial representation is available.

3. **Shared Visual Representation**

   * The model may learn partially shared representations for visually similar classes, resulting in insufficient class discrimination in some regions of the feature space.

These are hypotheses rather than confirmed causal explanations. Grad-CAM and visual inspection are treated as supporting evidence, not causal proof.


### 2.11.2 Improvement Priorities

Phase 1 focuses on controlled interventions that directly target the failure patterns identified during Error Analysis.

The first experiment evaluates whether increasing input resolution can provide a finer spatial representation while keeping the rest of the training setup as consistent as possible.

Existing augmentation experiments are treated as prior evidence and are not selected as the first intervention.

The intervention order will therefore be evidence-driven: each experiment will be evaluated before deciding whether to retain, modify, or move to the next intervention.


### 2.11.3 Experiment 1 — Higher Input Size

#### Hypothesis

Increasing the input resolution from `64×64` to `128×128` may provide the CNN with a finer spatial representation and improve discrimination between visually similar classes involved in the major baseline failure patterns.

##### Experimental Setup

The primary intervention is:

`64×64×3 → 128×128×3`

The following components should remain unchanged:

* Dataset
* Train/Validation split
* Locked Test Set
* Model architecture and layer configuration
* Number of filters
* Kernel sizes and strides
* Optimizer
* Learning rate
* Loss function
* Existing augmentation pipeline
* Training budget
* Callbacks
* Early stopping configuration
* Checkpoint selection criterion
* Evaluation protocol

Because the model uses `GlobalAveragePooling2D` before the classification head, changing the input resolution does not change the dimensionality of the classifier input or the number of model parameters, assuming the architecture itself remains unchanged.

Batch size should remain unchanged at `32` unless GPU memory limitations require a reduction. Any such change must be explicitly recorded because it introduces an additional experimental variable.

The same random seed and initialization policy should be used where possible.

##### Configuration

| Component       |  Baseline | Experiment 1 |
| --------------- | --------: | -----------: |
| Input Size      | `64×64×3` |  `128×128×3` |
| Architecture    |      Same |         Same |
| Filters         |      Same |         Same |
| Optimizer       |      Same |         Same |
| Learning Rate   |      Same |         Same |
| Loss            |      Same |         Same |
| Augmentation    |      Same |         Same |
| Batch Size      |      `32` |        `32`* |
| Dataset         |      Same |         Same |
| Train/Val Split |      Same |         Same |
| Test Set        |    Locked |       Locked |

* Reduced only if required by GPU memory constraints, and explicitly recorded.

##### Important Experimental Consideration

Increasing the input size does not create new information from the original image source. Instead, it provides the network with a higher-resolution spatial representation.

Therefore, any observed improvement should be interpreted as evidence that the finer spatial representation may help the model process or distinguish relevant visual structures. It should not be interpreted as proof that new image information has been introduced.

The use of `GlobalAveragePooling2D` ensures that the classifier head receives the same feature dimension (`256`) for both input resolutions. As a result, the total number of trainable parameters is expected to remain unchanged.

However, the spatial dimensions of intermediate feature maps increase with the larger input. Consequently, computational cost and memory usage will increase.

##### Training

Train the `128×128` model using the same training procedure as the baseline, including:

* the same optimizer and learning rate,
* the same augmentation pipeline,
* the same epoch budget,
* the same callbacks,
* the same checkpoint-selection criterion,
* and the same logging protocol.

The baseline model and locked test set must not be modified.

##### Evaluation

The experiment will be evaluated using the same validation protocol as the baseline.

###### Primary Metrics

* Validation Accuracy
* Per-class Recall
* Per-class F1
* Main confusion pairs

###### Secondary Metrics

* Validation Loss
* High-confidence error count

##### Error Analysis Comparison

The following baseline failure patterns will be compared directly before and after the intervention:

* `SeaLake → Forest`
* `PermanentCrop → HerbaceousVegetation`
* `Pasture → Forest`
* `AnnualCrop → River`
* `AnnualCrop → PermanentCrop`

The experiment will not be considered successful solely because overall accuracy increases. Improvement should also be reflected in the diagnosed failure patterns while avoiding important regressions in other classes.

##### Decision

**Meaningful Improvement**

Overall performance improves, major failure patterns show meaningful reduction, and no important regression is observed.

**Marginal Improvement**

Overall metrics improve slightly, but the main confusion patterns remain approximately unchanged.

**No Meaningful Improvement**

Performance changes are negligible, major failure patterns remain essentially unchanged, or important regression is introduced.

The decision will be based on the combined evidence rather than on validation accuracy alone.


#### 2.11.3.1 Experiment Configuration

In [ ]:
# ============================================
# 2.11.3.1 Experiment Configuration
# ============================================

# --------------------------------------------
# Project Information
# --------------------------------------------

EXPERIMENT_PROJECT_NAME = "CNN_Image_Classification"
EXPERIMENT_PROJECT_VERSION = "1.0"

# Identify the model used in this experiment.
EXPERIMENT_MODEL_NAME = "high"  # "high" | "low" | "mobilenetv2"

EXPERIMENT_RUN_TYPE = "input_128"
EXPERIMENT_INTERVENTION = "higher_input_size"


# --------------------------------------------
# Dataset Configuration
# --------------------------------------------

EXPERIMENT_IMAGE_SIZE = (128, 128)
EXPERIMENT_IMAGE_CHANNELS = 3
EXPERIMENT_NUM_CLASSES = 10


# --------------------------------------------
# Data Loading Configuration
# --------------------------------------------

EXPERIMENT_BATCH_SIZE = 32
EXPERIMENT_EPOCHS = 50


# --------------------------------------------
# Project Paths
# --------------------------------------------

# Assumes the notebook is run from the project root directory.
EXPERIMENT_PROJECT_ROOT = Path.cwd()

EXPERIMENT_OUTPUT_DIR = (
    EXPERIMENT_PROJECT_ROOT / "outputs"
)

# Experiment-specific subdirectories so that Experiment 1
# never overwrites baseline outputs.
EXPERIMENT_MODEL_OUTPUT_DIR = (
     EXPERIMENT_OUTPUT_DIR / EXPERIMENT_MODEL_NAME/ EXPERIMENT_RUN_TYPE
)

EXPERIMENT_CHECKPOINT_DIR = (
    EXPERIMENT_MODEL_OUTPUT_DIR / "checkpoints"
)

EXPERIMENT_REPORTS_DIR = (
    EXPERIMENT_MODEL_OUTPUT_DIR / "reports"
)

EXPERIMENT_MODEL_DIR = (
    EXPERIMENT_MODEL_OUTPUT_DIR / "models"
)

EXPERIMENT_PLOTS_DIR = (
    EXPERIMENT_MODEL_OUTPUT_DIR / "plots"
)


# --------------------------------------------
# Create Required Experiment Output Directories
# --------------------------------------------

for directory in [
    EXPERIMENT_OUTPUT_DIR,
    EXPERIMENT_MODEL_OUTPUT_DIR,
    EXPERIMENT_CHECKPOINT_DIR,
    EXPERIMENT_REPORTS_DIR,
    EXPERIMENT_MODEL_DIR,
    EXPERIMENT_PLOTS_DIR,
]:
    directory.mkdir(parents=True, exist_ok=True)


# --------------------------------------------
# Configuration Summary
# --------------------------------------------

print("Experiment Configuration")
print("-" * 40)

print(f"Project Name       : {EXPERIMENT_PROJECT_NAME}")
print(f"Project Version    : {EXPERIMENT_PROJECT_VERSION}")
print(f"Model Name         : {EXPERIMENT_MODEL_NAME}")
print(f"Experiment Type    : {EXPERIMENT_RUN_TYPE}")
print(f"Intervention       : {EXPERIMENT_INTERVENTION}")
print(f"Image Size         : {EXPERIMENT_IMAGE_SIZE}")
print(f"Image Channels     : {EXPERIMENT_IMAGE_CHANNELS}")
print(f"Number of Classes  : {EXPERIMENT_NUM_CLASSES}")
print(f"Batch Size         : {EXPERIMENT_BATCH_SIZE}")
print(f"Epochs             : {EXPERIMENT_EPOCHS}")


print(f"Project Root       : {EXPERIMENT_PROJECT_ROOT}")
print(f"Output Directory   : {EXPERIMENT_MODEL_OUTPUT_DIR}")
print(f"Checkpoint Dir     : {EXPERIMENT_CHECKPOINT_DIR}")
print(f"Reports Dir        : {EXPERIMENT_REPORTS_DIR}")
print(f"Model Dir          : {EXPERIMENT_MODEL_DIR}")
print(f"Plots Dir          : {EXPERIMENT_PLOTS_DIR}")

#### 2.11.3.2 Experiment Dataset Configuration & Dataset Loading

In [ ]:
# ============================================
# 2.11.3.2 Experiment Dataset
# ============================================
#
# Note: The dataset is already split into train/val/test on disk.
# No in-code splitting or stratification is performed here.
#
# This experiment is designed for execution on a local system.
# No Google Colab or Google Drive dependencies are required.
#
# The test set remains locked and is NOT loaded into a tf.data.Dataset
# here. It will be loaded only when the official test-set evaluation
# is performed in Section 4.2.


# --------------------------------------------
# Dataset Configuration
# --------------------------------------------

# Dataset folder name.
EXPERIMENT_DATASET_NAME = "EuroSAT_RGB"
EXPERIMENT_DATA_DIR = PROJECT_ROOT / "data"
# Dataset is stored directly under the experiment data directory.
EXPERIMENT_DATASET_DIR = (
    EXPERIMENT_DATA_DIR
    / EXPERIMENT_DATASET_NAME
)

EXPERIMENT_TRAIN_DIR = (
    EXPERIMENT_DATASET_DIR
    / "train"
)

EXPERIMENT_VAL_DIR = (
    EXPERIMENT_DATASET_DIR
    / "val"
)

EXPERIMENT_TEST_DIR = (
    EXPERIMENT_DATASET_DIR
    / "test"
)


# --------------------------------------------
# Validate Dataset Structure
# --------------------------------------------

if not EXPERIMENT_DATASET_DIR.exists():
    raise FileNotFoundError(
        f"Dataset directory not found: {EXPERIMENT_DATASET_DIR}"
    )

for split_name, split_dir in [
    ("train", EXPERIMENT_TRAIN_DIR),
    ("validation", EXPERIMENT_VAL_DIR),
    ("test", EXPERIMENT_TEST_DIR),
]:
    if not split_dir.exists():
        raise FileNotFoundError(
            f"{split_name.capitalize()} directory not found: "
            f"{split_dir}"
        )


# --------------------------------------------
# Class Names
# --------------------------------------------

# Extract class names from the experiment training directory.
# Each subdirectory represents one class.
EXPERIMENT_CLASS_NAMES = sorted(
    directory.name
    for directory in EXPERIMENT_TRAIN_DIR.iterdir()
    if directory.is_dir()
)

if not EXPERIMENT_CLASS_NAMES:
    raise ValueError(
        f"No class directories found in training directory: "
        f"{EXPERIMENT_TRAIN_DIR}"
    )


# --------------------------------------------
# Validate Class Consistency Across Splits
# --------------------------------------------

# Ensure validation/test directories contain exactly the same
# classes as train. Fail fast if there is any mismatch.
for split_name, split_dir in [
    ("validation", EXPERIMENT_VAL_DIR),
    ("test", EXPERIMENT_TEST_DIR),
]:
    split_classes = sorted(
        directory.name
        for directory in split_dir.iterdir()
        if directory.is_dir()
    )

    if split_classes != EXPERIMENT_CLASS_NAMES:
        missing = (
            set(EXPERIMENT_CLASS_NAMES)
            - set(split_classes)
        )

        extra = (
            set(split_classes)
            - set(EXPERIMENT_CLASS_NAMES)
        )

        raise ValueError(
            f"Class mismatch in {split_name} directory "
            f"({split_dir}).\n"
            f"Missing: "
            f"{sorted(missing) if missing else 'None'}\n"
            f"Extra   : "
            f"{sorted(extra) if extra else 'None'}"
        )


# --------------------------------------------
# Validate Class Configuration
# --------------------------------------------

if len(EXPERIMENT_CLASS_NAMES) != EXPERIMENT_NUM_CLASSES:
    raise ValueError(
        f"Expected {EXPERIMENT_NUM_CLASSES} classes, "
        f"but found {len(EXPERIMENT_CLASS_NAMES)} classes "
        f"in {EXPERIMENT_TRAIN_DIR}"
    )


# --------------------------------------------
# Dataset Loading
# --------------------------------------------

def load_experiment_dataset(
    directory: Path,
    image_size: tuple[int, int],
    batch_size: int,
    class_names: list[str],
    shuffle: bool,
    seed: int | None = None,
) -> tf.data.Dataset:
    """Load an experiment dataset from a directory structure.

    Args:
        directory: Path to the split directory containing one
            subdirectory per class.
        image_size: Target (height, width) for image resizing.
        batch_size: Number of samples per batch.
        class_names: Explicit ordered list of class names. Ensures
            consistent label indices across train and validation.
        shuffle: Whether to shuffle the dataset.
        seed: Random seed used when shuffling is enabled.

    Returns:
        A batched tf.data.Dataset containing images and one-hot labels.
    """

    if not directory.exists():
        raise FileNotFoundError(
            f"Dataset split directory not found: {directory}"
        )

    dataset = tf.keras.utils.image_dataset_from_directory(
        directory,
        labels="inferred",
        label_mode="categorical",
        class_names=class_names,
        image_size=image_size,
        batch_size=batch_size,
        shuffle=shuffle,
        seed=seed,
    )

    return dataset


# --------------------------------------------
# Train / Validation Pipeline
# --------------------------------------------

experiment_train_ds = load_experiment_dataset(
    EXPERIMENT_TRAIN_DIR,
    EXPERIMENT_IMAGE_SIZE,
    EXPERIMENT_BATCH_SIZE,
    EXPERIMENT_CLASS_NAMES,
    shuffle=True,
    seed=SEED,
)

experiment_val_ds = load_experiment_dataset(
    EXPERIMENT_VAL_DIR,
    EXPERIMENT_IMAGE_SIZE,
    EXPERIMENT_BATCH_SIZE,
    EXPERIMENT_CLASS_NAMES,
    shuffle=False,
)


# --------------------------------------------
# Test-Set Lock
# --------------------------------------------
#
# The test set is intentionally NOT loaded as test_ds here.
# Only its directory structure and class consistency were validated
# above. The official test-set evaluation remains reserved for the
# final evaluation stage.


# --------------------------------------------
# Performance Optimization
# --------------------------------------------

EXPERIMENT_AUTOTUNE = tf.data.AUTOTUNE

experiment_train_ds = (
    experiment_train_ds.prefetch(
        buffer_size=EXPERIMENT_AUTOTUNE
    )
)

experiment_val_ds = (
    experiment_val_ds.prefetch(
        buffer_size=EXPERIMENT_AUTOTUNE
    )
)


# --------------------------------------------
# Dataset Summary
# --------------------------------------------

print("Experiment Dataset Summary")
print("-" * 40)

print(
    f"Dataset Name       : "
    f"{EXPERIMENT_DATASET_NAME}"
)

print(
    f"Dataset Directory  : "
    f"{EXPERIMENT_DATASET_DIR}"
)

print(
    f"Train Directory    : "
    f"{EXPERIMENT_TRAIN_DIR}"
)

print(
    f"Validation Dir     : "
    f"{EXPERIMENT_VAL_DIR}"
)

print(
    f"Test Directory     : "
    f"{EXPERIMENT_TEST_DIR}"
)

print(
    f"Number of Classes  : "
    f"{len(EXPERIMENT_CLASS_NAMES)}"
)

print(
    f"Image Size         : "
    f"{EXPERIMENT_IMAGE_SIZE}"
)

print(
    f"Batch Size         : "
    f"{EXPERIMENT_BATCH_SIZE}"
)

print(
    f"Train batches      : "
    f"{experiment_train_ds.cardinality().numpy()}"
)

print(
    f"Validation batches : "
    f"{experiment_val_ds.cardinality().numpy()}"
)


# --------------------------------------------
# Inspect Tensor Shapes
# --------------------------------------------

for images, labels in experiment_train_ds.take(1):
    print("\nSample Batch")

    print(
        f"Images shape       : "
        f"{images.shape}"
    )

    print(
        f"Labels shape       : "
        f"{labels.shape}"
    )

#### 2.11.3.3 Experiment Dataset Sanity Checks

In [ ]:
# ============================================
# 2.11.3.3 Experiment Dataset Sanity Checks
# ============================================

# Note:
# This section performs lightweight validation checks on the
# train/validation datasets used by the training pipeline.
#
# The train/validation datasets are loaded from the local Colab VM
# according to Section 1.5.
#
# The test set remains locked and is NOT loaded into a tf.data.Dataset
# here. Only its directory structure is validated against the class
# configuration defined in Section 1.4.
#
# No preprocessing, augmentation, splitting, or model evaluation
# is performed here.


# --------------------------------------------
# Expected Tensor Configuration
# --------------------------------------------

EXPECTED_IMAGE_SHAPE = (
    *IMAGE_SIZE,
    IMAGE_CHANNELS,
)

EXPECTED_LABEL_SHAPE = (
    NUM_CLASSES,
)


# --------------------------------------------
# Validate a Dataset Batch
# --------------------------------------------

def validate_dataset_batch(
    dataset: tf.data.Dataset,
    dataset_name: str,
) -> None:
    """Run sanity checks on the first batch of a dataset."""

    for images, labels in dataset.take(1):

        # Check image tensor shape.
        if images.shape.rank != 4:
            raise ValueError(
                f"{dataset_name}: Expected image tensor with rank 4, "
                f"got shape {images.shape}."
            )

        if tuple(images.shape[1:]) != EXPECTED_IMAGE_SHAPE:
            raise ValueError(
                f"{dataset_name}: Expected image shape "
                f"(batch, {EXPECTED_IMAGE_SHAPE[0]}, "
                f"{EXPECTED_IMAGE_SHAPE[1]}, "
                f"{EXPECTED_IMAGE_SHAPE[2]}), "
                f"got {images.shape}."
            )

        # Check label tensor shape.
        if labels.shape.rank != 2:
            raise ValueError(
                f"{dataset_name}: Expected one-hot labels with rank 2, "
                f"got shape {labels.shape}."
            )

        if labels.shape[-1] != NUM_CLASSES:
            raise ValueError(
                f"{dataset_name}: Expected {NUM_CLASSES} label columns, "
                f"got {labels.shape[-1]}."
            )

        # Check image dtype.
        # image_dataset_from_directory returns float32 by default,
        # even though values are still in the raw [0, 255] range
        # (normalization is deferred to the model's Rescaling layer).
        if images.dtype != tf.float32:
            raise TypeError(
                f"{dataset_name}: Expected images with dtype float32 "
                f"(raw [0, 255] range), got {images.dtype}."
            )

        # Check image value range.
        image_min = tf.reduce_min(images).numpy()
        image_max = tf.reduce_max(images).numpy()

        if image_min < 0 or image_max > 255:
            raise ValueError(
                f"{dataset_name}: Image values must be in [0, 255], "
                f"got range [{image_min}, {image_max}]."
            )

        # Check for invalid numeric values.
        if not tf.reduce_all(
            tf.math.is_finite(images)
        ):
            raise ValueError(
                f"{dataset_name}: Images contain NaN or Inf values."
            )

        if not tf.reduce_all(
            tf.math.is_finite(labels)
        ):
            raise ValueError(
                f"{dataset_name}: Labels contain NaN or Inf values."
            )

        # Check one-hot encoding.
        label_sums = tf.reduce_sum(
            labels,
            axis=1,
        )

        if not tf.reduce_all(
            tf.equal(label_sums, 1.0)
        ):
            raise ValueError(
                f"{dataset_name}: "
                f"Labels are not valid one-hot vectors."
            )

        if not tf.reduce_all(
            tf.logical_or(
                tf.equal(labels, 0.0),
                tf.equal(labels, 1.0),
            )
        ):
            raise ValueError(
                f"{dataset_name}: "
                f"One-hot labels must contain only 0 or 1."
            )

        print(
            f"{dataset_name:<12} "
            f"✓ images={images.shape} | "
            f"labels={labels.shape} | "
            f"dtype={images.dtype.name} | "
            f"range=[{image_min}, {image_max}]"
        )

        return

    raise ValueError(
        f"{dataset_name}: Dataset is empty."
    )


# --------------------------------------------
# Validate Train / Validation Datasets
# --------------------------------------------

print("Dataset Sanity Checks")
print("-" * 60)

datasets_to_check = [
    ("Train", train_ds),
    ("Validation", val_ds),
]

for dataset_name, dataset in datasets_to_check:
    validate_dataset_batch(
        dataset,
        dataset_name,
    )


# --------------------------------------------
# Dataset Cardinality Checks
# --------------------------------------------

for dataset_name, dataset in datasets_to_check:

    cardinality = (
        dataset.cardinality().numpy()
    )

    if cardinality <= 0:
        raise ValueError(
            f"{dataset_name}: "
            f"Dataset contains no batches."
        )

    print(
        f"{dataset_name:<12} "
        f"✓ batches={cardinality}"
    )


# --------------------------------------------
# Validate Test-Set Directory Structure
# --------------------------------------------

# The test set remains locked.
# Only directory names are inspected.

if not TEST_DIR.exists():
    raise FileNotFoundError(
        f"Test dataset directory not found: {TEST_DIR}"
    )

if not TRAIN_DIR.exists():
    raise FileNotFoundError(
        f"Train dataset directory not found: {TRAIN_DIR}"
    )


# Extract class names directly from the training directory.
CLASS_NAMES = sorted(
    directory.name
    for directory in TRAIN_DIR.iterdir()
    if directory.is_dir()
)

if not CLASS_NAMES:
    raise ValueError(
        f"No class directories found in: {TRAIN_DIR}"
    )


# Extract class names from the test directory.
test_classes = sorted(
    directory.name
    for directory in TEST_DIR.iterdir()
    if directory.is_dir()
)


# Validate class consistency.
if test_classes != CLASS_NAMES:

    missing = (
        set(CLASS_NAMES) - set(test_classes)
    )

    extra = (
        set(test_classes) - set(CLASS_NAMES)
    )

    raise ValueError(
        f"Test: Class mismatch in test directory "
        f"({TEST_DIR}).\n"
        f"Missing: "
        f"{sorted(missing) if missing else 'None'}\n"
        f"Extra   : "
        f"{sorted(extra) if extra else 'None'}"
    )

print(
    f"Test         ✓ directory structure valid | "
    f"classes={len(test_classes)} | "
    f"tf.data loading skipped (test-set locked)"
)


# --------------------------------------------
# Final Sanity Check
# --------------------------------------------

print(
    "\nAll dataset sanity checks passed successfully."
)

#### 2.11.3.4 Experiment Model

In [ ]:
# ==============================
# 2.11.3.4 Experiment Model
# ==============================

# --------------------------------------------
# Build Experiment Model
# --------------------------------------------

experiment_input_shape = (
    *EXPERIMENT_IMAGE_SIZE,
    EXPERIMENT_IMAGE_CHANNELS,
)

experiment_model = build_custom_cnn_high_flops(
    input_shape=experiment_input_shape,
    num_classes=EXPERIMENT_NUM_CLASSES,
    augmentation=augmentation,
)


# --------------------------------------------
# Model Summary
# --------------------------------------------

experiment_model.summary()


# --------------------------------------------
# Parameter Summary
# --------------------------------------------

print("\nExperiment Model Parameters")
print("-" * 40)

print(
    f"Total parameters     : "
    f"{experiment_model.count_params():,}"
)

print(
    f"Input shape          : "
    f"{experiment_model.input_shape}"
)

#### 2.11.3.5 Experiment Compilation

In [ ]:
# ============================================
# 2.11.3.5 Experiment Compilation
# ============================================

import tensorflow as tf


# --------------------------------------------
# Optimizer & Loss Configuration
# --------------------------------------------

EXPERIMENT_LEARNING_RATE = 1e-4
EXPERIMENT_WEIGHT_DECAY = 1e-4

EXPERIMENT_OPTIMIZER_NAME = "AdamW"
EXPERIMENT_LOSS_NAME = "CategoricalCrossentropy"

# --------------------------------------------
# Optimizer
# --------------------------------------------

experiment_optimizer = tf.keras.optimizers.AdamW(
    learning_rate=EXPERIMENT_LEARNING_RATE,
    weight_decay=EXPERIMENT_WEIGHT_DECAY,
)


# --------------------------------------------
# Model Compilation
# --------------------------------------------

experiment_model.compile(
    optimizer=experiment_optimizer,
    loss=tf.keras.losses.CategoricalCrossentropy(),
    metrics=[
        tf.keras.metrics.CategoricalAccuracy(
            name="accuracy"
        )
    ],
)


print("Experiment model compilation completed successfully.")
print(f"Optimizer     : {EXPERIMENT_OPTIMIZER_NAME}")
print(f"Learning Rate : {EXPERIMENT_LEARNING_RATE}")
print(f"Weight Decay  : {EXPERIMENT_WEIGHT_DECAY}")
print(f"Loss Function : {EXPERIMENT_LOSS_NAME}")

#### 2.11.3.6 Experiment Output & Tracking Setup

In [ ]:
# ============================================
# 2.11.3.6 Experiment Output & Tracking Setup
# ============================================

# --------------------------------------------
# Run Identity
# --------------------------------------------

EXPERIMENT_PHASE = "phase-1"
EXPERIMENT_MODEL_DISPLAY_NAME = "High-FLOPs Custom CNN"

EXPERIMENT_WANDB_RUN_STATE_PATH = os.path.join(
    EXPERIMENT_REPORTS_DIR,
    f"{EXPERIMENT_MODEL_NAME}_{EXPERIMENT_PHASE}_{EXPERIMENT_RUN_TYPE}_wandb_run_state.json",
)


# --------------------------------------------
# Experiment W&B Run Configuration
# --------------------------------------------

EXPERIMENT_WANDB_CONFIG = build_wandb_config(
    model_name=EXPERIMENT_MODEL_DISPLAY_NAME,
    model_type=EXPERIMENT_MODEL_NAME,
    image_size=EXPERIMENT_IMAGE_SIZE,
    image_channels=EXPERIMENT_IMAGE_CHANNELS,
    num_classes=EXPERIMENT_NUM_CLASSES,
    batch_size=EXPERIMENT_BATCH_SIZE,
    epochs=EXPERIMENT_EPOCHS,
    optimizer_name=EXPERIMENT_OPTIMIZER_NAME,
    learning_rate=EXPERIMENT_LEARNING_RATE,
    loss_name=EXPERIMENT_LOSS_NAME,
    extra_config={
        "project_name": EXPERIMENT_PROJECT_NAME,
        "project_version": EXPERIMENT_PROJECT_VERSION,
        "optimizer_weight_decay": EXPERIMENT_WEIGHT_DECAY,
        "experiment": EXPERIMENT_RUN_TYPE,
        "intervention": EXPERIMENT_INTERVENTION,
    },
)


# --------------------------------------------
# Initialize or Resume W&B Run
# --------------------------------------------
# If a run-state file already exists (e.g. from before a Colab
# runtime reset), resume that exact experiment run instead of
# creating a new one, so training/evaluation logs stay on one timeline.

if os.path.isfile(EXPERIMENT_WANDB_RUN_STATE_PATH):
    previous_state = load_wandb_run_state(
        EXPERIMENT_WANDB_RUN_STATE_PATH
    )

    experiment_wandb_run = initialize_wandb_run(
        model_type=EXPERIMENT_MODEL_NAME,
        phase=EXPERIMENT_PHASE,
        experiment_type=EXPERIMENT_RUN_TYPE,
        run_id=previous_state["run_id"],
    )

    print("Resumed existing Experiment 1 W&B run.")

else:
    experiment_wandb_run = initialize_wandb_run(
        model_type=EXPERIMENT_MODEL_NAME,
        phase=EXPERIMENT_PHASE,
        experiment_type=EXPERIMENT_RUN_TYPE,
        config=EXPERIMENT_WANDB_CONFIG,
    )

    save_wandb_run_state(
        experiment_wandb_run,
        EXPERIMENT_WANDB_RUN_STATE_PATH,
    )

    print("Experiment 1 W&B run initialized successfully.")


print(f"Project         : {WANDB_PROJECT}")
print(f"Run Name        : {experiment_wandb_run.name}")
print(f"Run ID          : {experiment_wandb_run.id}")
print(f"Model Name      : {EXPERIMENT_MODEL_DISPLAY_NAME}")
print(f"Phase           : {EXPERIMENT_PHASE}")
print(f"Experiment Type : {EXPERIMENT_RUN_TYPE}")
print(f"Intervention    : {EXPERIMENT_INTERVENTION}")
print(f"Run State       : {EXPERIMENT_WANDB_RUN_STATE_PATH}")

#### 2.11.3.7 Experiment Training

In [ ]:
# ============================================
# 2.11.3.7 Experiment Training
# ============================================

# --------------------------------------------
# Training Callbacks
# --------------------------------------------

EXPERIMENT_CHECKPOINT_PATH = os.path.join(
    EXPERIMENT_CHECKPOINT_DIR,
    f"{EXPERIMENT_MODEL_NAME}_{EXPERIMENT_RUN_TYPE}.keras",
)

experiment_early_stopping = build_early_stopping(
    monitor="val_loss",
    patience=5,
    mode="min",
    restore_best_weights=True,
)

experiment_model_checkpoint = build_model_checkpoint(
    filepath=EXPERIMENT_CHECKPOINT_PATH,
    monitor="val_loss",
    mode="min",
    save_best_only=True,
    save_weights_only=False,
)

experiment_reduce_lr = build_reduce_lr_on_plateau(
    monitor="val_loss",
    factor=0.5,
    patience=2,
    min_lr=1e-6,
    mode="min",
)

experiment_wandb_epoch_logger = WandBEpochLogger(
    experiment_wandb_run
)


# --------------------------------------------
# Callback List
# --------------------------------------------

EXPERIMENT_TRAINING_CALLBACKS = [
    experiment_early_stopping,
    experiment_model_checkpoint,
    experiment_reduce_lr,
    experiment_wandb_epoch_logger,
]


# --------------------------------------------
# Experiment Training
# --------------------------------------------

experiment_history = experiment_model.fit(
    experiment_train_ds,
    validation_data=experiment_val_ds,
    epochs=EXPERIMENT_EPOCHS,
    callbacks=EXPERIMENT_TRAINING_CALLBACKS,
    verbose=1,
)


# --------------------------------------------
# Actual Epoch Count
# --------------------------------------------

EXPERIMENT_COMPLETED_EPOCHS = len(
    experiment_history.history["loss"]
)


# --------------------------------------------
# Persist Training History
# --------------------------------------------
# Saved to disk so later sections (e.g. Evaluation and
# Error Analysis) can reload it even in a fresh Colab
# runtime where `experiment_history` no longer exists in memory.

EXPERIMENT_TRAINING_HISTORY_PATH = os.path.join(
    EXPERIMENT_REPORTS_DIR,
    f"{EXPERIMENT_MODEL_NAME}_{EXPERIMENT_PHASE}_{EXPERIMENT_RUN_TYPE}_training_history.json",
)

experiment_training_history_dict = get_training_history_dict(
    experiment_history
)

with open(EXPERIMENT_TRAINING_HISTORY_PATH, "w", encoding="utf-8") as file:
    json.dump(
        experiment_training_history_dict,
        file,
        indent=4,
    )


# --------------------------------------------
# Sync Actual Epoch Count to W&B
# --------------------------------------------

experiment_wandb_run.config.update(
    {
        "training_completed_epochs": EXPERIMENT_COMPLETED_EPOCHS,
    },
    allow_val_change=True,
)


# --------------------------------------------
# Log Training Summary
# --------------------------------------------
# Note: log_wandb_training_history() is NOT called here.
# WandBEpochLogger already logged every epoch in real time
# during model.fit(). Calling it here would duplicate those logs.

log_wandb_training_summary(
    experiment_wandb_run,
    experiment_history,
)


# --------------------------------------------
# Training Completion Summary
# --------------------------------------------

print("Experiment 1 training completed successfully.")
print(f"Planned Epochs    : {EXPERIMENT_EPOCHS}")
print(f"Completed Epochs  : {EXPERIMENT_COMPLETED_EPOCHS}")
print(f"Checkpoint        : {EXPERIMENT_CHECKPOINT_PATH}")
print(f"Training History  : {EXPERIMENT_TRAINING_HISTORY_PATH}")

##### Training Progress Log

**Experiment:** High-FLOPs Custom CNN  
**Training Configuration:** 50 epochs, batch size 32, input resolution 128 × 128  
**Checkpoint:** `high_input_128.keras`

###### Epoch Summary

| Epoch | Train Accuracy | Train Loss | Validation Accuracy | Validation Loss | Learning Rate |
|---:|---:|---:|---:|---:|---:|
| 1 | 0.6203 | 1.1233 | 0.6500 | 1.0365 | 1.00e-04 |
| 2 | 0.7194 | 0.8158 | 0.7040 | 0.9444 | 1.00e-04 |
| 3 | 0.7719 | 0.6651 | 0.7869 | 0.5748 | 1.00e-04 |
| 4 | 0.8169 | 0.5406 | 0.8034 | 0.5933 | 1.00e-04 |
| 5 | 0.8484 | 0.4618 | 0.7568 | 0.7680 | 1.00e-04 |
| 6 | 0.8836 | 0.3585 | 0.8165 | 0.5573 | 5.00e-05 |
| 7 | 0.8965 | 0.3176 | 0.9087 | 0.2754 | 5.00e-05 |
| 8 | 0.9001 | 0.3071 | 0.9188 | 0.2384 | 5.00e-05 |
| 9 | 0.9064 | 0.2805 | 0.8764 | 0.3680 | 5.00e-05 |
| 10 | 0.9118 | 0.2703 | 0.9225 | 0.2245 | 5.00e-05 |
| 11 | 0.9174 | 0.2572 | 0.9031 | 0.3200 | 5.00e-05 |
| 12 | 0.9173 | 0.2480 | 0.9048 | 0.2701 | 5.00e-05 |
| 13 | 0.9318 | 0.2059 | 0.9309 | 0.2106 | 2.50e-05 |
| 14 | 0.9358 | 0.1943 | 0.9497 | 0.1457 | 2.50e-05 |
| 15 | 0.9392 | 0.1850 | 0.9425 | 0.1632 | 2.50e-05 |
| 16 | 0.9380 | 0.1836 | 0.9339 | 0.1830 | 2.50e-05 |
| 17 | 0.9455 | 0.1643 | 0.9507 | 0.1407 | 1.25e-05 |
| 18 | 0.9493 | 0.1515 | 0.9623 | 0.1089 | 1.25e-05 |
| 19 | 0.9476 | 0.1552 | 0.9618 | 0.1086 | 1.25e-05 |
| 20 | 0.9505 | 0.1498 | 0.9642 | 0.1065 | 1.25e-05 |
| 21 | 0.9506 | 0.1462 | 0.9662 | 0.0982 | 1.25e-05 |
| 22 | 0.9534 | 0.1431 | 0.9573 | 0.1348 | 1.25e-05 |
| 23 | 0.9514 | 0.1430 | 0.9630 | 0.1065 | 1.25e-05 |
| 24 | 0.9567 | 0.1333 | 0.9669 | 0.0973 | 6.25e-06 |

###### Checkpoint and Learning Rate Events

- **Epoch 1:** Initial best validation loss recorded at `1.03648`.
- **Epoch 2:** Validation loss improved to `0.94441`.
- **Epoch 3:** Validation loss improved to `0.57481`.
- **Epoch 5:** `ReduceLROnPlateau` reduced the learning rate from `1.00e-04` to approximately `5.00e-05`.
- **Epoch 6:** Best validation loss improved to `0.55729`.
- **Epoch 7:** Validation loss improved to `0.27537`.
- **Epoch 8:** Validation loss improved to `0.23844`.
- **Epoch 10:** Validation loss improved to `0.22452`.
- **Epoch 12:** Learning rate reduced to approximately `2.50e-05`.
- **Epoch 13:** Validation loss improved to `0.21063`.
- **Epoch 14:** Validation loss improved to `0.14566`.
- **Epoch 16:** Learning rate reduced to approximately `1.25e-05`.
- **Epoch 17:** Validation loss improved to `0.14069`.
- **Epoch 18:** Validation loss improved to `0.10894`.
- **Epoch 19:** Validation loss improved to `0.10858`.
- **Epoch 20:** Validation loss improved to `0.10653`.
- **Epoch 21:** Validation loss improved to `0.09825`.
- **Epoch 23:** Learning rate reduced to approximately `6.25e-06`.
- **Epoch 24:** Validation loss improved to `0.09731`.

###### Training Status

- **Completed epochs:** 24 of 50.
- **Next epoch:** Epoch 25.
- **Best validation accuracy:** `96.69%` at epoch 24.
- **Best validation loss:** `0.09731` at epoch 24.
- **Latest checkpoint:** Saved at epoch 24 to `high_input_128.keras`.
- **Current status in the provided log:** Epoch 25 has started; its results are not included in this log.

**Note:** The training and validation metrics above are taken from the completed-epoch summaries. The checkpoint events reflect improvements in validation loss, not necessarily improvements in validation accuracy.

#####
**Training Status: Incomplete**

Training was configured for 50 epochs but was interrupted before completion due to hardware resource limitations. Repeated training interruptions made it impractical to continue within the available resources.

The best saved checkpoint was recovered for further evaluation. No additional training was performed during the recovery process.

#### 2.11.3.8 Experiment Decision

##### Experiment Decision

The recovered 128×128 checkpoint showed promising preliminary results, outperforming the 64×64 baseline on both validation accuracy and validation loss.

These results suggest that increasing the input size may help the model capture useful spatial information. However, since training was interrupted before the planned 50 epochs, the experiment remains incomplete and the observed improvement cannot be considered conclusive.

Given the hardware limitations and the repeated interruptions during training, further attempts were deferred to avoid spending disproportionate time and resources on this experiment.

The 128×128 checkpoint and its validation results are retained as a reference for future work. The 64×64 model remains the established baseline, and no conclusions about test-set performance are drawn from this experiment.

**Decision:** Suspend the experiment and proceed with the remaining project objectives. Revisit the hypothesis when sufficient computational resources are available.

### 2.11.4 Phase 1 → Phase 2 Decision

####

Phase 1 established the 64×64 CNN as the baseline model and provided a detailed view of its classification performance and error patterns.

The higher input size experiment produced promising preliminary validation results. The recovered 128×128 checkpoint achieved 96.69% validation accuracy and a validation loss of 0.09731, compared with 96.23% and 0.1088 for the baseline. However, training was interrupted by hardware limitations before the planned 50 epochs were completed. Therefore, the results were not considered sufficient to establish the larger input size as a definitive improvement.

Based on these findings, the project proceeded to Phase 2, focusing on retraining and hyperparameter tuning to improve the baseline model through controlled experiments. The 64×64 model remained the reference point for subsequent comparisons, while the 128×128 experiment was documented as an incomplete investigation with promising preliminary results.

The next phase focused on identifying a better-performing configuration without drawing conclusions from incomplete experiments or compromising the reliability of the final evaluation.